# ARC Grid-RL v3 — paired evaluation and evidence-gated Kaggle inference
**No model weights are included. Neither 37% nor another score is guaranteed.**

`MODEL_CHOICE="trained_grid_rl"` explicitly requires a complete, independently
reloaded export with real RL-update counters. Missing trained weights are an error,
not a quiet fallback to the old model. Use `MODEL_CHOICE="baseline"` explicitly
for the unchanged control. Keep all other settings and data identical for comparison.

The v2 program specialist, DSL restriction, and program-promotion dependency are
absent from this experimental branch. The inherited grid tokenizer, TTT, search,
shape proposals, and selector remain unchanged, except for an expired-deadline
bug fix. In hidden reruns, trained weights additionally require a matching paired
score approval. This does not guarantee transfer of the public score to hidden data.

In [ ]:
from pathlib import Path
import os,sys,json,time
ROOT=Path(os.environ.get("ARC_GRIDRL_WORKDIR", "/kaggle/working/arc_gridrl_code"))
ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
for sub in ("grid_rl","anchor","arc_exec_rl","tests"):
    (ROOT/sub).mkdir(exist_ok=True)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
os.environ["PYTHONHASHSEED"]="0"
os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"


In [ ]:
# ========================= EDITABLE SETTINGS =========================
import json
import math
import os
import time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
RUN_PROFILE = "DATASET_SAFE_MAX"  # DATASET_SAFE_MAX | PARITY_FULL | COVERAGE_FAST

# Public diagnostic gates. 80/172 is 46.51%; literal 60% is 104/172.
TARGET_REQUESTED_PAIRS = 80
TARGET_MILESTONE = 0.37
TARGET_MILESTONE_PAIRS = math.ceil(172*TARGET_MILESTONE)
TARGET_LITERAL_60_PAIRS = math.ceil(172 * 0.60)

# Full competition coverage. Keep empty for every serious run.
ARC_DEV_KEYS = ""
NPROCS = 4

# Competition/model paths. Custom primary is disabled because the previous
# ARC-Spark package scored 0/172. Re-enable only after exact parity validation.
COMPETITION_DIR_OVERRIDE = ""
PRIMARY_MODEL_OVERRIDE = ""  # Assigned only after explicit model-choice validation
ALLOW_CUSTOM_PRIMARY = False
REQUIRE_EXACT_16_TOKEN_IDS = True

# Qwen3-4B per-task adaptation. adamw_torch is the stable, known baseline.
TTT_OPTIMIZER = "adamw_torch"
TTT_LEARNING_RATE = 5e-5
TTT_LORA_R = 256
TTT_LORA_ALPHA = 32
MAX_TRAIN_AUGMENTS = 15        # 15 -> about 128 mixed views
FORCE_FULL_TTT = False         # True reproduces full TTT but risks coverage
MAX_ACCEPTED_TTT_LOSS = 5.0    # pathological/non-finite runs are reverted
ALLOW_EXACT_RETRIEVAL_SKIP = True

# Decode/search settings.
MAX_SEQ_LENGTH = 8192
SYMBOLIC_PROGRAMS = 500
ALLOW_SYMBOLIC_ATTEMPT2 = False
TTA_COLOR_PERMUTATIONS = 2     # 8 geometry x 2 color = 16 views
MAX_SHAPE_HYPOTHESES = 3
CONSTRAINED_BRANCH_CAP = 4
UNCONSTRAINED_BRANCH_CAP = 6
BEAM_CAP = 14
MAX_CANDIDATES_PER_VIEW = 6
MAX_PUZZLE_SECONDS = 900
MIN_PUZZLE_SECONDS = 240
RESERVE_SCORING_SECONDS = 55

# Runtime and diagnostics.
TOTAL_NOTEBOOK_HOURS = 12.0
FINAL_RESERVE_MINUTES = 45
RUN_PUBLIC_CPU_AUDIT = False
SAVE_DIAGNOSTICS_ZIP = True

SAFE_DIR = Path("/kaggle/working/arc_gridrl_run")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/kaggle/inference_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["ARC_DEV_KEYS"] = ARC_DEV_KEYS
os.environ["ARC_DEBUG_KEYS"] = ARC_DEV_KEYS
os.environ["ARC_OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["ARC_SAFE_ROUTE_PATH"] = str(SAFE_DIR / "safe_routes.json")
os.environ["ARC_ALLOW_SYMBOLIC_ATTEMPT2"] = "1" if ALLOW_SYMBOLIC_ATTEMPT2 else "0"
os.environ["ARC_ALLOW_EXACT_RETRIEVAL_SKIP"] = "1" if ALLOW_EXACT_RETRIEVAL_SKIP else "0"
os.environ["ARC_FORCE_FULL_TTT"] = "1" if FORCE_FULL_TTT else "0"
os.environ["ARC_MAX_ACCEPTED_TTT_LOSS"] = str(MAX_ACCEPTED_TTT_LOSS)
os.environ["ARC60_TTT_OPTIMIZER"] = TTT_OPTIMIZER
os.environ["ARC60_TTT_LR"] = str(TTT_LEARNING_RATE)
os.environ["ARC60_TTT_LORA_R"] = str(TTT_LORA_R)
os.environ["ARC60_TTT_LORA_ALPHA"] = str(TTT_LORA_ALPHA)
os.environ["ARC_MAX_SEQ_LENGTH"] = str(MAX_SEQ_LENGTH)
os.environ["ARC_MAX_TRAIN_AUGMENTS"] = str(MAX_TRAIN_AUGMENTS)
os.environ["ARC_SYMBOLIC_PROGRAMS"] = str(SYMBOLIC_PROGRAMS)
os.environ["ARC_TTA_COLOR_PERMUTATIONS"] = str(TTA_COLOR_PERMUTATIONS)
os.environ["ARC_MAX_SHAPE_HYPOTHESES"] = str(MAX_SHAPE_HYPOTHESES)
os.environ["ARC_CONSTRAINED_BRANCH_CAP"] = str(CONSTRAINED_BRANCH_CAP)
os.environ["ARC_UNCONSTRAINED_BRANCH_CAP"] = str(UNCONSTRAINED_BRANCH_CAP)
os.environ["ARC_BEAM_CAP"] = str(BEAM_CAP)
os.environ["ARC_MAX_CANDIDATES_PER_VIEW"] = str(MAX_CANDIDATES_PER_VIEW)
os.environ["ARC_MAX_PUZZLE_SECONDS"] = str(MAX_PUZZLE_SECONDS)
os.environ["ARC_MIN_PUZZLE_SECONDS"] = str(MIN_PUZZLE_SECONDS)
os.environ["ARC_RESERVE_SCORING_SECONDS"] = str(RESERVE_SCORING_SECONDS)
os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

settings = {
    "run_profile": RUN_PROFILE,
    "target_requested_pairs": TARGET_REQUESTED_PAIRS,
    "target_literal_60_pairs": TARGET_LITERAL_60_PAIRS,
    "arc_dev_keys": ARC_DEV_KEYS,
    "nprocs": NPROCS,
    "allow_custom_primary": ALLOW_CUSTOM_PRIMARY,
    "ttt_optimizer": TTT_OPTIMIZER,
    "ttt_learning_rate": TTT_LEARNING_RATE,
    "ttt_lora_r": TTT_LORA_R,
    "max_train_augments": MAX_TRAIN_AUGMENTS,
    "force_full_ttt": FORCE_FULL_TTT,
    "allow_symbolic_attempt2": ALLOW_SYMBOLIC_ATTEMPT2,
    "tta_color_permutations": TTA_COLOR_PERMUTATIONS,
    "max_shape_hypotheses": MAX_SHAPE_HYPOTHESES,
    "final_reserve_minutes": FINAL_RESERVE_MINUTES,
}
(SAFE_DIR / "settings.json").write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))


# ========= EXPLICIT CHECKPOINT CHOICE: NO SILENT BASELINE FALLBACK =========
MODEL_CHOICE="trained_grid_rl"        # baseline | trained_grid_rl | trained_sft_control
BASELINE_MODEL_PATH=""               # Set for the control run if discovery is ambiguous
TRAINED_MODEL_DIR=""                 # Directory containing grid_rl_manifest.json and merged weights
PROMOTION_FILE=""                    # Required ONLY for a hidden rerun using trained weights
INPUT_ROOT="/kaggle/input"
COMPETITION_DIR_OVERRIDE=""
WHEELHOUSE=""
INSTALL_OFFLINE_DEPS=False
if MODEL_CHOICE not in ("baseline","trained_grid_rl","trained_sft_control"):raise ValueError("Unknown MODEL_CHOICE")


## Embedded grid inference and artifact verification

In [ ]:
%%writefile grid_rl/__init__.py
"""Direct-grid RLOO research branch; no trained weights or score guarantee."""
__version__ = "3.0.0"


In [ ]:
%%writefile grid_rl/audit.py
"""Read-only audit of actual saved run artifacts. No API credentials or weights read.
Missing evidence stays UNKNOWN; a low leaderboard score does not prove a lane skipped.
"""
from __future__ import annotations
import argparse,json,os
from pathlib import Path
from .common import atomic_json
NAMES={'execution_plan.json','program_stage_summary.json','stage_report.json','rl_report.json',
       'training_report.json','sft_report.json','report.json','run_status.json',
       'hybrid_promotion.json','grid_rl_manifest.json','model_manifest.json','run_evidence.json'}


def audit_run(root: str) -> dict:
    base=Path(root)
    if not base.is_dir():raise FileNotFoundError(root)
    found=[];errors=[];seen=0
    for here,dirs,files in os.walk(base):
        dirs[:]=[d for d in dirs if d not in ('.git','__pycache__')]
        if len(Path(here).relative_to(base).parts)>=8:dirs[:]=[]
        for name in files:
            seen+=1
            if seen>50000:raise RuntimeError('Scan cap reached; choose a narrower run directory')
            if name not in NAMES:continue
            p=Path(here)/name
            if p.stat().st_size>16*1024*1024:errors.append({'path':str(p),'error':'JSON exceeds audit size cap'});continue
            try:d=json.loads(p.read_text())
            except (ValueError,OSError) as ex:errors.append({'path':str(p),'error':str(ex)});continue
            if not isinstance(d,dict):continue
            fields={k:d[k] for k in ('status','program_ready','program_mode','program_model','approved','applied',
               'tasks_completed','task_count','changed_attempt2','optimizer_steps_this_run','optimizer_steps_total',
               'rl_optimizer_steps','rl_updates_this_session','weights_id','reload_verified','passed','pass2_exact',
               'pair_micro_pass2','method_used','model_choice','trained_weights_id','score_claim') if k in d}
            if isinstance(fields.get('changed_attempt2'),list):fields['changed_attempt2_count']=len(fields.pop('changed_attempt2'))
            found.append({'path':str(p.relative_to(base)),'file':name,'fields':fields})
    plans=[x['fields'] for x in found if x['file']=='execution_plan.json']
    stages=[x['fields'] for x in found if x['file'] in ('program_stage_summary.json','stage_report.json')]
    facts=[]
    if any(x.get('program_ready') is False for x in plans):facts.append('A saved execution plan says the program specialist was not ready.')
    if any(str(x.get('status','')).startswith('NOT_RUN') for x in stages):facts.append('A saved program-stage report explicitly records NOT_RUN.')
    if any(x.get('applied') is False for x in stages):facts.append('A program-stage report explicitly records applied=false.')
    if any(x.get('applied') is True for x in stages):facts.append('A stage reports applied=true; verify changed-output count and model identity before attributing a gain.')
    return {'audit_scope':str(base.resolve()),'evidence_files':found,'parse_errors':errors,'supported_observations':facts,
            'latest_user_score':'user-reported below 30; not independently verified by this audit',
            'causal_attribution':'UNKNOWN without matched baseline/checkpoint/candidate reports',
            'training_status':'UNKNOWN' if not any(x['file'] in ('rl_report.json','training_report.json') for x in found) else 'SEE_REPORTED_COUNTERS',
            'action':'Do not remove integrity or score gates. Fix the missing model, invalid package, failed learning, or scheduling issue identified by artifacts.'}

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--output',required=True);a=p.parse_args()
    r=audit_run(a.root);atomic_json(a.output,r);print(json.dumps(r,indent=2))


In [ ]:
%%writefile grid_rl/codec.py
"""Exact legacy grid serialization and variable-shape autoregressive grammar.
No target height or width is supplied to the sampler. Values below are validated
against the attached tokenizer BEFORE model work. They are not for normal Qwen.
"""
from __future__ import annotations
from dataclasses import dataclass
import numpy as np
from .common import validate_grid

PAD, NL, EOS = 13, 10, 15
USER, ASSISTANT = [14, 11, 10], [14, 12, 10]
MAX_GRID_TOKENS = 930  # 30*30 cells + 29 row separators + EOS
CODEC_VERSION = 'legacy-grid16-free-shape-v3'


def grid_tokens(grid: list[list[int]]) -> list[int]:
    validate_grid(grid)
    out = []
    for i, row in enumerate(grid):
        if i: out.append(NL)
        out.extend(row)
    return out


def prompt_tokens(demos: list[dict], query: list[list[int]]) -> list[int]:
    if not demos: raise ValueError('At least one demonstration is required')
    out = []
    for d in demos:
        out += USER + grid_tokens(d['input']) + [EOS]
        out += ASSISTANT + grid_tokens(d['output']) + [EOS]
    return out + USER + grid_tokens(query) + [EOS] + ASSISTANT


def reply_tokens(grid: list[list[int]]) -> list[int]:
    return grid_tokens(grid) + [EOS]


def verify_tokenizer(tokenizer) -> dict:
    expected = {**{str(i): [i] for i in range(10)}, '\n': [NL],
                '<|im_start|>user\n': USER, '<|im_start|>assistant\n': ASSISTANT,
                '<|im_end|>': [EOS], '<|endoftext|>': [PAD]}
    if len(tokenizer) != 16: raise ValueError('Expected exactly 16 tokens, not the normal Qwen vocabulary')
    for text, ids in expected.items():
        actual = list(map(int, tokenizer.encode(text, add_special_tokens=False)))
        if actual != ids: raise ValueError(f'Token contract failed: {text!r}: {actual} != {ids}')
    return {'codec': CODEC_VERSION, 'vocabulary': 16, 'semantic_contract': 'PASS'}


@dataclass
class GridGrammar:
    """First row sets width; subsequent rows must match; shape remains unknown."""
    width: int | None = None
    row_length: int = 0
    completed_rows: int = 0
    ended: bool = False

    def allowed(self) -> tuple[int, ...]:
        if self.ended: return ()
        if self.width is None:
            return tuple(range(10)) + ((NL, EOS) if self.row_length else ()) if self.row_length < 30 else (NL, EOS)
        if self.row_length < self.width: return tuple(range(10))
        return (EOS,) if self.completed_rows == 29 else (NL, EOS)

    def consume(self, token: int) -> None:
        if type(token) is not int or token not in self.allowed():
            raise ValueError(f'Illegal token {token} in grammar state {self}')
        if token < 10: self.row_length += 1
        elif token == NL:
            if self.width is None: self.width = self.row_length
            self.completed_rows += 1; self.row_length = 0
        else: self.ended = True


def decode_reply(tokens: list[int]) -> list[list[int]]:
    state = GridGrammar(); rows = [[]]
    for i, t in enumerate(tokens):
        state.consume(t)
        if t < 10: rows[-1].append(t)
        elif t == NL: rows.append([])
        elif i != len(tokens)-1: raise ValueError('Tokens follow EOS')
    if not state.ended: raise ValueError('Incomplete reply is not a solved grid')
    return validate_grid(rows)


def allowed_masks(tokens: list[int]) -> list[list[bool]]:
    state = GridGrammar(); masks = []
    for t in tokens:
        allowed = state.allowed(); masks.append([i in allowed for i in range(16)])
        state.consume(t)
    return masks


def transform_grid(grid: list[list[int]], k: int, flip: bool, palette: list[int]) -> list[list[int]]:
    if sorted(palette) != list(range(10)): raise ValueError('Palette must be a bijection')
    a = np.rot90(np.asarray(validate_grid(grid), dtype=np.int64), k)
    if flip: a = np.fliplr(a)
    return np.asarray(palette)[a].tolist()


def inverse_grid(grid: list[list[int]], k: int, flip: bool, palette: list[int]) -> list[list[int]]:
    if sorted(palette) != list(range(10)): raise ValueError('Palette must be a bijection')
    a = np.argsort(palette)[np.asarray(validate_grid(grid), dtype=np.int64)]
    if flip: a = np.fliplr(a)
    return np.rot90(a, -k).tolist()


In [ ]:
%%writefile grid_rl/common.py
from __future__ import annotations
import hashlib
import json
import math
import os
import platform
import tempfile
import time
from pathlib import Path
from typing import Any


def truthy(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes", "on"}


def stable_seed(*parts: Any) -> int:
    return int(hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:8], 16)


def json_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def file_hash(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path: str | Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".writing-", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(value, f, separators=(",", ":"), sort_keys=True, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
        with open(tmp, encoding="utf-8") as f:
            json.load(f)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)


def append_event(path: str | Path, event: str, **fields: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Each process uses its own event file. No cross-process buffering assumptions.
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"event": event, "time": time.time(), **fields}, allow_nan=False) + "\n")
        f.flush()


def validate_grid(g: Any) -> list[list[int]]:
    # Strict: floats, booleans and stringified arrays are not accepted.
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        raise ValueError("Grid must have 1..30 rows")
    if not isinstance(g[0], list) or not 1 <= len(g[0]) <= 30:
        raise ValueError("Grid must have 1..30 columns")
    w = len(g[0])
    if any(not isinstance(row, list) or len(row) != w for row in g):
        raise ValueError("Ragged grid")
    if any(type(x) is not int or not 0 <= x <= 9 for row in g for x in row):
        raise ValueError("Cells must be integer colors 0..9")
    return g


def public_task(task: dict) -> dict:
    """The only view supplied to inference: all known demos, test INPUTS only."""
    if not task.get("train") or not task.get("test"):
        raise ValueError("Task requires demonstrations and test inputs")
    return {"train": [{"input": validate_grid(p["input"]), "output": validate_grid(p["output"])}
                      for p in task["train"]],
            "test": [{"input": validate_grid(p["input"])} for p in task["test"]]}


def validate_submission(tasks: dict, submission: dict) -> None:
    if set(tasks) != set(submission):
        raise ValueError("Submission IDs differ from challenge IDs")
    for key, task in tasks.items():
        pairs = submission[key]
        if not isinstance(pairs, list) or len(pairs) != len(task["test"]):
            raise ValueError(f"Wrong number of test records: {key}")
        for p in pairs:
            if set(p) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"Invalid attempt keys: {key}")
            validate_grid(p["attempt_1"])
            validate_grid(p["attempt_2"])
    # Duplicate attempts are legal. Do not invent a one-cell mutation.


def initial_submission(tasks: dict) -> dict:
    out = {k: [{"attempt_1": p["input"], "attempt_2": p["input"]} for p in public_task(t)["test"]]
           for k, t in tasks.items()}
    validate_submission(tasks, out)
    return out


def score_submission(tasks: dict, solutions: dict, submission: dict, candidates: dict | None = None) -> dict:
    validate_submission(tasks, submission)
    if set(tasks) != set(solutions):
        raise ValueError("Solutions must match the evaluated challenge subset exactly")
    first = either = fully = oracle = total = 0
    per_task = {}
    for key, task in tasks.items():
        ys = solutions[key]
        if len(ys) != len(task["test"]):
            raise ValueError(f"Solution count mismatch: {key}")
        wins = []
        for i, y in enumerate(ys):
            validate_grid(y)
            p = submission[key][i]
            a = p["attempt_1"] == y
            b = a or p["attempt_2"] == y
            all_grids = [p["attempt_1"], p["attempt_2"]]
            if candidates:
                all_grids += candidates.get(f"{key}:{i}", [])
            total += 1
            first += int(a)
            either += int(b)
            oracle += int(y in all_grids)
            wins.append(int(b))
        fully += int(all(wins))
        per_task[key] = {"pairs": len(wins), "exact": sum(wins), "fraction": sum(wins) / len(wins)}
    return {"tasks": len(tasks), "pairs": total, "attempt1_exact": first, "pass2_exact": either,
            "pair_micro_pass2": either / max(total, 1),
            "task_macro_pass2": sum(x["fraction"] for x in per_task.values()) / max(len(tasks), 1),
            "fully_solved_tasks": fully, "candidate_oracle_exact": oracle,
            "target_60_required_pairs": math.ceil(.60 * total),
            "target_60_pair_micro_met": either >= math.ceil(.60 * total),
            "per_task": per_task}


def environment_report() -> dict:
    import importlib.metadata as md
    r = {"python": platform.python_version(), "architecture": platform.machine(), "system": platform.platform()}
    for p in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "tokenizers"):
        try:
            r[p] = md.version(p)
        except md.PackageNotFoundError:
            r[p] = "NOT_INSTALLED"
    try:
        import torch
        r["cuda_available"] = torch.cuda.is_available()
        r["torch_cuda"] = torch.version.cuda
        r["gpus"] = [{"name": torch.cuda.get_device_name(i),
                      "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
                     for i in range(torch.cuda.device_count())]
    except ImportError:
        r["cuda_available"] = False
    return r


def score_banks(tasks: dict,solutions: dict,banks: dict) -> dict:
    """Abstention-aware scoring: no candidates cannot accidentally count as a solution."""
    total=first=either=oracle=fully=0; pt={}
    if set(tasks)!=set(solutions): raise ValueError("Mismatched solutions")
    for key,t in tasks.items():
        ys=solutions[key]
        if len(ys)!=len(t["test"]): raise ValueError("Mismatched test outputs")
        wins=[]
        for i,y in enumerate(ys):
            validate_grid(y); candidates=banks.get(f"{key}:{i}",[])
            for g in candidates: validate_grid(g)
            a=bool(candidates) and candidates[0]==y
            b=any(g==y for g in candidates[:2]); o=any(g==y for g in candidates)
            first+=int(a);either+=int(b);oracle+=int(o);total+=1;wins.append(int(b))
        fully+=int(all(wins));pt[key]={"pairs":len(wins),"exact":sum(wins),"fraction":sum(wins)/len(wins)}
    return {"tasks":len(tasks),"pairs":total,"attempt1_exact":first,"pass2_exact":either,
            "pair_micro_pass2":either/max(total,1),"task_macro_pass2":sum(v['fraction'] for v in pt.values())/max(len(pt),1),
            "fully_solved_tasks":fully,"candidate_oracle_exact":oracle,"per_task":pt,
            "target_60_required_pairs":math.ceil(.60*total),"target_60_pair_micro_met":either>=math.ceil(.60*total)}


def atomic_candidate_pickle(path: str | Path,value: Any) -> None:
    """Only for our own legacy worker outputs, never for untrusted downloaded pickles."""
    import bz2,pickle
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix='.candidate-',dir=p.parent);os.close(fd)
    try:
        with bz2.BZ2File(tmp,'wb') as f: pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp,p)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)


In [ ]:
%%writefile grid_rl/data.py
"""ARC grid supervision, task/family splits, and on-the-fly exact augmentation.
Official training holdouts are incremental-regression diagnostics: sft139 may
already have seen them. They are NEVER labelled clean base-model holdouts.
"""
from __future__ import annotations
import copy
import json
import random
from collections import Counter
from pathlib import Path
from .common import atomic_json, file_hash, json_hash, public_task, stable_seed, validate_grid
from .codec import prompt_tokens, reply_tokens, transform_grid


def family_split(family: str) -> str:
    x = stable_seed('grid-rl-v3-split', family) % 10
    return 'development' if x == 0 else 'confirmation' if x == 1 else 'train'


def validate_record(r: dict, forbidden: set[str]) -> dict:
    required = {'record_id','family_id','source','license','source_task_ids','task','query_outputs','split'}
    if not required <= set(r): raise ValueError(f'Missing fields: {required-set(r)}')
    if not isinstance(r['family_id'], str) or not r['family_id']: raise ValueError('Empty family')
    if not isinstance(r['record_id'], str) or not r['record_id']: raise ValueError('Empty record id')
    if r['split'] not in ('train','development','confirmation'): raise ValueError('Bad split')
    if not isinstance(r['source_task_ids'], list): raise ValueError('List of provenance IDs required')
    if forbidden.intersection(r['source_task_ids']) or r['record_id'] in forbidden:
        raise ValueError('Public-evaluation provenance is not allowed in this training branch')
    task = public_task(r['task']); ys = r['query_outputs']
    if len(ys) != len(task['test']): raise ValueError('Query/answer count mismatch')
    for p,y in zip(r['task']['test'], ys):
        validate_grid(y)
        if 'output' in p and p['output'] != y: raise ValueError('Conflicting inline query answer')
    if not r['source'] or not r['license']: raise ValueError('Source and license declarations required')
    return {**r,'task':task,'base_model_exposure':r.get('base_model_exposure','unknown')}


def build_corpus(data_dir: str, output_dir: str, external_jsonl: str = '') -> dict:
    root = Path(data_dir); out = Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    train = json.loads((root/'arc-agi_training_challenges.json').read_text())
    sol = json.loads((root/'arc-agi_training_solutions.json').read_text())
    forbidden = set(json.loads((root/'arc-agi_evaluation_challenges.json').read_text()))
    if set(train) != set(sol): raise ValueError('Official training solution IDs mismatch')
    rows = []
    for k in sorted(train):
        family = 'official-'+k
        rows.append(validate_record({'record_id':k,'family_id':family,'source':'official-training',
            'license':'Apache-2.0','source_task_ids':[k],'task':train[k],'query_outputs':sol[k],
            'split':family_split(family),'base_model_exposure':'potentially-seen-by-sft139'}, forbidden))
    if external_jsonl:
        for line in Path(external_jsonl).read_text().splitlines():
            if line.strip(): rows.append(validate_record(json.loads(line), forbidden))
    ids = set(); family_splits = {}; canonical_splits = {}; pair_splits = {}
    # Exact cross-split examples are forbidden. D4/color/semantic near-duplicates
    # require a separate family-lineage audit; this checker does not prove it.
    for r in rows:
        if r['record_id'] in ids: raise ValueError('Duplicate record ID')
        ids.add(r['record_id']); f = r['family_id']; s = r['split']
        if f in family_splits and family_splits[f] != s: raise ValueError('Family crosses split')
        family_splits[f] = s
        for p in r['task']['train'] + [{'input':p['input'],'output':y} for p,y in zip(r['task']['test'],r['query_outputs'])]:
            h=json_hash(p)
            if h in pair_splits and pair_splits[h]!=s: raise ValueError('Exact input/output pair crosses split')
            pair_splits[h]=s
        h = json_hash([sorted((json_hash(p) for p in r['task']['train'])), sorted(json_hash([p['input'],y]) for p,y in zip(r['task']['test'],r['query_outputs']))])
        if h in canonical_splits and canonical_splits[h] != s: raise ValueError('Exact task crosses split')
        canonical_splits[h]=s
    hashes={};counts={};families={}
    for split in ('train','development','confirmation'):
        selected=sorted([r for r in rows if r['split']==split],key=lambda r:r['record_id'])
        path=out/f'{split}.jsonl'
        path.write_text(''.join(json.dumps(r,sort_keys=True,separators=(',',':'))+'\n' for r in selected))
        hashes[path.name]=file_hash(path);counts[split]=len(selected);families[split]=len(set(r['family_id'] for r in selected))
    manifest={'format':'grid-rl-corpus-3','files':hashes,'dataset_hash':json_hash(hashes),
        'counts':counts,'families':families,'sources':dict(Counter(r['source'] for r in rows)),
        'evaluation_ids_used_for_exclusion_only':len(forbidden),'no_public_evaluation_rewards':True,
        'holdout_status':'incremental-regression; base-model exposure unknown; NOT clean benchmark evidence',
        'semantic_near_duplicate_audit':'NOT_PERFORMED',
        'source_hashes':{n:file_hash(root/n) for n in ('arc-agi_training_challenges.json','arc-agi_training_solutions.json','arc-agi_evaluation_challenges.json')}}
    atomic_json(out/'manifest.json',manifest);return manifest


def load_split(root: str, split: str) -> tuple[list[dict],dict]:
    p=Path(root);m=json.loads((p/'manifest.json').read_text())
    for name,h in m['files'].items():
        if Path(name).name!=name or file_hash(p/name)!=h: raise ValueError('Corpus file hash mismatch')
    if json_hash(m['files'])!=m['dataset_hash']: raise ValueError('Manifest hash mismatch')
    rows=[json.loads(x) for x in (p/f'{split}.jsonl').read_text().splitlines() if x.strip()]
    if len(rows)!=m['counts'][split] or any(r['split']!=split for r in rows): raise ValueError('Split mismatch')
    return rows,m


def episode(record: dict, seed: int, augment: bool = True, max_prompt: int = 7000,
            hold_demo_probability: float = .5) -> dict:
    rng=random.Random(seed); task=copy.deepcopy(record['task'])
    if rng.random()<hold_demo_probability and len(task['train'])>=3:
        idx=rng.randrange(len(task['train'])); q=task['train'].pop(idx)
        x,y=q['input'],q['output'];kind='leave-one-demo-out'
    else:
        idx=rng.randrange(len(task['test']));x=task['test'][idx]['input'];y=record['query_outputs'][idx];kind='training-query'
    k=rng.randrange(4) if augment else 0;flip=bool(rng.randrange(2)) if augment else False
    palette=list(range(10))
    if augment: rng.shuffle(palette);rng.shuffle(task['train'])
    conv=lambda g:transform_grid(g,k,flip,palette)
    demos=[{'input':conv(p['input']),'output':conv(p['output'])} for p in task['train']]
    tx,ty=conv(x),conv(y);prompt=prompt_tokens(demos,tx);target=reply_tokens(ty)
    if len(prompt)>max_prompt: raise ValueError('PROMPT_TOO_LONG: no demonstrations silently removed')
    return {'record_id':record['record_id'],'family_id':record['family_id'],'kind':kind,
            'prompt':prompt,'target':target,'target_grid':ty,
            'transform':{'k':k,'flip':flip,'palette':palette}}


In [ ]:
%%writefile grid_rl/evidence.py
"""Paired final-output comparison; public scores are diagnostics, never hidden guarantees."""
from __future__ import annotations
import json,math
from pathlib import Path
from .common import atomic_json,json_hash,score_submission,file_hash


def decoder_identity(root: str) -> str:
    names=['arc_loader.py','arc_solver.py','arc_decoder.py','arc_symbolic.py','arc_retrieval.py','starter.py','arc_dataset_guard.py']
    return json_hash({n:file_hash(Path(root)/n) for n in names})


def compare_runs(baseline: dict,candidate: dict,tasks: dict,solutions: dict,baseline_submission: dict,candidate_submission: dict) -> dict:
    for k in ('challenge_hash','solutions_hash','decoder_hash','decoder_config_hash'):
        if not baseline.get(k) or baseline.get(k)!=candidate.get(k):raise ValueError('Paired run mismatch: '+k)
    if baseline['challenge_hash']!=json_hash(tasks) or baseline['solutions_hash']!=json_hash(solutions):raise ValueError('Dataset content mismatch')
    if baseline.get('model_choice')!='baseline' or candidate.get('model_choice')!='trained_grid_rl':raise ValueError('Need baseline and trained-grid-RL runs')
    if candidate.get('rl_optimizer_steps',0)<1:raise ValueError('Candidate does not prove an RL update')
    if baseline.get('submission_hash')!=json_hash(baseline_submission) or candidate.get('submission_hash')!=json_hash(candidate_submission):
        raise ValueError('Predictions do not match the recorded run artifact')
    if not baseline.get('base_weights_id') or candidate.get('training_base_weights_id')!=baseline['base_weights_id']:
        raise ValueError('RL candidate was trained from a different or unidentified baseline')
    if not candidate.get('trained_weights_id'):raise ValueError('Missing trained checkpoint identity')
    a=score_submission(tasks,solutions,baseline_submission);b=score_submission(tasks,solutions,candidate_submission)
    gains=[];losses=[]
    for key in tasks:
        for i,y in enumerate(solutions[key]):
            aa=any(g==y for g in baseline_submission[key][i].values());bb=any(g==y for g in candidate_submission[key][i].values())
            if bb and not aa:gains.append(f'{key}:{i}')
            if aa and not bb:losses.append(f'{key}:{i}')
    needed=math.ceil(.37*b['pairs']);target=b['pass2_exact']>=needed
    r={'scope':'paired-full-deployment-diagnostic','passed':bool(target and len(gains)-len(losses)>=3 and
       baseline.get('neural_coverage',0)>=.95 and candidate.get('neural_coverage',0)>=.95 and
       baseline.get('worker_returncode')==0 and candidate.get('worker_returncode')==0 and
       baseline.get('full_dataset') is True and candidate.get('full_dataset') is True and len(tasks)>=100),
       'target_37_pair_micro_met':target,'needed_pairs':needed,'net_new_pairs':len(gains)-len(losses),
       'gains':gains,'losses':losses,'baseline':a,'candidate':b,
       'weights_id':candidate.get('trained_weights_id'),'decoder_hash':candidate['decoder_hash'],
       'decoder_config_hash':candidate['decoder_config_hash'],'challenge_hash':candidate['challenge_hash'],
       'baseline_evidence_hash':json_hash(baseline),'candidate_evidence_hash':json_hash(candidate),
       'score_protocol':'pair-micro exact pass@2; also report task-macro separately',
       'hidden_score_guarantee':False,'training_exposure':'unknown inherited exposure; public diagnostic is not clean generalization proof'}
    return r


In [ ]:
%%writefile grid_rl/inputs.py
"""Bounded offline discovery for Kaggle mounts. A URL is not a filesystem path.
Does not download models or silently choose among ambiguous checkpoints.
"""
from __future__ import annotations
import importlib.metadata
import json
import os
import struct
import subprocess
import sys
import zipfile
from pathlib import Path
from .common import atomic_json, file_hash, public_task

DATA_FILES = ('arc-agi_training_challenges.json','arc-agi_training_solutions.json',
              'arc-agi_evaluation_challenges.json','arc-agi_evaluation_solutions.json',
              'arc-agi_test_challenges.json','sample_submission.json')
PACKAGES = {'transformers':'4.55.4','peft':'0.17.1','accelerate':'1.10.1'}


def scan(root: str | Path, names: set[str], limit: int=100000) -> list[Path]:
    root=Path(root); found=[]; count=0
    if not root.is_dir(): raise FileNotFoundError(f'Input root does not exist: {root}')
    for here,dirs,files in os.walk(root, followlinks=False):
        if len(Path(here).relative_to(root).parts)>=10: dirs[:]=[]
        dirs[:]=sorted(d for d in dirs if d not in ('.git','__pycache__','.cache'))
        for name in sorted(files):
            count+=1
            if count>limit: raise RuntimeError('Input scan cap reached; specify the exact mounted input')
            if name in names: found.append(Path(here)/name)
    return found


def tensor_headers(path: str | Path) -> dict:
    """Read only safetensors headers; detect missing/truncated tensors without loading GBs."""
    path=Path(path); size=path.stat().st_size
    with path.open('rb') as f:
        b=f.read(8)
        if len(b)!=8: raise ValueError(f'Truncated safetensors header: {path.name}')
        n=struct.unpack('<Q',b)[0]
        if not 2<=n<=16*1024*1024 or 8+n>size: raise ValueError('Invalid safetensors header size')
        h=json.loads(f.read(n))
    data_size=size-8-n
    widths={'F64':8,'F32':4,'F16':2,'BF16':2,'I64':8,'I32':4,'I16':2,'I8':1,'U8':1,'BOOL':1}
    intervals=[]; result={}
    for key,v in h.items():
        if key=='__metadata__': continue
        shape=v.get('shape'); off=v.get('data_offsets'); dtype=v.get('dtype')
        if not isinstance(shape,list) or any(type(x)is not int or x<0 for x in shape): raise ValueError('Bad tensor shape')
        if not isinstance(off,list) or len(off)!=2 or any(type(x)is not int for x in off) or not 0<=off[0]<=off[1]<=data_size:
            raise ValueError(f'Truncated/invalid tensor offsets: {key}')
        if dtype in widths:
            count=1
            for x in shape: count*=x
            if count*widths[dtype]!=off[1]-off[0]: raise ValueError(f'Tensor byte length mismatch: {key}')
        intervals.append((off[0],off[1],key));result[key]={'shape':shape,'dtype':dtype}
    end=0
    for begin,finish,key in sorted(intervals):
        if begin<end and finish>begin: raise ValueError(f'Overlapping tensor bytes: {key}')
        end=max(end,finish)
    if not result: raise ValueError('No tensors')
    return result


def inspect_weights(root: str | Path, cfg: dict) -> dict:
    root=Path(root); index=root/'model.safetensors.index.json'; mapping=None
    if index.exists():
        mapping=json.loads(index.read_text())['weight_map']; names=sorted(set(mapping.values()))
    else: names=[p.name for p in sorted(root.glob('model*.safetensors'))]
    if not names: raise ValueError('No base/merged safetensors weights (adapter alone is insufficient)')
    headers={}
    for n in names:
        if Path(n).name!=n: raise ValueError('Unsafe shard path')
        if not(root/n).is_file(): raise ValueError(f'Missing shard: {n}')
        h=tensor_headers(root/n)
        if set(headers)&set(h): raise ValueError('Duplicate tensors across shards')
        if mapping:
            for key in h:
                if mapping.get(key)!=n: raise ValueError('Index/shard contents disagree')
        headers.update(h)
    if mapping and set(headers)!=set(mapping): raise ValueError('Incomplete weight index')
    emb=headers.get('model.embed_tokens.weight'); head=headers.get('lm_head.weight')
    expected=[cfg['vocab_size'],cfg['hidden_size']]
    if not emb or emb['shape']!=expected: raise ValueError('Embedding shape does not match config')
    if head is None and not cfg.get('tie_word_embeddings',False): raise ValueError('Missing untied LM head')
    if head is not None and head['shape']!=expected: raise ValueError('LM head shape mismatch')
    layers=cfg['num_hidden_layers'];d=cfg['hidden_size'];heads=cfg['num_attention_heads']
    kv=cfg['num_key_value_heads'];hd=cfg.get('head_dim',d//heads);ff=cfg['intermediate_size']
    shapes={'model.norm.weight':[d]}
    for i in range(layers):
        base=f'model.layers.{i}.'
        shapes.update({base+'self_attn.q_proj.weight':[heads*hd,d],
            base+'self_attn.k_proj.weight':[kv*hd,d],base+'self_attn.v_proj.weight':[kv*hd,d],
            base+'self_attn.o_proj.weight':[d,heads*hd],
            base+'self_attn.q_norm.weight':[hd],base+'self_attn.k_norm.weight':[hd],
            base+'mlp.gate_proj.weight':[ff,d],base+'mlp.up_proj.weight':[ff,d],base+'mlp.down_proj.weight':[d,ff],
            base+'input_layernorm.weight':[d],base+'post_attention_layernorm.weight':[d]})
    for name,shape in shapes.items():
        if name not in headers or headers[name]['shape']!=shape:
            raise ValueError(f'Missing/wrong Qwen tensor: {name}; expected {shape}')
    return {'shards':names,'bytes':sum((root/n).stat().st_size for n in names),'tensor_count':len(headers),
            'header_integrity':'PASS','full_tensor_numerics':'NOT_CHECKED'}


def model_contract(root: str | Path, role: str, check_headers: bool=True) -> dict:
    root=Path(root).resolve();cfg=json.loads((root/'config.json').read_text())
    if role not in ('grid','program'): raise ValueError('Role must be grid or program')
    if cfg.get('model_type')!='qwen3' or cfg.get('hidden_size')!=2560 or cfg.get('num_hidden_layers')!=36:
        raise ValueError('Expected the Qwen3-4B architecture')
    vocab=cfg.get('vocab_size',0)
    if role=='grid' and vocab!=16: raise ValueError('G0 requires the exact 16-token ARC model')
    if role=='program' and vocab<100000: raise ValueError('P1 requires the full vocabulary, never the 16-token ARC model')
    tj=json.loads((root/'tokenizer.json').read_text());tv=tj.get('model',{}).get('vocab',{})
    if role=='grid':
        for tok,num in {**{str(i):i for i in range(10)},'Ċ':10,'user':11,'assistant':12}.items():
            if tv.get(tok)!=num: raise ValueError(f'ARC token mapping mismatch: {tok}')
        full=dict(tv)
        for t in tj.get('added_tokens',[]): full[t['content']]=t['id']
        for t,n in {'<|endoftext|>':13,'<|im_start|>':14,'<|im_end|>':15}.items():
            if full.get(t)!=n: raise ValueError(f'ARC control mismatch: {t}')
    elif len(tv)<100000: raise ValueError('P1 tokenizer vocabulary is incomplete')
    tc=root/'tokenizer_config.json'
    if not tc.exists(): raise ValueError('Missing tokenizer_config.json')
    info={'role':role,'path':str(root),'config_sha256':file_hash(root/'config.json'),
          'tokenizer_sha256':file_hash(root/'tokenizer.json'),'architecture':'qwen3-4b'}
    if check_headers: info.update(inspect_weights(root,cfg))
    return info


def resolve_model(input_root: str | Path, role: str, override: str='', prefer: str='') -> tuple[str,dict]:
    if override:
        info=model_contract(override,role);return info['path'],info
    possible=[]; rejected=[]
    for p in scan(input_root,{'config.json'}):
        try:
            cfg=json.loads(p.read_text())
            if cfg.get('model_type')!='qwen3' or cfg.get('hidden_size')!=2560 or cfg.get('num_hidden_layers')!=36: continue
            if (cfg.get('vocab_size')==16)!=(role=='grid'): continue
            info=model_contract(p.parent,role)
            possible.append(info)
        except (ValueError,OSError,KeyError) as e: rejected.append({'path':str(p.parent),'error':str(e)})
    preferred=[i for i in possible if prefer.lower() in i['path'].lower()] if prefer else []
    pool=preferred or possible
    if len(pool)!=1:
        raise ValueError(f'{role} model: expected one unambiguous mounted checkpoint, found {len(pool)}. '
                         f'Set an explicit override. Candidates={[p["path"] for p in pool]}; rejected={rejected[:5]}')
    result=dict(pool[0]);result['rejected_candidates']=rejected
    return result['path'],result


def resolve_data(input_root: str | Path, output_dir: str | Path, override: str='') -> tuple[str,dict]:
    """Accept an ordinary competition mount or an uploaded official ZIP; normalize JSON only."""
    if override: paths=[Path(override)]
    else:
        paths=list({p.parent for p in scan(input_root,{'arc-agi_training_challenges.json'})})
        if not paths:
            paths=[p for p in Path(input_root).rglob('*.zip') if 'arc' in p.name.lower()]
    good=[]
    for p in sorted(paths):
        try:
            if p.is_dir():
                if all((p/n).is_file() for n in DATA_FILES[:2]): good.append(p)
            else:
                with zipfile.ZipFile(p) as z:
                    names=[Path(n).name for n in z.namelist()]
                    if all(n in names for n in DATA_FILES[:2]): good.append(p)
        except (OSError,zipfile.BadZipFile): continue
    if len(good)!=1: raise ValueError(f'Need one unambiguous ARC data input. Set DATA_SOURCE. Candidates={list(map(str,good))}')
    source=good[0]
    if source.is_file():
        root=Path(output_dir);root.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(source) as z:
            for name in DATA_FILES:
                matches=[i for i in z.infolist() if Path(i.filename).name==name and not i.is_dir()]
                if len(matches)>1: raise ValueError('Duplicate canonical JSON filenames in archive')
                if not matches: continue
                info=matches[0]
                if info.file_size>64*1024*1024: raise ValueError('ARC JSON unexpectedly large')
                data=json.loads(z.read(info));atomic_json(root/name,data)
    else: root=source
    train=json.loads((root/DATA_FILES[0]).read_text());sol=json.loads((root/DATA_FILES[1]).read_text())
    if set(train)!=set(sol): raise ValueError('Training challenge/solution IDs differ')
    report={'source':str(source),'directory':str(root),'training_tasks':len(train),
            'files':{n:file_hash(root/n) for n in DATA_FILES if (root/n).exists()}}
    for prefix in ('evaluation','test'):
        q=root/f'arc-agi_{prefix}_challenges.json'
        if q.exists():
            tasks=json.loads(q.read_text());report[prefix+'_tasks']=len(tasks)
            report[prefix+'_outputs']=sum(len(t['test']) for t in tasks.values())
            report[prefix+'_exact_training_overlap']=sum(k in train and public_task(t)==public_task(train[k]) for k,t in tasks.items())
    report['test_is_public_training_placeholder']=bool(report.get('test_tasks',0) and report.get('test_tasks')==report.get('test_exact_training_overlap'))
    report['score_claim']=None
    return str(root),report


def install_offline(wheelhouse: str, packages: dict | None=None) -> None:
    if not wheelhouse: raise ValueError('Attach a wheel dataset and specify WHEELHOUSE; no online installation')
    p=Path(wheelhouse)
    if not p.is_dir() or not list(p.glob('*.whl')): raise ValueError('WHEELHOUSE must directly contain compatible wheels')
    wanted=packages or PACKAGES
    if any(x.lower().startswith(('torch','nvidia')) for x in wanted): raise ValueError('Do not replace the working CUDA/PyTorch stack')
    subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--find-links',str(p),
                   *[f'{k}=={v}' for k,v in wanted.items()]],check=True,timeout=240)


def dependency_report() -> dict:
    versions={}
    for n in ('torch','transformers','peft','accelerate','safetensors','tokenizers','huggingface-hub','numpy'):
        try: versions[n]=importlib.metadata.version(n)
        except importlib.metadata.PackageNotFoundError: versions[n]=None
    return {'versions':versions,'reference_versions':PACKAGES,
            'missing':[k for k in versions if versions[k] is None],
            'cuda_kernels':'NOT_TESTED_BY_VERSION_CHECK'}


In [ ]:
%%writefile grid_rl/modeling.py
from __future__ import annotations
import inspect,json,os,shutil
from pathlib import Path
from .common import atomic_json,file_hash,json_hash
from .inputs import model_contract
from .codec import verify_tokenizer,CODEC_VERSION

TARGETS=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']


def source_fingerprint(root: str) -> dict:
    p=Path(root);info=model_contract(p,'grid')
    names=['config.json','tokenizer.json','tokenizer_config.json']+info['shards']
    for n in ('special_tokens_map.json','generation_config.json','model.safetensors.index.json'):
        if (p/n).exists():names.append(n)
    files={n:file_hash(p/n) for n in sorted(set(names))}
    return {'files':files,'weights_id':json_hash({n:h for n,h in files.items() if n.endswith('.safetensors')}),
            'contract':info}


def load_grid_model(path: str, device: str='cuda:0', training: bool=False):
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer
    model_contract(path,'grid')
    if device.startswith('cuda') and not torch.cuda.is_available():raise RuntimeError('CUDA unavailable; Qwen training NOT_RUN')
    tok=AutoTokenizer.from_pretrained(path,local_files_only=True,trust_remote_code=False);verify_tokenizer(tok)
    model=AutoModelForCausalLM.from_pretrained(path,torch_dtype=torch.bfloat16,
                attn_implementation='sdpa',local_files_only=True,trust_remote_code=False)
    if 'logits_to_keep' not in inspect.signature(model.forward).parameters:raise ValueError('Need compatible Qwen forward with logits_to_keep')
    model.to(device);model.eval();model.config.use_cache=not training
    return model,tok


def attach_adapter(model,rank=32,alpha=64):
    import torch
    from peft import LoraConfig,get_peft_model
    model=get_peft_model(model,LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=0.,bias='none',
                        task_type='CAUSAL_LM',target_modules=TARGETS))
    model.enable_input_require_grads();model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    for m in model.modules():
        if isinstance(m,torch.nn.Dropout):m.p=0.
    if hasattr(model.config,'attention_dropout'):model.config.attention_dropout=0.
    for p in model.parameters():
        if p.requires_grad:p.data=p.data.float()
    model.train();return model


def adapter_witness(model):
    """Small deterministic probe of every trainable tensor, not a full checksum."""
    import torch
    values=[]
    for name,p in model.named_parameters():
        if p.requires_grad:
            flat=p.detach().reshape(-1);i=torch.linspace(0,max(0,flat.numel()-1),min(8,flat.numel()),device=flat.device).long()
            values.extend(flat[i].float().cpu().tolist())
    return values


def export_merged(model,tok,path: str,report: dict,probes: list[list[int]],atol: float=.2):
    import torch
    dst=Path(path);tmp=dst.with_name(dst.name+'.staging')
    if dst.exists() or tmp.exists():raise FileExistsError('Export exists; no silent overwrite')
    tmp.mkdir(parents=True);model.eval();model.gradient_checkpointing_disable()
    probes=[p[-512:] for p in probes[:3]]
    if not probes:raise ValueError('Missing export probes')
    dev=next(model.parameters()).device
    def logits():
        with torch.no_grad():
            return [model(input_ids=torch.tensor([x],device=dev),logits_to_keep=1,use_cache=False).logits[0,-1].float().cpu() for x in probes]
    before=logits();model=model.merge_and_unload(safe_merge=True);after=logits()
    maxerr=max(float((a-b).abs().max()) for a,b in zip(before,after))
    if maxerr>atol or any(not torch.isfinite(x).all() for x in after):
        atomic_json(tmp/'FAILED.json',{'error':'merge_parity','max_abs':maxerr});raise RuntimeError('Merge parity failed; adapter checkpoint preserved')
    model.save_pretrained(tmp,safe_serialization=True,max_shard_size='2GB');tok.save_pretrained(tmp)
    atomic_json(tmp/'reload_probes.json',{'ids':probes,'logits':[x.tolist() for x in after]})
    atomic_json(tmp/'training_report.json',report)
    names={p.name:file_hash(p) for p in tmp.iterdir() if p.is_file()}
    m={'format':'arc-grid-rl-export-3','codec':CODEC_VERSION,'architecture':'qwen3-grid16',
       'files':names,'weights_id':json_hash({n:h for n,h in names.items() if n.endswith('.safetensors')}),
       'reload_verified':False,'merge_max_abs':maxerr,'training':report,'score_claim':None}
    atomic_json(tmp/'grid_rl_manifest.json',m);(tmp/'COMPLETE').write_text('Independent reload still required. No score claim.\n')
    os.replace(tmp,dst);return m


def check_export(path: str, require_rl: bool=True, require_reload: bool=True):
    root=Path(path);m=json.loads((root/'grid_rl_manifest.json').read_text())
    if m.get('format')!='arc-grid-rl-export-3' or m.get('codec')!=CODEC_VERSION or not (root/'COMPLETE').exists():raise ValueError('Not a completed grid-RL export')
    for n,h in m['files'].items():
        if Path(n).name!=n or file_hash(root/n)!=h:raise ValueError('Export file mismatch: '+n)
    if m['weights_id']!=json_hash({n:h for n,h in m['files'].items() if n.endswith('.safetensors')}):raise ValueError('Weight identity mismatch')
    model_contract(root,'grid')
    if m.get('training')!=json.loads((root/'training_report.json').read_text()):raise ValueError('Training manifest/report disagreement')
    if require_reload and not m.get('reload_verified'):raise ValueError('Independent reload verification missing')
    if require_rl and m['training'].get('rl_optimizer_steps',0)<1:raise ValueError('No model-level RL updates completed')
    return m


def verify_export(path: str,device='cuda:0',atol=.2):
    import torch
    m=check_export(path,require_rl=False,require_reload=False);root=Path(path)
    model,tok=load_grid_model(path,device);probe=json.loads((root/'reload_probes.json').read_text());errors=[]
    with torch.no_grad():
        for ids,expected in zip(probe['ids'],probe['logits']):
            actual=model(input_ids=torch.tensor([ids],device=device),logits_to_keep=1,use_cache=False).logits[0,-1].float().cpu()
            if not torch.isfinite(actual).all():raise FloatingPointError('Nonfinite reload logits')
            errors.append(float((actual-torch.tensor(expected)).abs().max()))
    r={'passed':max(errors)<=atol,'max_abs_errors':errors,'weights_id':m['weights_id'],'token_contract':'PASS'}
    atomic_json(root/'reload_verification.json',r)
    if not r['passed']:raise RuntimeError('Reload parity failed')
    m['reload_verified']=True;atomic_json(root/'grid_rl_manifest.json',m);return r


In [ ]:
%%writefile grid_rl/policy.py
"""One-update on-policy RLOO with exact-grid reward and explicit grammar policy.
The behavior AND differentiated policy use the same legal-token mask, T=1,
no top-k/top-p filtering. Shape is not provided from the target.
"""
from __future__ import annotations
import time
import torch
import torch.nn.functional as F
from .codec import GridGrammar, allowed_masks, decode_reply, PAD


def rloo_advantages(rewards: torch.Tensor) -> torch.Tensor:
    if rewards.ndim!=1 or rewards.numel()<2 or not torch.isfinite(rewards).all():
        raise ValueError('Need >=2 finite same-prompt rewards')
    return (rewards-rewards.mean())*(rewards.numel()/(rewards.numel()-1))


def policy_stats(logits: torch.Tensor, tokens: list[int]) -> tuple[torch.Tensor,torch.Tensor,torch.Tensor]:
    if logits.shape!=(len(tokens),16) or not tokens: raise ValueError('Grid logit alignment mismatch')
    mask=torch.tensor(allowed_masks(tokens),device=logits.device,dtype=torch.bool)
    lp=F.log_softmax(logits.float().masked_fill(~mask,float('-inf')),dim=-1)
    labels=torch.tensor(tokens,device=logits.device,dtype=torch.long)
    chosen=lp.gather(1,labels[:,None]).squeeze(1)
    if not torch.isfinite(chosen).all(): raise FloatingPointError('Nonfinite selected log probabilities')
    return chosen,lp,mask


def rloo_loss(logits: torch.Tensor, reference_logits: torch.Tensor, tokens: list[int],
              advantage: float, group_size: int, max_completion: int=930, beta: float=.02):
    """Exact categorical KL over the 16-token grammar support; no sampled-KL cap.
    Fixed max-length normalization is a deliberate variant of standard RLOO.
    """
    if group_size<2 or not 0<len(tokens)<=max_completion: raise ValueError('Invalid group/length')
    chosen,lp,mask=policy_stats(logits,tokens)
    _,ref,_=policy_stats(reference_logits.detach(),tokens)
    delta=(lp-ref).masked_fill(~mask,0.)
    kl=(lp.exp()*delta).sum(-1)
    loss=(-float(advantage)*chosen+beta*kl).sum()/(group_size*max_completion)
    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite RL loss')
    return loss,{'kl':float(kl.detach().mean()),'tokens':len(tokens)}


def completion_logits(model,prompt: list[int],reply: list[int],device) -> torch.Tensor:
    if not prompt or not reply: raise ValueError('Empty prompt/completion')
    ids=torch.tensor([prompt+reply],dtype=torch.long,device=device)
    c=len(reply)
    out=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,logits_to_keep=c+1)
    logits=out.logits[0,-c-1:-1,:]
    if logits.shape!=(c,16): raise ValueError('Expected aligned 16-token logits')
    return logits


def supervised_loss(model,prompt,reply,device):
    # Full 16-token vocabulary CE, including grammar/control errors: mirrors the
    # downstream language model's likelihood rather than only the masked sampler.
    logits=completion_logits(model,prompt,reply,device)
    return F.cross_entropy(logits.float(),torch.tensor(reply,device=device))


def sample_group(model,prompt: list[int],count: int,deadline: float,device,
                 max_tokens: int=930,microbatch: int=2,greedy: bool=False):
    if not prompt or count<1 or microbatch<1 or not 1<=max_tokens<=930: raise ValueError('Bad rollout config')
    was_training=model.training;checkpoint=bool(getattr(model,'is_gradient_checkpointing',False))
    if checkpoint:model.gradient_checkpointing_disable()
    model.eval(); results=[];calls=0
    try:
        while len(results)<count and time.monotonic()<deadline:
            b=min(microbatch,count-len(results)); states=[GridGrammar() for _ in range(b)]
            seqs=[[] for _ in range(b)]; probs=[[] for _ in range(b)]
            ids=torch.tensor([prompt]*b,device=device,dtype=torch.long)
            attn=torch.ones_like(ids);past=None;expired=False
            with torch.no_grad():
                for _ in range(max_tokens):
                    if time.monotonic()>=deadline:expired=True;break
                    pos=(attn.cumsum(-1)-1).clamp_min(0)
                    if past is not None:pos=pos[:,-1:]
                    out=model(input_ids=ids,attention_mask=attn,position_ids=pos,past_key_values=past,
                              use_cache=True,logits_to_keep=1)
                    past=out.past_key_values;logits=out.logits[:,-1,:].float();calls+=1
                    if logits.shape!=(b,16):raise ValueError('Rollout uses wrong vocabulary')
                    mask=torch.zeros_like(logits,dtype=torch.bool)
                    for i,s in enumerate(states):mask[i,list(s.allowed() if not s.ended else (PAD,))]=True
                    lp=F.log_softmax(logits.masked_fill(~mask,float('-inf')),dim=-1)
                    if not torch.isfinite(lp.exp()).all():raise FloatingPointError('Rollout nonfinite')
                    nxt=lp.argmax(-1) if greedy else torch.multinomial(lp.exp(),1).squeeze(1)
                    active=[not s.ended for s in states]
                    # Transfer one token/log-probability vector, rather than
                    # synchronizing separately for every scalar of every row.
                    next_tokens=nxt.tolist()
                    next_logps=lp.gather(1,nxt[:,None]).squeeze(1).tolist()
                    for i,s in enumerate(states):
                        if active[i]:
                            token=next_tokens[i];seqs[i].append(token);probs[i].append(next_logps[i]);s.consume(token)
                    if all(s.ended for s in states):
                        del out,logits,lp
                        break
                    ids=nxt[:,None]
                    attn=torch.cat([attn,torch.tensor(active,device=device,dtype=attn.dtype)[:,None]],dim=1)
                    del out,logits,lp
            for tokens,logps,s in zip(seqs,probs,states):
                results.append({'tokens':tokens,'behavior_logps':logps,'complete':s.ended,
                                'deadline_interrupted':expired,'stop':'eos' if s.ended else 'deadline' if expired else 'length',
                                'grid':decode_reply(tokens) if s.ended else None})
            del ids,attn,past
            if expired:break
    finally:
        if checkpoint:model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        model.train(was_training)
    return results,{'model_forward_calls':calls,'returned':len(results),'requested':count,
                    'sampling':'same grammar mask; temperature=1; no top-k/top-p; no target shape'}


In [ ]:
%%writefile grid_rl/probe.py
"""Disposable real-Qwen smoke test. One optimizer step, no exported weights or score."""
import argparse,json,time
from .common import atomic_json
from .data import load_split,episode
from .modeling import load_grid_model,attach_adapter,adapter_witness
from .policy import supervised_loss,sample_group,completion_logits,policy_stats


def probe(model_path,corpus_dir,output,device='cuda:0'):
    import torch
    if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable: real-Qwen probe NOT_RUN')
    rows,_=load_split(corpus_dir,'train');rows=sorted(rows,key=lambda r:sum(len(p['input'])*len(p['input'][0]) for p in r['task']['train']))
    e=episode(rows[0],19,False,7000);model,tok=load_grid_model(model_path,device,True);model=attach_adapter(model)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-6)
    before=adapter_witness(model);start=time.monotonic();loss=supervised_loss(model,e['prompt'],e['target'],device)
    if not torch.isfinite(loss):raise FloatingPointError('Nonfinite CE')
    loss.backward();norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.,error_if_nonfinite=True)
    opt.step();opt.zero_grad(set_to_none=True);after=adapter_witness(model)
    delta=max(abs(a-b) for a,b in zip(after,before))
    if delta<=0:raise RuntimeError('Probe could not observe a trainable-parameter change')
    group,diag=sample_group(model,e['prompt'],2,time.monotonic()+90,device,microbatch=1)
    errors=[]
    for g in group:
        if g['tokens']:
            with torch.no_grad():x=completion_logits(model,e['prompt'],g['tokens'],device);lp,_,_=policy_stats(x,g['tokens'])
            errors.append(float((lp.cpu()-torch.tensor(g['behavior_logps'])).abs().max()))
    if not errors or max(errors)>.25:raise RuntimeError('Sampler/scoring parity probe failed')
    r={'passed':True,'scope':'disposable single-record real-model smoke test; NOT accuracy or full-memory benchmark',
       'loss':float(loss.detach()),'grad_norm':float(norm),'parameter_witness_delta':delta,
       'max_sampler_recompute_error':max(errors),'seconds':time.monotonic()-start,
       'peak_memory_gib':torch.cuda.max_memory_allocated()/1024**3,'score':None,**diag}
    atomic_json(output,r);return r

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--model',required=True);p.add_argument('--corpus',required=True);p.add_argument('--output',required=True);a=p.parse_args()
    print(json.dumps(probe(a.model,a.corpus,a.output),indent=2))


In [ ]:
%%writefile grid_rl/train.py
"""Single-device direct-grid fine-tuning; never runs inside hidden inference.
Use fixed-vocabulary sft139, not the language/program base. Exact reward only.
Uninformative groups do not count as RL updates. Limited CE rescue is separately
counted and cannot make an export qualify as an RL-trained checkpoint.
"""
from __future__ import annotations
import argparse,gc,json,math,os,random,time
from pathlib import Path
import torch
from .common import atomic_json,append_event,file_hash,json_hash,environment_report,stable_seed
from .codec import CODEC_VERSION,decode_reply,MAX_GRID_TOKENS
from .data import load_split,episode
from .modeling import load_grid_model,attach_adapter,source_fingerprint,adapter_witness,export_merged
from .policy import sample_group,completion_logits,policy_stats,rloo_advantages,rloo_loss,supervised_loss

DEFAULTS={'mode':'rl','base_model_path':'','corpus_dir':'','output_dir':'','device':'cuda:0',
 'seed':20260907,'rank':32,'alpha':64,'learning_rate':5e-7,'sft_learning_rate':1e-6,
 'max_updates':96,'max_groups':192,'max_hours':2.,'group_size':4,'rollout_microbatch':2,
 'max_prompt_tokens':7000,'max_completion_tokens':930,'beta':.02,'grad_clip':1.,'weight_decay':.0,
 'max_zero_groups':32,'rescue_every':4,'max_rescue_updates':8,'rescue_lr':2e-7,
 'replay_every':4,'replay_weight':.05,'save_every':8,'resume_from':'',
 'rollout_recompute_tolerance':.25,'max_grad_norm_abort':100.,'augment':True}


def load_config(path):
    given=json.loads(Path(path).read_text());unknown=set(given)-set(DEFAULTS)
    if unknown:raise ValueError('Unknown settings: '+str(sorted(unknown)))
    c={**DEFAULTS,**given}
    if c['mode'] not in ('rl','sft_control'):raise ValueError('Unsupported training mode')
    for k in ('rank','alpha','max_updates','max_groups','group_size','rollout_microbatch','max_prompt_tokens','max_completion_tokens','max_zero_groups','save_every'):
        if type(c[k])is not int or c[k]<1:raise ValueError('Invalid positive integer: '+k)
    if c['group_size']<2:raise ValueError('RLOO needs a group of at least two')
    if c['max_completion_tokens']!=930:raise ValueError('Use 930 tokens so all valid 30x30 shapes remain representable')
    if c['max_prompt_tokens']+930>8192:raise ValueError('Context exceeds frozen deployment length')
    if c['max_hours']<=0 or c['rollout_microbatch']>c['group_size']:raise ValueError('Invalid budget')
    for k in ('learning_rate','sft_learning_rate','rescue_lr','grad_clip','max_grad_norm_abort','rollout_recompute_tolerance'):
        if not math.isfinite(c[k]) or c[k]<=0:raise ValueError('Invalid float '+k)
    for k in ('beta','replay_weight','weight_decay'):
        if not math.isfinite(c[k]) or c[k]<0:raise ValueError('Invalid nonnegative float '+k)
    for k in ('rescue_every','max_rescue_updates','replay_every'):
        if type(c[k])is not int or c[k]<0:raise ValueError('Invalid nonnegative integer '+k)
    return c


def run_identity(c,manifest,base):
    mutable={'resume_from','max_hours','output_dir','max_groups','max_updates','base_model_path','corpus_dir','device'}
    impl={p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')}
    return json_hash({'config':{k:v for k,v in c.items() if k not in mutable},'data':manifest['dataset_hash'],
                      'base_files':base['files'],'implementation':impl})


def optimizer(model,c):
    ps=[p for p in model.parameters() if p.requires_grad]
    if not ps:raise ValueError('No trainable parameters')
    kw=dict(lr=c['learning_rate'],betas=(.9,.95),eps=1e-8,weight_decay=c['weight_decay'])
    if all(p.is_cuda for p in ps):kw['fused']=True
    try:return torch.optim.AdamW(ps,**kw)
    except (TypeError,RuntimeError):kw.pop('fused',None);return torch.optim.AdamW(ps,**kw)


def write_checkpoint(model,tok,opt,root,identity,stats,cursor,probes):
    root=Path(root);root.mkdir(exist_ok=True,parents=True)
    name=f"step-{stats['optimizer_steps']:06d}-episode-{cursor:06d}";dst=root/name;tmp=root/(name+'.staging')
    if dst.exists():return dst
    if tmp.exists():raise FileExistsError('Preserved partial checkpoint: '+str(tmp))
    model.save_pretrained(tmp,safe_serialization=True);tok.save_pretrained(tmp)
    torch.save({'optimizer':opt.state_dict(),'torch_rng':torch.get_rng_state(),
                'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},tmp/'state.pt')
    meta={'identity':identity,'stats':stats,'cursor':cursor,'probes':probes,
          'files':{n:file_hash(tmp/n) for n in ('adapter_model.safetensors','adapter_config.json','state.pt')}}
    atomic_json(tmp/'state.json',meta);(tmp/'COMPLETE').write_text('Local training state\n');os.replace(tmp,dst)
    atomic_json(root/'latest.json',{'path':str(dst.resolve())});return dst


def read_checkpoint(path,identity,model,opt):
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    p=Path(path);m=json.loads((p/'state.json').read_text())
    if not (p/'COMPLETE').exists() or m['identity']!=identity:raise ValueError('Resume identity mismatch')
    for n,h in m['files'].items():
        if Path(n).name!=n or file_hash(p/n)!=h:raise ValueError('Resume checksum mismatch')
    set_peft_model_state_dict(model,load_file(str(p/'adapter_model.safetensors')))
    state=torch.load(p/'state.pt',map_location='cpu',weights_only=True);opt.load_state_dict(state['optimizer'])
    torch.set_rng_state(state['torch_rng'])
    if torch.cuda.is_available() and state['cuda_rng']:torch.cuda.set_rng_state_all(state['cuda_rng'])
    return m['stats'],m['cursor'],m['probes']


def update(opt,model,c,lr):
    ps=[p for p in model.parameters() if p.requires_grad]
    norm=torch.nn.utils.clip_grad_norm_(ps,c['grad_clip'],error_if_nonfinite=True)
    if float(norm)>c['max_grad_norm_abort']:raise FloatingPointError('Gradient norm crossed abort threshold')
    if float(norm)==0:return {'applied':False,'gradient_norm':0.,'witness_delta':0.}
    before=adapter_witness(model)
    for g in opt.param_groups:g['lr']=lr
    opt.step();after=adapter_witness(model)
    delta=max((abs(a-b) for a,b in zip(after,before)),default=0.)
    return {'applied':True,'gradient_norm':float(norm),'witness_delta':delta}


def train(c):
    out=Path(c['output_dir']);out.mkdir(parents=True,exist_ok=True)
    atomic_json(out/'environment.json',environment_report());atomic_json(out/'config.json',c)
    if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable; no Qwen fine-tuning occurred')
    torch.manual_seed(c['seed']);torch.cuda.manual_seed_all(c['seed'])
    rows,manifest=load_split(c['corpus_dir'],'train')
    if not rows:raise ValueError('No training records')
    base=source_fingerprint(c['base_model_path']);identity=run_identity(c,manifest,base)
    old=out/'run_identity.json'
    if old.exists() and json.loads(old.read_text())['identity']!=identity:raise ValueError('Use a new experiment directory')
    if old.exists() and not c['resume_from']:raise ValueError('Existing experiment; specify resume_from or use a fresh directory')
    atomic_json(old,{'identity':identity});atomic_json(out/'base_model_audit.json',base)
    model,tok=load_grid_model(c['base_model_path'],c['device'],True);model=attach_adapter(model,c['rank'],c['alpha']);opt=optimizer(model,c)
    random.Random(c['seed']).shuffle(rows)
    stats={'optimizer_steps':0,'rl_optimizer_steps':0,'sft_control_steps':0,'rescue_ce_steps':0,
           'rollout_groups':0,'exact_training_rollouts':0,'total_training_rollouts':0,
           'zero_variance_groups':0,'prompt_skips':0,'forward_calls':0,'max_witness_delta':0.,'consecutive_zero_groups':0}
    cursor=0;probes=[];initial_witness=adapter_witness(model)
    if c['resume_from']:stats,cursor,probes=read_checkpoint(c['resume_from'],identity,model,opt)
    start_rl=stats['rl_optimizer_steps'];start_opt=stats['optimizer_steps'];start_cursor=cursor
    deadline=time.monotonic()+c['max_hours']*3600;event=out/'events.jsonl';reason='budget_complete';begin=time.monotonic()
    def report(status):
        r={**stats,'status':status,'identity':identity,'data_hash':manifest['dataset_hash'],
           'base_weights_id':base['weights_id'],'mode':c['mode'],'cursor':cursor,
           'rl_updates_this_session':stats['rl_optimizer_steps']-start_rl,
           'optimizer_updates_this_session':stats['optimizer_steps']-start_opt,
           'elapsed_seconds_this_session':time.monotonic()-begin,'stop_reason':reason,
           'score_claim':None,'target_37_measured':False,'trained_on_public_evaluation':False,
           'holdout_status':manifest['holdout_status'],'codec':CODEC_VERSION,
           'objective':'single-update RLOO; exact grid reward; fixed-length normalization; exact grammar KL; separately counted CE rescue',
           'sampling':'variable shape, no target dimensions, same policy mask in rollout and loss'}
        atomic_json(out/'training_report.json',r);return r
    try:
        while stats['optimizer_steps']<c['max_updates'] and cursor-start_cursor<c['max_groups'] and time.monotonic()<deadline:
            record=rows[cursor%len(rows)];ep_seed=stable_seed(c['seed'],cursor,record['record_id']);cursor+=1
            try:e=episode(record,ep_seed,c['augment'],c['max_prompt_tokens'])
            except ValueError as ex:
                stats['prompt_skips']+=1;append_event(event,'episode_skipped',record_id=record['record_id'],error=str(ex))
                if stats['prompt_skips']>=len(rows):reason='too_many_context_skips';break
                continue
            if len(probes)<3:probes.append(e['prompt'][-480:]+e['target'][:32])
            opt.zero_grad(set_to_none=True)
            if c['mode']=='sft_control':
                loss=supervised_loss(model,e['prompt'],e['target'],c['device']);loss.backward()
                u=update(opt,model,c,c['sft_learning_rate'])
                if u['applied']:stats['optimizer_steps']+=1;stats['sft_control_steps']+=1
                stats['max_witness_delta']=max(stats['max_witness_delta'],u['witness_delta'])
                append_event(event,'sft_control_step',record_id=e['record_id'],loss=float(loss.detach()),**u)
            else:
                group,diag=sample_group(model,e['prompt'],c['group_size'],deadline,c['device'],c['max_completion_tokens'],c['rollout_microbatch'])
                stats['forward_calls']+=diag['model_forward_calls']
                if len(group)!=c['group_size'] or any(g['deadline_interrupted'] for g in group):reason='incomplete_deadline_group';break
                rewards=torch.tensor([float(g['complete'] and g['grid']==e['target_grid']) for g in group])
                stats['rollout_groups']+=1;stats['exact_training_rollouts']+=int(rewards.sum());stats['total_training_rollouts']+=len(group)
                append_event(out/'rollouts.jsonl','training_rollout',record_id=e['record_id'],family_id=e['family_id'],
                    seed=ep_seed,kind=e['kind'],prompt_hash=json_hash(e['prompt']),target_grid=e['target_grid'],
                    reward_source='known TRAINING query; never public evaluation',
                    samples=[{'tokens':g['tokens'],'grid':g['grid'],'stop':g['stop'],'reward':float(rewards[j]),
                              'mean_behavior_logp':sum(g['behavior_logps'])/max(1,len(g['behavior_logps']))}
                             for j,g in enumerate(group)])
                adv=rloo_advantages(rewards)
                append_event(event,'rollout_group',record_id=e['record_id'],family=e['family_id'],kind=e['kind'],
                             rewards=rewards.tolist(),lengths=[len(g['tokens']) for g in group],stops=[g['stop'] for g in group],
                             transform=e['transform'],rollout_digest=json_hash([g['tokens'] for g in group]),**diag)
                if float(adv.abs().max())==0:
                    stats['zero_variance_groups']+=1;stats['consecutive_zero_groups']+=1
                    # A separate supervised update from a known TRAIN label, not
                    # a fabricated RL advantage or target inserted in a rollout.
                    if rewards.sum()==0 and c['rescue_every'] and stats['consecutive_zero_groups']%c['rescue_every']==0 and stats['rescue_ce_steps']<c['max_rescue_updates']:
                        loss=supervised_loss(model,e['prompt'],e['target'],c['device']);loss.backward()
                        u=update(opt,model,c,c['rescue_lr'])
                        if u['applied']:stats['optimizer_steps']+=1;stats['rescue_ce_steps']+=1
                        stats['max_witness_delta']=max(stats['max_witness_delta'],u['witness_delta'])
                        append_event(event,'supervised_rescue_not_rl',record_id=e['record_id'],loss=float(loss.detach()),**u)
                    if stats['consecutive_zero_groups']>=c['max_zero_groups']:reason='no_informative_reward';break
                    report('IN_PROGRESS_NO_SCORE')
                    continue
                stats['consecutive_zero_groups']=0;errors=[];kls=[]
                # Recompute sequentially: one full base remains resident. No old-policy
                # copy is needed because exactly one update follows each rollout group.
                for i,g in enumerate(group):
                    if not g['tokens']:continue
                    with torch.no_grad(),model.disable_adapter():
                        ref=completion_logits(model,e['prompt'],g['tokens'],c['device']).detach()
                    current=completion_logits(model,e['prompt'],g['tokens'],c['device'])
                    chosen,_,_=policy_stats(current,g['tokens'])
                    err=float((chosen.detach().cpu()-torch.tensor(g['behavior_logps'])).abs().max());errors.append(err)
                    if err>c['rollout_recompute_tolerance']:raise RuntimeError('Behavior/recompute log-probability mismatch; update rejected')
                    loss,d=rloo_loss(current,ref,g['tokens'],float(adv[i]),c['group_size'],c['max_completion_tokens'],c['beta'])
                    loss.backward();kls.append(d['kl']);del current,ref,chosen,loss
                if c['replay_every'] and (stats['rl_optimizer_steps']+1)%c['replay_every']==0:
                    replay=rows[(cursor+17)%len(rows)]
                    try:
                        ep=episode(replay,stable_seed('replay',cursor,c['seed']),c['augment'],c['max_prompt_tokens'])
                        ce=supervised_loss(model,ep['prompt'],ep['target'],c['device']);(c['replay_weight']*ce).backward()
                        append_event(event,'interleaved_ce_regularization',record_id=ep['record_id'],loss=float(ce.detach()),weight=c['replay_weight'])
                    except ValueError:append_event(event,'replay_skipped_context')
                u=update(opt,model,c,c['learning_rate'])
                if u['applied']:stats['optimizer_steps']+=1;stats['rl_optimizer_steps']+=1
                stats['max_witness_delta']=max(stats['max_witness_delta'],u['witness_delta'])
                append_event(event,'rl_optimizer_step',step=stats['rl_optimizer_steps'],mean_reward=float(rewards.mean()),
                             mean_kl=sum(kls)/max(1,len(kls)),max_recompute_error=max(errors,default=0),**u)
            report('IN_PROGRESS_NO_SCORE')
            print(json.dumps({'updates':stats['optimizer_steps'],'rl_updates':stats['rl_optimizer_steps'],
                              'rescue_ce':stats['rescue_ce_steps'],'groups':stats['rollout_groups']}),flush=True)
            if stats['optimizer_steps'] and stats['optimizer_steps']%c['save_every']==0:
                write_checkpoint(model,tok,opt,out/'checkpoints',identity,stats,cursor,probes)
        ck=write_checkpoint(model,tok,opt,out/'checkpoints',identity,stats,cursor,probes)
        r=report('TRAINED_CANDIDATE_NOT_SCORED' if stats['rl_optimizer_steps'] or stats['sft_control_steps'] else 'NO_RL_UPDATE_COMPLETED')
        r['checkpoint']=str(ck);atomic_json(out/'training_report.json',r)
        eligible=stats['rl_optimizer_steps']>0 if c['mode']=='rl' else stats['sft_control_steps']>0
        if eligible:
            del opt;gc.collect();torch.cuda.empty_cache()
            export_merged(model,tok,str(out/'grid_rl_merged'),r,probes)
        return r
    except Exception as ex:
        reason=type(ex).__name__+':'+str(ex)
        append_event(event,'training_failed',error=reason)
        report('FAILED_NO_SCORE')
        # Gradients are discarded; parameters include only completed opt.step calls.
        opt.zero_grad(set_to_none=True)
        try:write_checkpoint(model,tok,opt,out/'checkpoints',identity,stats,cursor,probes)
        except Exception as save_exc:append_event(event,'emergency_checkpoint_failed',error=str(save_exc))
        raise

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('command',choices=['train','verify-export']);p.add_argument('--config');p.add_argument('--model');a=p.parse_args()
    if a.command=='train':print(json.dumps(train(load_config(a.config)),indent=2))
    else:
        from .modeling import verify_export
        print(json.dumps(verify_export(a.model),indent=2))


In [ ]:
%%writefile arc_exec_rl/__init__.py
"""Legacy atomic-write compatibility only; no program-model lane."""


In [ ]:
%%writefile arc_exec_rl/common.py
from __future__ import annotations
import hashlib
import json
import math
import os
import platform
import tempfile
import time
from pathlib import Path
from typing import Any


def truthy(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes", "on"}


def stable_seed(*parts: Any) -> int:
    return int(hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:8], 16)


def json_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def file_hash(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path: str | Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".writing-", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(value, f, separators=(",", ":"), sort_keys=True, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
        with open(tmp, encoding="utf-8") as f:
            json.load(f)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)


def append_event(path: str | Path, event: str, **fields: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Each process uses its own event file. No cross-process buffering assumptions.
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"event": event, "time": time.time(), **fields}, allow_nan=False) + "\n")
        f.flush()


def validate_grid(g: Any) -> list[list[int]]:
    # Strict: floats, booleans and stringified arrays are not accepted.
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        raise ValueError("Grid must have 1..30 rows")
    if not isinstance(g[0], list) or not 1 <= len(g[0]) <= 30:
        raise ValueError("Grid must have 1..30 columns")
    w = len(g[0])
    if any(not isinstance(row, list) or len(row) != w for row in g):
        raise ValueError("Ragged grid")
    if any(type(x) is not int or not 0 <= x <= 9 for row in g for x in row):
        raise ValueError("Cells must be integer colors 0..9")
    return g


def public_task(task: dict) -> dict:
    """The only view supplied to inference: all known demos, test INPUTS only."""
    if not task.get("train") or not task.get("test"):
        raise ValueError("Task requires demonstrations and test inputs")
    return {"train": [{"input": validate_grid(p["input"]), "output": validate_grid(p["output"])}
                      for p in task["train"]],
            "test": [{"input": validate_grid(p["input"])} for p in task["test"]]}


def validate_submission(tasks: dict, submission: dict) -> None:
    if set(tasks) != set(submission):
        raise ValueError("Submission IDs differ from challenge IDs")
    for key, task in tasks.items():
        pairs = submission[key]
        if not isinstance(pairs, list) or len(pairs) != len(task["test"]):
            raise ValueError(f"Wrong number of test records: {key}")
        for p in pairs:
            if set(p) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"Invalid attempt keys: {key}")
            validate_grid(p["attempt_1"])
            validate_grid(p["attempt_2"])
    # Duplicate attempts are legal. Do not invent a one-cell mutation.


def initial_submission(tasks: dict) -> dict:
    out = {k: [{"attempt_1": p["input"], "attempt_2": p["input"]} for p in public_task(t)["test"]]
           for k, t in tasks.items()}
    validate_submission(tasks, out)
    return out


def score_submission(tasks: dict, solutions: dict, submission: dict, candidates: dict | None = None) -> dict:
    validate_submission(tasks, submission)
    if set(tasks) != set(solutions):
        raise ValueError("Solutions must match the evaluated challenge subset exactly")
    first = either = fully = oracle = total = 0
    per_task = {}
    for key, task in tasks.items():
        ys = solutions[key]
        if len(ys) != len(task["test"]):
            raise ValueError(f"Solution count mismatch: {key}")
        wins = []
        for i, y in enumerate(ys):
            validate_grid(y)
            p = submission[key][i]
            a = p["attempt_1"] == y
            b = a or p["attempt_2"] == y
            all_grids = [p["attempt_1"], p["attempt_2"]]
            if candidates:
                all_grids += candidates.get(f"{key}:{i}", [])
            total += 1
            first += int(a)
            either += int(b)
            oracle += int(y in all_grids)
            wins.append(int(b))
        fully += int(all(wins))
        per_task[key] = {"pairs": len(wins), "exact": sum(wins), "fraction": sum(wins) / len(wins)}
    return {"tasks": len(tasks), "pairs": total, "attempt1_exact": first, "pass2_exact": either,
            "pair_micro_pass2": either / max(total, 1),
            "task_macro_pass2": sum(x["fraction"] for x in per_task.values()) / max(len(tasks), 1),
            "fully_solved_tasks": fully, "candidate_oracle_exact": oracle,
            "target_60_required_pairs": math.ceil(.60 * total),
            "target_60_pair_micro_met": either >= math.ceil(.60 * total),
            "per_task": per_task}


def environment_report() -> dict:
    import importlib.metadata as md
    r = {"python": platform.python_version(), "architecture": platform.machine(), "system": platform.platform()}
    for p in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "tokenizers"):
        try:
            r[p] = md.version(p)
        except md.PackageNotFoundError:
            r[p] = "NOT_INSTALLED"
    try:
        import torch
        r["cuda_available"] = torch.cuda.is_available()
        r["torch_cuda"] = torch.version.cuda
        r["gpus"] = [{"name": torch.cuda.get_device_name(i),
                      "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
                     for i in range(torch.cuda.device_count())]
    except ImportError:
        r["cuda_available"] = False
    return r


def score_banks(tasks: dict,solutions: dict,banks: dict) -> dict:
    """Abstention-aware scoring: no candidates cannot accidentally count as a solution."""
    total=first=either=oracle=fully=0; pt={}
    if set(tasks)!=set(solutions): raise ValueError("Mismatched solutions")
    for key,t in tasks.items():
        ys=solutions[key]
        if len(ys)!=len(t["test"]): raise ValueError("Mismatched test outputs")
        wins=[]
        for i,y in enumerate(ys):
            validate_grid(y); candidates=banks.get(f"{key}:{i}",[])
            for g in candidates: validate_grid(g)
            a=bool(candidates) and candidates[0]==y
            b=any(g==y for g in candidates[:2]); o=any(g==y for g in candidates)
            first+=int(a);either+=int(b);oracle+=int(o);total+=1;wins.append(int(b))
        fully+=int(all(wins));pt[key]={"pairs":len(wins),"exact":sum(wins),"fraction":sum(wins)/len(wins)}
    return {"tasks":len(tasks),"pairs":total,"attempt1_exact":first,"pass2_exact":either,
            "pair_micro_pass2":either/max(total,1),"task_macro_pass2":sum(v['fraction'] for v in pt.values())/max(len(pt),1),
            "fully_solved_tasks":fully,"candidate_oracle_exact":oracle,"per_task":pt,
            "target_60_required_pairs":math.ceil(.60*total),"target_60_pair_micro_met":either>=math.ceil(.60*total)}


def atomic_candidate_pickle(path: str | Path,value: Any) -> None:
    """Only for our own legacy worker outputs, never for untrusted downloaded pickles."""
    import bz2,pickle
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix='.candidate-',dir=p.parent);os.close(fd)
    try:
        with bz2.BZ2File(tmp,'wb') as f: pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp,p)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)


In [ ]:
%%writefile arc_dataset_guard.py
from __future__ import annotations

import hashlib
import json
import math
import os
import time
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from arc_symbolic import SymbolicSolver, as_grid, infer_output_shapes, valid_grid


def env_truthy(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "1" if default else "0").strip().lower()
    return raw not in ("", "0", "false", "no", "none")


def atomic_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    json.loads(tmp.read_text(encoding="utf-8"))
    os.replace(tmp, path)


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def find_competition_dir() -> Path:
    explicit = os.getenv("ARC_COMPETITION_DIR", "").strip()
    if explicit and Path(explicit).exists():
        return Path(explicit)
    direct = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    if direct.exists():
        return direct
    root = Path("/kaggle/input")
    for name in ("arc-agi_evaluation_challenges.json", "arc-agi_test_challenges.json"):
        hit = next(root.rglob(name), None) if root.exists() else None
        if hit is not None:
            return hit.parent
    raise FileNotFoundError("ARC-AGI-2 competition directory was not found")


def _load(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def _count_outputs(challenges: dict[str, Any]) -> int:
    return sum(len(task.get("test", [])) for task in challenges.values())


def dataset_audit(output_dir: str | Path, run_cpu_public_audit: bool = True) -> dict[str, Any]:
    start = time.time()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    comp = find_competition_dir()
    files = {
        name: comp / name
        for name in (
            "arc-agi_training_challenges.json",
            "arc-agi_training_solutions.json",
            "arc-agi_evaluation_challenges.json",
            "arc-agi_evaluation_solutions.json",
            "arc-agi_test_challenges.json",
            "sample_submission.json",
        )
        if (comp / name).exists()
    }
    hashes = {name: sha256_file(path) for name, path in files.items()}
    training = _load(files["arc-agi_training_challenges.json"]) if "arc-agi_training_challenges.json" in files else {}
    evaluation = _load(files["arc-agi_evaluation_challenges.json"]) if "arc-agi_evaluation_challenges.json" in files else {}
    test = _load(files["arc-agi_test_challenges.json"]) if "arc-agi_test_challenges.json" in files else {}

    shared_test_train_ids = sorted(set(test) & set(training))
    exact_test_train = sum(test[k] == training[k] for k in shared_test_train_ids)
    placeholder = bool(test and len(shared_test_train_ids) == len(test) and exact_test_train == len(test))
    report: dict[str, Any] = {
        "competition_dir": str(comp),
        "file_sha256": hashes,
        "training_tasks": len(training),
        "training_outputs": _count_outputs(training),
        "evaluation_tasks": len(evaluation),
        "evaluation_outputs": _count_outputs(evaluation),
        "test_tasks": len(test),
        "test_outputs": _count_outputs(test),
        "test_training_shared_ids": len(shared_test_train_ids),
        "test_training_exact_content": exact_test_train,
        "test_is_training_placeholder": placeholder,
        "warning": (
            "The local test file is a 240-task public-training placeholder. It is not hidden-score evidence; "
            "Kaggle replaces it during the competition rerun."
            if placeholder else "The test file is not the known public-training placeholder."
        ),
    }

    # Public diagnostic only. It never writes predictions or routes from solutions.
    if run_cpu_public_audit and evaluation and "arc-agi_evaluation_solutions.json" in files and not env_truthy("KAGGLE_IS_COMPETITION_RERUN"):
        solutions = _load(files["arc-agi_evaluation_solutions.json"])
        shape_hits = Counter()
        symbolic_top1 = 0
        symbolic_oracle = 0
        total = 0
        solver = SymbolicSolver(max_programs=int(os.getenv("ARC_PUBLIC_AUDIT_SYMBOLIC_PROGRAMS", "500")), max_outputs=12)
        for task_id, task in evaluation.items():
            for test_idx, test_ex in enumerate(task.get("test", [])):
                total += 1
                target = as_grid(solutions[task_id][test_idx])
                hypotheses = infer_output_shapes(task, as_grid(test_ex["input"]), max_shapes=12)
                shapes = [(h, w) for h, w, _, _ in hypotheses]
                for k in (1, 2, 3, 5, 8, 12):
                    shape_hits[k] += int(tuple(target.shape) in shapes[:k])
                candidates = solver.synthesize(task, test_idx)
                matches = [i for i, cand in enumerate(candidates) if np.array_equal(cand.grid, target)]
                symbolic_top1 += int(bool(matches) and matches[0] == 0)
                symbolic_oracle += int(bool(matches))
        report["public_cpu_audit"] = {
            "outputs": total,
            "shape_recall": {f"top{k}": int(shape_hits[k]) for k in (1, 2, 3, 5, 8, 12)},
            "shape_recall_percent": {f"top{k}": 100.0 * shape_hits[k] / max(1, total) for k in (1, 2, 3, 5, 8, 12)},
            "symbolic_top1_exact": symbolic_top1,
            "symbolic_union_exact": symbolic_oracle,
            "symbolic_promotion_allowed": False,
            "interpretation": (
                "Shape inference is useful as a constrained-decoding prior. The exact symbolic lane produced no "
                "public-evaluation saves in this audit, so it is not allowed to displace a neural attempt by default."
            ),
        }
    report["runtime_seconds"] = time.time() - start
    atomic_json(out / "dataset_audit.json", report)
    return report


def _model_candidates(explicit: str = "") -> list[Path]:
    values: list[Path] = []
    if explicit:
        values.append(Path(explicit))
    values.extend([
        Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        Path("/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
    ])
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            text = str(cfg).lower()
            if "qwen3_4b_grids15_sft139" in text:
                values.append(cfg.parent)
    unique: list[Path] = []
    seen: set[str] = set()
    for value in values:
        key = str(value.resolve()) if value.exists() else str(value)
        if key not in seen:
            seen.add(key)
            unique.append(value)
    return unique


def audit_grid16_model(path: str | Path, require_exact_ids: bool = True) -> dict[str, Any]:
    path = Path(path)
    report: dict[str, Any] = {"path": str(path), "passed": False, "errors": []}
    if not path.is_dir():
        report["errors"].append("directory_missing")
        return report
    config_path = path / "config.json"
    if not config_path.exists():
        report["errors"].append("config_missing")
        return report
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as exc:
        report["errors"].append(f"config_parse:{type(exc).__name__}")
        return report
    report["vocab_size"] = config.get("vocab_size")
    report["model_type"] = config.get("model_type")
    report["hidden_size"] = config.get("hidden_size")
    report["layers"] = config.get("num_hidden_layers")
    if int(config.get("vocab_size", -1)) != 16:
        report["errors"].append("vocab_size_not_16")
    weights = sorted(path.glob("*.safetensors")) + sorted(path.glob("pytorch_model*.bin"))
    report["weight_files"] = [p.name for p in weights]
    if not weights:
        report["errors"].append("weights_missing")
    try:
        from transformers import AutoTokenizer
        from arc_loader import ArcTokenSpec
        tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        spec = ArcTokenSpec.from_tokenizer(tokenizer)
        report["tokenizer_length"] = len(tokenizer)
        report["token_spec"] = {
            "digits": list(spec.digit_ids),
            "newline": spec.newline_id,
            "eos": spec.eos_id,
            "pad": spec.pad_id,
            "user_marker": list(spec.user_marker),
            "assistant_marker": list(spec.assistant_marker),
        }
        if len(tokenizer) != 16:
            report["errors"].append("tokenizer_length_not_16")
        if require_exact_ids:
            expected = {
                "digits": list(range(10)), "newline": 10, "eos": 15,
                "user_marker": [14, 11, 10], "assistant_marker": [14, 12, 10],
            }
            actual = report["token_spec"]
            for name, value in expected.items():
                if actual.get(name) != value:
                    report["errors"].append(f"token_semantics:{name}:{actual.get(name)}!={value}")
    except Exception as exc:
        report["errors"].append(f"tokenizer_contract:{type(exc).__name__}:{exc}")
    report["passed"] = not report["errors"]
    return report


def resolve_primary_model(output_dir: str | Path, override: str = "", allow_custom: bool = False, require_exact_ids: bool = True) -> tuple[str, dict[str, Any]]:
    if override and not allow_custom:
        raise RuntimeError("PRIMARY_MODEL_OVERRIDE was set while ALLOW_CUSTOM_PRIMARY=False")
    audits = []
    for candidate in _model_candidates(override if allow_custom else ""):
        audit = audit_grid16_model(candidate, require_exact_ids=require_exact_ids)
        audits.append(audit)
        if audit["passed"]:
            payload = {"selected": str(candidate), "audits": audits}
            atomic_json(Path(output_dir) / "model_audit.json", payload)
            return str(candidate), payload
    payload = {"selected": None, "audits": audits}
    atomic_json(Path(output_dir) / "model_audit.json", payload)
    raise RuntimeError("No exact 16-token Qwen3-4B sft139 checkpoint passed the model contract")


def _task_complexity(task: dict[str, Any]) -> dict[str, float]:
    train = task.get("train", [])
    tests = task.get("test", [])
    input_cells = [np.asarray(ex["input"]).size for ex in train + tests]
    output_cells = [np.asarray(ex["output"]).size for ex in train]
    shape_change = np.mean([
        tuple(np.asarray(ex["input"]).shape) != tuple(np.asarray(ex["output"]).shape)
        for ex in train
    ]) if train else 1.0
    colors = [len(np.unique(np.asarray(ex["input"]))) for ex in train + tests]
    return {
        "demos": float(len(train)),
        "tests": float(len(tests)),
        "mean_input_cells": float(np.mean(input_cells or [1])),
        "max_input_cells": float(max(input_cells or [1])),
        "mean_output_cells": float(np.mean(output_cells or [1])),
        "shape_change": float(shape_change),
        "mean_colors": float(np.mean(colors or [1])),
    }


def build_safe_routes(output_dir: str | Path, profile: str = "DATASET_SAFE_MAX") -> dict[str, Any]:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    comp = find_competition_dir()
    rerun = env_truthy("KAGGLE_IS_COMPETITION_RERUN")
    challenge_name = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
    challenges = _load(comp / challenge_name)
    train_q = comp / "arc-agi_training_challenges.json"
    training = _load(train_q) if train_q.exists() else {}
    # Routing may skip TTT only for byte-equivalent public-training tasks. The
    # more expensive D4/color-canonical retrieval remains available inside
    # starter.py, but never controls the task budget before it has produced an
    # actual exact answer.
    raw_training_index = {json.dumps(task, sort_keys=True, separators=(",", ":")): key for key, task in training.items()}
    routes: dict[str, Any] = {}
    action_counts: Counter[str] = Counter()
    for task_id, task in challenges.items():
        exact_source = raw_training_index.get(json.dumps(task, sort_keys=True, separators=(",", ":")))
        exact_all = exact_source is not None
        stats = _task_complexity(task)
        test_grid = as_grid(task["test"][0]["input"]) if task.get("test") else np.zeros((1, 1), dtype=np.int8)
        shapes = infer_output_shapes(task, test_grid, max_shapes=8)
        top_shape_score = float(shapes[0][2]) if shapes else 0.0
        high_shape_confidence = top_shape_score >= 5.0

        if exact_all:
            augments, action = 0, "skip_exact_canonical_retrieval"
        elif profile.upper() == "PARITY_FULL":
            augments, action = 15, "full_128_parity"
        elif profile.upper() == "COVERAGE_FAST":
            if stats["shape_change"] == 0.0 and stats["max_input_cells"] <= 225 and high_shape_confidence:
                augments, action = 3, "light_32_simple"
            else:
                augments, action = 7, "standard_64_coverage"
        else:
            hard = (
                stats["shape_change"] > 0.0
                or stats["max_input_cells"] > 400
                or stats["mean_colors"] >= 5.0
                or not high_shape_confidence
            )
            if hard:
                augments, action = 15, "full_128_hard"
            else:
                augments, action = 7, "standard_64_simple"
        action_counts[action] += 1
        routes[task_id] = {
            "train_augments": int(augments),
            "skip_ttt": bool(exact_all),
            "exact_retrieval": bool(exact_all),
            "exact_training_source": exact_source,
            "action": action,
            "stats": stats,
            "shape_top_score": top_shape_score,
            "shape_hypotheses": [[int(h), int(w), float(score), reason] for h, w, score, reason in shapes[:5]],
            "reason": "Dataset-aware static prior; final cap is recomputed from remaining wall time inside each GPU worker.",
        }
    atomic_json(out / "safe_routes.json", routes)
    report = {
        "profile": profile,
        "challenge_file": challenge_name,
        "tasks": len(routes),
        "outputs": _count_outputs(challenges),
        "action_counts": dict(action_counts),
        "route_path": str(out / "safe_routes.json"),
        "uses_evaluation_solutions": False,
        "symbolic_confidence_reduces_ttt": False,
    }
    atomic_json(out / "safe_route_report.json", report)
    return report


def make_initial_submission(output_path: str | Path) -> dict[str, Any]:
    comp = find_competition_dir()
    rerun = env_truthy("KAGGLE_IS_COMPETITION_RERUN")
    challenge_name = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
    challenges = _load(comp / challenge_name)
    submission: dict[str, Any] = {}
    for task_id, task in challenges.items():
        rows = []
        for test_ex in task.get("test", []):
            grid = as_grid(test_ex["input"])
            second = grid.T.copy()
            if np.array_equal(grid, second):
                second = np.full(grid.shape, (int(grid[0, 0]) + 1) % 10, dtype=np.int8)
            rows.append({"attempt_1": grid.tolist(), "attempt_2": second.tolist()})
        submission[task_id] = rows
    atomic_json(output_path, submission)
    return submission


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", default="/kaggle/working/arc60_dataset_safe")
    parser.add_argument("--profile", default="DATASET_SAFE_MAX")
    parser.add_argument("--audit", action="store_true")
    parser.add_argument("--routes", action="store_true")
    parser.add_argument("--initial-submission", default="")
    args = parser.parse_args()
    payload: dict[str, Any] = {}
    if args.audit:
        payload["audit"] = dataset_audit(args.output_dir, run_cpu_public_audit=True)
    if args.routes:
        payload["routes"] = build_safe_routes(args.output_dir, args.profile)
    if args.initial_submission:
        payload["initial_submission_tasks"] = len(make_initial_submission(args.initial_submission))
    print(json.dumps(payload, indent=2))


In [ ]:
%%writefile arc_decoder.py
from __future__ import annotations

import bz2
import math
import os
import pickle
from collections import Counter, defaultdict
from typing import Any, Callable

import numpy as np




def _env_truthy(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "1" if default else "0").strip().lower()
    return raw not in ("", "0", "false", "no", "none")


def _allow_symbolic_attempt2() -> bool:
    return _env_truthy("ARC_ALLOW_SYMBOLIC_ATTEMPT2", False)

def hashable(guess: Any) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(map(int, row)) for row in np.asarray(guess))


def _answer_len(solution: Any) -> int:
    grid = np.asarray(solution)
    return int(grid.size + max(0, grid.shape[0] - 1) + 1)  # cells + newlines + EOS


def _beam_norm(sample: dict[str, Any]) -> float:
    if "beam_nll" in sample:
        return float(sample["beam_nll"])
    return float(sample.get("beam_score", 99.0)) / max(1, int(sample.get("beam_len", _answer_len(sample["solution"]))))


def _aug_norms(sample: dict[str, Any]) -> list[float]:
    if sample.get("score_aug_norm"):
        return [float(x) for x in sample["score_aug_norm"]]
    length = max(1, int(sample.get("answer_len", _answer_len(sample["solution"]))))
    return [float(x) / length for x in sample.get("score_aug", [])]


def group_outputs(guesses: dict[str, dict[str, Any]]) -> dict[tuple[tuple[int, ...], ...], dict[str, Any]]:
    groups: dict[tuple[tuple[int, ...], ...], dict[str, Any]] = {}
    for key, sample in guesses.items():
        grid = np.asarray(sample["solution"], dtype=np.int8)
        h = hashable(grid)
        record = groups.setdefault(h, {"solution": grid, "samples": [], "keys": []})
        record["samples"].append(sample)
        record["keys"].append(key)
    return groups


def score_sum(guesses: dict[str, dict[str, Any]], getter: Callable[[list[dict[str, Any]]], float]) -> list[np.ndarray]:
    scored = [(getter(record["samples"]), record["solution"]) for record in group_outputs(guesses).values()]
    scored.sort(key=lambda item: item[0], reverse=True)
    return [grid for _, grid in scored]


def getter_full_probmul_3(samples: list[dict[str, Any]], baseline: float = 3.0) -> float:
    inference = sum(baseline - float(sample.get("beam_score", baseline)) for sample in samples)
    augmented = [sum(baseline - float(score) for score in sample.get("score_aug", [])) for sample in samples]
    return float(inference + (np.mean(augmented) if augmented else 0.0))


def score_full_probmul_3(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    return score_sum(guesses, getter_full_probmul_3)


def getter_kgmon(samples: list[dict[str, Any]]) -> float:
    augmented = [np.mean(sample["score_aug"]) for sample in samples if sample.get("score_aug")]
    return float(len(samples) - (np.mean(augmented) if augmented else 50.0))


def score_kgmon(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    return score_sum(guesses, getter_kgmon)


def _geometry_view(key: str) -> str:
    ops = [part for part in key.split(".")[1:] if part in ("transpose", "rot90")]
    return ".".join(ops) or "id"


def normalized_group_score(record: dict[str, Any]) -> float:
    samples = record["samples"]
    support = len(samples)
    views = len({_geometry_view(key) for key in record.get("keys", [])})
    beam = np.asarray([_beam_norm(sample) for sample in samples], dtype=float)
    aug = [value for sample in samples for value in _aug_norms(sample)]
    aug_mean = float(np.median(aug)) if aug else 0.09
    structural = float(np.mean([sample.get("structural_score", 0.5) for sample in samples]))
    shape_score = float(max(sample.get("shape_score", 0.0) for sample in samples))
    symbolic = float(max(sample.get("symbolic_confidence", 0.0) for sample in samples))
    source_bonus = 8.0 if (_allow_symbolic_attempt2() and symbolic >= 0.999) else 0.0
    return float(
        1.45 * math.log1p(support)
        + 0.22 * views
        - 45.0 * float(np.median(beam))
        - 65.0 * aug_mean
        + 0.55 * structural
        + 0.08 * shape_score
        + source_bonus
    )


def best_likelihood_score(record: dict[str, Any]) -> float:
    samples = record["samples"]
    best_beam = min(_beam_norm(sample) for sample in samples)
    aug_means = [float(np.mean(_aug_norms(sample))) for sample in samples if _aug_norms(sample)]
    best_aug = min(aug_means) if aug_means else 0.09
    structural = max(float(sample.get("structural_score", 0.5)) for sample in samples)
    return float(-55.0 * best_beam - 55.0 * best_aug + 0.35 * structural + 0.15 * math.log1p(len(samples)))


def score_normalized(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = sorted(group_outputs(guesses).values(), key=normalized_group_score, reverse=True)
    return [record["solution"] for record in records]


def score_best_likelihood(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = sorted(group_outputs(guesses).values(), key=best_likelihood_score, reverse=True)
    return [record["solution"] for record in records]


def _strict_symbolic_grids(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    candidates: list[tuple[float, np.ndarray]] = []
    for record in group_outputs(guesses).values():
        confidence = max(float(sample.get("symbolic_confidence", 0.0)) for sample in record["samples"])
        if confidence >= 0.999:
            candidates.append((confidence, record["solution"]))
    candidates.sort(key=lambda item: item[0], reverse=True)
    return [grid for _, grid in candidates]


def _neural_pool(guesses: dict[str, dict[str, Any]]) -> dict[str, dict[str, Any]]:
    return {
        key: sample
        for key, sample in guesses.items()
        if sample.get("source", "llm") not in (("fallback", "symbolic_low") if _allow_symbolic_attempt2() else ("fallback", "symbolic_low", "symbolic"))
    }


def _fallback_order(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = group_outputs(guesses).values()
    ordered = sorted(
        records,
        key=lambda rec: (
            max(float(sample.get("symbolic_confidence", 0.0)) for sample in rec["samples"]),
            max(float(sample.get("fallback_priority", -99.0)) for sample in rec["samples"]),
        ),
        reverse=True,
    )
    return [record["solution"] for record in ordered]


def score_portfolio(guesses: dict[str, dict[str, Any]], n_guesses: int = 2) -> list[np.ndarray]:
    """Conservative pass@2 portfolio.

    Attempt 1 preserves the proven consensus/NLL selector unless at least three
    independent selectors agree on another grid. Attempt 2 uses Borda evidence
    and reserves a slot for a uniquely verified symbolic program.
    """
    if not guesses:
        return []

    retrieval_groups = {}
    for key, sample in guesses.items():
        if sample.get("source") == "retrieval":
            retrieval_groups[hashable(sample["solution"])] = np.asarray(sample["solution"], dtype=np.int8)
    if len(retrieval_groups) == 1:
        exact = next(iter(retrieval_groups.values()))
        if n_guesses <= 1:
            return [exact]
        exact_hash = hashable(exact)
        remaining = {
            key: sample for key, sample in guesses.items()
            if hashable(sample["solution"]) != exact_hash and sample.get("source") != "retrieval"
        }
        return [exact] + score_portfolio(remaining, n_guesses - 1)

    neural = _neural_pool(guesses)
    if not neural:
        return _fallback_order(guesses)[:n_guesses]

    orders = [
        score_kgmon(neural),
        score_full_probmul_3(neural),
        score_normalized(neural),
        score_best_likelihood(neural),
    ]
    orders = [order for order in orders if order]
    if not orders:
        return _fallback_order(guesses)[:n_guesses]

    top_votes = Counter(hashable(order[0]) for order in orders)
    consensus_hash, votes = top_votes.most_common(1)[0]
    first = next(grid for order in orders for grid in order if hashable(grid) == consensus_hash) if votes >= 3 else orders[0][0]
    selected = [np.asarray(first, dtype=np.int8)]
    if n_guesses <= 1:
        return selected

    rank_points = (8.0, 4.0, 2.0, 1.0, 0.4, 0.2)
    weights = (1.15, 1.0, 1.0, 0.85)
    points: dict[tuple[tuple[int, ...], ...], float] = defaultdict(float)
    grids: dict[tuple[tuple[int, ...], ...], np.ndarray] = {}
    for weight, order in zip(weights, orders):
        for rank, grid in enumerate(order[: len(rank_points)]):
            h = hashable(grid)
            grids[h] = np.asarray(grid, dtype=np.int8)
            points[h] += weight * rank_points[rank]
    first_hash = hashable(first)
    points.pop(first_hash, None)

    strict_symbolic = [grid for grid in _strict_symbolic_grids(guesses) if hashable(grid) != first_hash] if _allow_symbolic_attempt2() else []
    if strict_symbolic:
        sym_hash = hashable(strict_symbolic[0])
        grids[sym_hash] = strict_symbolic[0]
        points[sym_hash] += 20.0

    if points:
        second_hash = max(points, key=points.get)
        selected.append(grids[second_hash])
    else:
        for grid in _fallback_order(guesses):
            if hashable(grid) != first_hash:
                selected.append(grid)
                break
    return selected[:n_guesses]


selection_algorithms = [score_full_probmul_3, score_kgmon, score_normalized, score_best_likelihood]


class ArcDecoder:
    def __init__(self, dataset: Any, n_guesses: int):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results: dict[str, dict[str, dict[str, Any]]] = {}

    def load_decoded_results(self, store: str, run_name: str = "") -> None:
        if not os.path.isdir(store):
            print(f"*** Result directory does not exist: {store}")
            return
        loaded = 0
        for filename in sorted(os.listdir(store)):
            path = os.path.join(store, filename)
            if not os.path.isfile(path):
                continue
            try:
                with bz2.BZ2File(path, "rb") as f:
                    outputs = pickle.load(f)
            except Exception as exc:
                print(f"*** Skip unreadable result {filename}: {exc}")
                continue
            if not isinstance(outputs, list):
                continue
            base_key = filename.split(".")[0]
            target = self.decoded_results.setdefault(base_key, {})
            for i, sample in enumerate(outputs):
                if isinstance(sample, dict) and "solution" in sample:
                    target[f"{filename}{run_name}.out{i}"] = sample
                    loaded += 1
        print(f"*** Loaded {loaded} candidate records for {len(self.decoded_results)} test outputs")

    def run_selection_algo(self, selection_algorithm: Callable | None = None) -> dict[str, list[np.ndarray]]:
        if selection_algorithm is None:
            return {base_key: score_portfolio(values, self.n_guesses) for base_key, values in self.decoded_results.items()}
        return {base_key: selection_algorithm(values)[: self.n_guesses] for base_key, values in self.decoded_results.items()}

    def benchmark_selection_algos(self) -> None:
        print("*** Benchmark selection algorithms...")
        labels: dict[str, np.ndarray] = {}
        tasks_per_puzzle: dict[str, int] = {}
        solved_subkeys = 0
        total_subkeys = 0

        for base_key, values in self.decoded_results.items():
            if base_key not in self.dataset.replies:
                continue
            puzzle, test_nr = base_key.rsplit("_", 1)
            tasks_per_puzzle[puzzle] = max(tasks_per_puzzle.get(puzzle, 0), int(test_nr) + 1)
            target = np.asarray(self.dataset.replies[base_key][0])
            labels[base_key] = target
            for subkey, sample in values.items():
                solution = np.asarray(sample["solution"])
                if solution.shape == target.shape and np.array_equal(solution, target):
                    solved_subkeys += 1
                    print(
                        f"ALL_CORRECT beam={float(sample.get('beam_score', np.nan)):8.5f} "
                        f"shape={solution.shape[0]}x{solution.shape[1]} [{subkey}]"
                    )
                total_subkeys += 1
        print(f" subkeys: {solved_subkeys}/{total_subkeys}")

        algorithms: list[tuple[str, Callable[[dict[str, dict[str, Any]]], list[np.ndarray]]]] = [
            (algo.__name__, lambda values, a=algo: a(values)[: self.n_guesses]) for algo in selection_algorithms
        ]
        algorithms.append(("score_portfolio", lambda values: score_portfolio(values, self.n_guesses)))
        for name, algorithm in algorithms:
            selected = {base_key: algorithm(values) for base_key, values in self.decoded_results.items()}
            correct = {
                key
                for key, grids in selected.items()
                if key in labels and any(np.array_equal(grid, labels[key]) for grid in grids)
            }
            score = sum(1.0 / tasks_per_puzzle[key.rsplit("_", 1)[0]] for key in correct)
            print(f" acc: {score:5.1f}/{len(tasks_per_puzzle):3d} ('{name}') solved={sorted(correct)}")


In [ ]:
%%writefile arc_loader.py
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from typing import Any, Callable

import numpy as np


def stable_seed(text: str, modulo: int = 2**31 - 1) -> int:
    """Stable cross-process seed (unlike Python's randomized hash())."""
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest(), "little") % modulo


def convert_grid_to_string(grid: Any) -> str:
    return "\n".join("".join(str(int(cell)) for cell in row) for row in grid)


def is_valid_solution(guess: Any) -> bool:
    return (
        isinstance(guess, np.ndarray)
        and guess.ndim == 2
        and all(0 < int(x) <= 30 for x in guess.shape)
        and bool(np.all((guess >= 0) & (guess <= 9)))
    )


def shuffled(data_list: list[Any]) -> list[Any]:
    return np.random.permutation(data_list).tolist()


def permute_mod(a: Any, descriptor: str, invert: bool = False) -> np.ndarray:
    permutation = [int(i) for i in descriptor if i.isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Bad color permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        return a[..., permutation]
    if a.ndim != 2:
        raise ValueError(f"ARC grid must be 2-D, got {a.ndim}-D")
    if invert:
        permutation = np.argsort(permutation)
    return np.asarray(permutation)[a]


def permute_rnd_all_(_query: Any) -> str:
    return "permute" + "".join(map(str, np.random.permutation(10).tolist()))


def permute_rnd_keep0_(_query: Any) -> str:
    return "permute" + "".join(map(str, [0] + (np.random.permutation(9) + 1).tolist()))


@dataclass(frozen=True)
class ArcTokenSpec:
    digit_ids: tuple[int, ...]
    newline_id: int
    eos_id: int
    pad_id: int
    assistant_marker: tuple[int, ...]
    user_marker: tuple[int, ...]

    @classmethod
    def from_tokenizer(cls, tokenizer: Any) -> "ArcTokenSpec":
        def enc(text: str) -> list[int]:
            try:
                return list(map(int, tokenizer.encode(text, add_special_tokens=False)))
            except TypeError:
                return list(map(int, tokenizer.encode(text)))

        digits: list[int] = []
        for i in range(10):
            ids = enc(str(i))
            if len(ids) != 1:
                raise RuntimeError(f"ARC checkpoint must tokenize digit {i} as one token; got {ids}")
            digits.append(ids[0])
        newline = enc("\n")
        eos = enc("<|im_end|>")
        user = enc("<|im_start|>user\n")
        assistant = enc("<|im_start|>assistant\n")
        if len(newline) != 1 or len(eos) != 1 or not user or not assistant:
            raise RuntimeError(
                "Unexpected tokenizer layout: "
                f"newline={newline}, eos={eos}, user={user}, assistant={assistant}"
            )
        pad = tokenizer.pad_token_id
        if pad is None:
            pad = getattr(tokenizer, "eos_token_id", eos[0])
        return cls(
            digit_ids=tuple(digits),
            newline_id=newline[0],
            eos_id=eos[0],
            pad_id=int(pad),
            assistant_marker=tuple(assistant),
            user_marker=tuple(user),
        )

    def schedule(self, shape: tuple[int, int]) -> list[tuple[int, ...]]:
        """Allowed token IDs at each output position for an exact HxW grid."""
        h, w = map(int, shape)
        if not (1 <= h <= 30 and 1 <= w <= 30):
            raise ValueError(f"Invalid ARC output shape: {shape}")
        out: list[tuple[int, ...]] = []
        for y in range(h):
            out.extend([self.digit_ids] * w)
            if y + 1 < h:
                out.append((self.newline_id,))
        out.append((self.eos_id,))
        return out

    def tokens_to_grid(self, tokens: list[int], expected_shape: tuple[int, int] | None = None) -> np.ndarray | None:
        tokens = list(map(int, tokens))
        if tokens and tokens[-1] == self.eos_id:
            tokens = tokens[:-1]
        reverse = {tok: i for i, tok in enumerate(self.digit_ids)}
        rows: list[list[int]] = [[]]
        for tok in tokens:
            if tok == self.newline_id:
                rows.append([])
            elif tok in reverse:
                rows[-1].append(reverse[tok])
            else:
                return None
        if rows and not rows[-1]:
            rows.pop()
        if not rows or not rows[0] or len({len(row) for row in rows}) != 1:
            return None
        arr = np.asarray(rows, dtype=np.int8)
        if expected_shape is not None and arr.shape != tuple(expected_shape):
            return None
        return arr if is_valid_solution(arr) else None


class QwenFormatter:
    def __init__(self, tokenizer: Any):
        self.tokenizer = tokenizer
        self.tokens = ArcTokenSpec.from_tokenizer(tokenizer)

    def fmt_query(self, query: list[dict[str, Any]]) -> str:
        return "<|im_start|>user\n" + convert_grid_to_string(query[0]["input"]) + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply: list[Any]) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train: list[dict[str, Any]], last_is_challenge: bool = False) -> str:
        # In TTFT mode the shuffled last demonstration is the held-out completion.
        examples = list(train)
        if last_is_challenge and examples:
            test = examples[-1]
            examples = examples[:-1]
        else:
            test = None
        text = ""
        for ex in examples:
            text += self.fmt_query([ex]) + self.fmt_reply([ex["output"]])
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self) -> int:
        return len(self.tokens.schedule((30, 30))) + 1

    def convert_tokens_to_array(
        self,
        tokens: list[int],
        limit_rows: int = 30,
        expected_shape: tuple[int, int] | None = None,
    ) -> np.ndarray | None:
        arr = self.tokens.tokens_to_grid(tokens, expected_shape=expected_shape)
        if arr is not None and arr.shape[0] <= limit_rows:
            return arr
        # Compatibility fallback for tokenizers with surprising decode behavior.
        try:
            text_tokens = list(tokens)
            if text_tokens and int(text_tokens[-1]) == self.tokens.eos_id:
                text_tokens = text_tokens[:-1]
            lines = self.tokenizer.decode(text_tokens).strip().split("\n")
            rows = [[int(ch) for ch in line if ch.isdigit()] for line in lines]
            rows = [row for row in rows if row][:limit_rows]
            if not rows or len({len(row) for row in rows}) != 1:
                return None
            arr = np.asarray(rows, dtype=np.int8)
            if expected_shape is not None and arr.shape != tuple(expected_shape):
                return None
            return arr if is_valid_solution(arr) else None
        except Exception:
            return None


class ArcDataset:
    @staticmethod
    def forward_mod(a: Any, key: str, use_perm: bool = True) -> Any:
        if a is None:
            return None
        out = np.asarray(a)
        for op in key.split(".")[1:]:
            if op == "rot90":
                out = np.rot90(out)
            elif op == "transpose":
                out = np.swapaxes(out, 0, 1)
            elif op.startswith("permute"):
                out = permute_mod(out, op, invert=False) if use_perm else out
            elif op.startswith("copy"):
                out = np.copy(out)
            elif op.startswith(("out", "ex", "run")):
                pass
            else:
                raise NotImplementedError(f"Unknown forward operation '{op}'")
        return out

    @staticmethod
    def invert_mod(a: Any, key: str, inv_perm: bool = True) -> Any:
        if a is None:
            return None
        out = np.asarray(a)
        for op in key.split(".")[1:][::-1]:
            if op == "rot90":
                out = np.rot90(out, k=3)
            elif op == "transpose":
                out = np.swapaxes(out, 0, 1)
            elif op.startswith("permute"):
                out = permute_mod(out, op, invert=True) if inv_perm else out
            elif op.startswith("copy"):
                out = np.copy(out)
            elif op.startswith(("out", "ex", "run")):
                pass
            else:
                raise NotImplementedError(f"Unknown inverse operation '{op}'")
        return out

    def __init__(
        self,
        queries: dict[str, Any],
        replies: dict[str, Any] | None = None,
        keys: list[str] | None = None,
        is_orig: bool = False,
    ):
        replies = {} if replies is None else replies
        if keys is not None:
            keys = [key for key in keys if key is not None]
        self.queries = queries if keys is None else {key: queries[key] for key in keys}
        self.replies = replies if keys is None else {key: replies[key] for key in keys if key in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries) if keys is None else list(keys)
        self.transposed_dataset: ArcDataset | None = None

    def __len__(self) -> int:
        return len(self.keys)

    def change_keys(self, keys: list[str], keep_flags: bool = False) -> "ArcDataset":
        flags = {"is_orig": self.is_orig} if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file: str, keys: list[str] | None = None) -> "ArcDataset":
        with open(queries_file, "r", encoding="utf-8") as f:
            queries = json.load(f)
        return cls(queries=queries, is_orig=True, keys=keys)

    def load_replies(self, replies_file: str) -> "ArcDataset":
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file, "r", encoding="utf-8") as f:
            parsed = json.load(f)
        self.replies = {key: parsed[key] for key in self.keys}
        return self

    def split_multi_replies(self) -> "ArcDataset":
        indices = [(key, i) for key in self.keys for i in range(len(self.queries[key]["test"]))]
        return self.__class__(
            keys=[f"{key}_{i}" for key, i in indices],
            queries={
                f"{key}_{i}": {"train": self.queries[key]["train"], "test": [self.queries[key]["test"][i]]}
                for key, i in indices
            },
            replies={f"{key}_{i}": [self.replies[key][i]] for key, i in indices if key in self.replies},
        )

    def shuffled(self) -> "ArcDataset":
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    @classmethod
    def append(cls, *datasets: "ArcDataset | None") -> "ArcDataset":
        valid = [d for d in datasets if d is not None and d.keys]
        if not valid:
            return cls({}, {})
        return cls(
            queries={key: value for dataset in valid for key, value in dataset.queries.items()},
            replies={key: value for dataset in valid for key, value in dataset.replies.items()},
            keys=[key for dataset in valid for key in dataset.keys],
        )

    def mod_single(
        self,
        mod_func: Callable[..., Any],
        descriptor: str | Callable[[Any], str] | None,
        i: int,
        keep_key: bool,
        inputs_only: bool,
    ) -> "ArcDataset":
        queries: dict[str, Any] = {}
        replies: dict[str, Any] = {}
        keys: list[str] = []
        for key0 in self.keys:
            if descriptor is None:
                raw_desc = "copy{i}" if mod_func is np.copy else mod_func.__name__
            elif isinstance(descriptor, str):
                raw_desc = descriptor
            else:
                raw_desc = descriptor(self.queries[key0])
            desc = raw_desc.format(i=i)

            def func(a: Any, d: str) -> list[Any]:
                value = mod_func(a) if descriptor is None else mod_func(a, d)
                return np.asarray(value).tolist()

            key1 = key0 if keep_key else f"{key0}.{'I' if inputs_only else ''}{desc}"
            keys.append(key1)
            queries[key1] = {
                mode: [
                    {
                        field: (func(array, desc) if field == "input" or not inputs_only else array)
                        for field, array in example.items()
                    }
                    for example in examples
                ]
                for mode, examples in self.queries[key0].items()
            }
            if key0 in self.replies:
                replies[key1] = [func(array, desc) for array in self.replies[key0]]
        return self.__class__(queries=queries, replies=replies, keys=keys)

    def mod(
        self,
        mod_func: Callable[..., Any],
        descriptor: str | Callable[[Any], str] | None = None,
        n: int = 1,
        stack: bool | None = None,
        keep: bool = False,
        keep_key: bool = False,
        shuffle: bool = False,
        join: bool = True,
        inputs_only: bool = False,
    ) -> "ArcDataset | list[ArcDataset]":
        if keep and keep_key:
            raise ValueError("keep and keep_key are mutually exclusive")
        cur: ArcDataset = self
        out: list[ArcDataset] = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None:
            stack = mod_func.__name__.startswith("rot")
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            out.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*out) if join else out

    def geometric(self) -> "ArcDataset":
        data = self.mod(np.transpose, keep=True)
        assert isinstance(data, ArcDataset)
        data = data.mod(np.rot90, n=3, keep=True)
        assert isinstance(data, ArcDataset)
        return data

    def get(self, key: str, formatter: QwenFormatter) -> dict[str, str]:
        train = formatter.fmt_train(self.queries[key]["train"])
        query = formatter.fmt_query(self.queries[key]["test"])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ""
        # TTFT uses leave-one-demonstration-out completion text.
        text = train + query + reply if reply else formatter.fmt_train(self.queries[key]["train"], last_is_challenge=True)
        return {"key": key, "train": train, "query": query, "reply": reply, "input": train + query, "text": text}

    def as_list(self, formatter: QwenFormatter) -> list[dict[str, str]]:
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key: str, formatter: QwenFormatter | None, name: str, max_of_transposed: bool = False) -> int:
        if formatter is None:
            if name == "input":
                return int(sum(np.prod(np.shape(v)) for groups in self.queries[key].values() for ex in groups for v in ex.values()))
            if name == "reply":
                return int(sum(np.prod(np.shape(v)) for v in self.replies[key]))
            raise ValueError(name)
        datasets = [self]
        if max_of_transposed:
            if self.transposed_dataset is None:
                transformed = self.mod(np.transpose, keep=False, keep_key=True)
                assert isinstance(transformed, ArcDataset)
                self.transposed_dataset = transformed
            datasets.append(self.transposed_dataset)
        return max(len(formatter.tokenizer.encode(ds.get(key, formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter: QwenFormatter, name: str, max_len: int, from_end: bool = False) -> "ArcDataset":
        temp = self.change_keys(self.keys)
        new_keys: list[str] = []
        new_queries: dict[str, Any] = {}
        new_replies: dict[str, Any] = {}
        for original_key in self.keys:
            key = original_key
            reply = temp.replies.get(key)
            while temp.get_length(key, formatter, name) > max_len:
                query = temp.queries[key]
                if len(query["train"]) <= 1:
                    break
                if not key.split(".")[-1].startswith("ex"):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                parts = key.split(".")
                payload = parts[-1][2:]
                if not payload:
                    break
                payload = payload[:-1] if from_end else payload[1:]
                key = ".".join(parts[:-1] + [f"ex{payload}"])
                temp.queries[key] = {
                    mode: ((examples[:-1] if from_end else examples[1:]) if mode == "train" else examples)
                    for mode, examples in query.items()
                }
                if reply is not None:
                    temp.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm: Any = None, keep_max: int | None = None) -> "ArcDataset":
        new_keys: list[str] = []
        new_queries: dict[str, Any] = {}
        new_replies: dict[str, Any] = {}
        for key in self.keys:
            n = len(self.queries[key]["train"])
            p = np.random.permutation(n) if perm is None else np.asarray(perm)
            if keep_max is not None:
                p = p[:keep_max]
            separator = "-" if len(p) and int(p.max()) > 9 else ""
            new_key = f"{key}.ex" + separator.join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {
                mode: (np.asarray(examples, dtype=object)[p].tolist() if mode == "train" else examples)
                for mode, examples in self.queries[key].items()
            }
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(
        self,
        n: int = 1,
        shfl_keys: bool = False,
        seed: int = 42,
        descriptor: Callable[[Any], str] = permute_rnd_all_,
        include_identity: bool = False,
    ) -> "ArcDataset":
        np.random.seed(seed)
        geom = self.geometric()
        perm = geom.mod(permute_mod, descriptor, n=n, shuffle=shfl_keys, keep=False)
        assert isinstance(perm, ArcDataset)
        data = self.__class__.append(geom, perm) if include_identity else perm
        return data.shuffle_ex()

    def augment_train_mixed(self, n_total: int = 15, seed: int = 42) -> "ArcDataset":
        """128 TTFT examples at n_total=15: 8 identity + 64 full + 56 zero-preserving."""
        np.random.seed(seed)
        geom = self.geometric().shuffle_ex()
        n_full = (n_total + 1) // 2
        n_keep0 = n_total - n_full
        full = self.geometric().mod(permute_mod, permute_rnd_all_, n=n_full, shuffle=True)
        assert isinstance(full, ArcDataset)
        full = full.shuffle_ex()
        keep0: ArcDataset | None = None
        if n_keep0:
            keep0_data = self.geometric().mod(permute_mod, permute_rnd_keep0_, n=n_keep0, shuffle=True)
            assert isinstance(keep0_data, ArcDataset)
            keep0 = keep0_data.shuffle_ex()
        return self.__class__.append(geom, full, keep0).shuffled()

    @staticmethod
    def geometry_group(key: str) -> tuple[bool, int]:
        ops = key.split(".")[1:]
        transposed = "transpose" in ops
        rotations = sum(op == "rot90" for op in ops) % 4
        # This tuple uniquely determines whether H/W are swapped for all grids.
        return transposed, rotations % 2

    @staticmethod
    def transformed_shape(shape: tuple[int, int], key: str) -> tuple[int, int]:
        return tuple(ArcDataset.forward_mod(np.zeros(shape, dtype=np.int8), key, use_perm=False).shape)

    def get_submission(self, results: dict[str, list[np.ndarray]] | None = None) -> dict[str, Any]:
        if not self.is_orig:
            raise AssertionError("Must be run on original dataset")
        submission = {
            key: [{f"attempt_{i + 1}": [[0]] for i in range(2)} for _ in range(len(self.queries[key]["test"]))]
            for key in self.keys
        }
        if results is not None:
            self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results: dict[str, list[np.ndarray]], submission: dict[str, Any]) -> None:
        print(f"*** Generating submission for {len(results)} outputs...")
        for key, grids in results.items():
            base_id, base_nr = key.rsplit("_", 1)
            if base_id not in submission or int(base_nr) >= len(submission[base_id]):
                continue
            target = submission[base_id][int(base_nr)]
            for i, grid in enumerate(grids[: len(target)]):
                target[f"attempt_{i + 1}"] = np.asarray(grid, dtype=int).tolist()

    def validate_submission(self, submission: dict[str, Any]) -> float:
        if not self.is_orig:
            raise AssertionError("Must be run on original dataset")
        score = 0.0
        for key, replies in self.replies.items():
            for i, target in enumerate(replies):
                if any(np.array_equal(target, submission[key][i][attempt]) for attempt in ("attempt_1", "attempt_2")):
                    score += 1.0 / len(replies)
        return score


In [ ]:
%%writefile arc_retrieval.py
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from typing import Any, Iterable

import numpy as np

from arc_symbolic import D4_NAMES, as_grid, d4, grid_key, valid_grid


INVERSE_D4 = {
    "id": "id",
    "r1": "r3",
    "r2": "r2",
    "r3": "r1",
    "t": "t",
    "tr1": "tr1",
    "tr2": "tr2",
    "tr3": "tr3",
}


@dataclass(frozen=True)
class CanonicalTask:
    fingerprint: str
    representation: str
    transform: str
    color_to_canonical: dict[int, int]
    canonical_to_color: dict[int, int]


@dataclass(frozen=True)
class ReferenceTask:
    task: dict[str, Any]
    solutions: list[Any]
    canonical: CanonicalTask
    source_key: str


def _normalize_colors(arrays: list[np.ndarray]) -> tuple[list[np.ndarray], dict[int, int]]:
    mapping: dict[int, int] = {}
    next_color = 0
    normalized: list[np.ndarray] = []
    for array in arrays:
        out = np.empty_like(array, dtype=np.int8)
        for index, value in np.ndenumerate(array):
            color = int(value)
            if color not in mapping:
                mapping[color] = next_color
                next_color += 1
            out[index] = mapping[color]
        normalized.append(out)
    return normalized, mapping


def _local_pair_key(inp: np.ndarray, out: np.ndarray) -> str:
    normalized, _ = _normalize_colors([inp, out])
    payload = {
        "in_shape": list(inp.shape),
        "out_shape": list(out.shape),
        "in": normalized[0].tolist(),
        "out": normalized[1].tolist(),
    }
    return json.dumps(payload, separators=(",", ":"), sort_keys=True)


def _canonical_for_transform(task: dict[str, Any], transform: str) -> tuple[str, dict[int, int]]:
    transformed_train: list[tuple[np.ndarray, np.ndarray]] = []
    for example in task["train"]:
        inp = d4(as_grid(example["input"]), transform)
        out = d4(as_grid(example["output"]), transform)
        transformed_train.append((inp, out))
    # Demonstration order has no semantic meaning. This key is itself invariant
    # to color labels, making sorting robust to globally permuted duplicates.
    transformed_train.sort(key=lambda pair: _local_pair_key(pair[0], pair[1]))
    transformed_test = [d4(as_grid(example["input"]), transform) for example in task["test"]]

    arrays: list[np.ndarray] = []
    for inp, out in transformed_train:
        arrays.extend([inp, out])
    arrays.extend(transformed_test)
    normalized, mapping = _normalize_colors(arrays)

    cursor = 0
    train_payload = []
    for _ in transformed_train:
        train_payload.append({"input": normalized[cursor].tolist(), "output": normalized[cursor + 1].tolist()})
        cursor += 2
    test_payload = [{"input": normalized[cursor + i].tolist()} for i in range(len(transformed_test))]
    representation = json.dumps({"train": train_payload, "test": test_payload}, separators=(",", ":"), sort_keys=True)
    return representation, mapping


def canonicalize_task(task: dict[str, Any]) -> CanonicalTask:
    variants: list[tuple[str, str, dict[int, int]]] = []
    for transform in D4_NAMES:
        representation, mapping = _canonical_for_transform(task, transform)
        variants.append((representation, transform, mapping))
    representation, transform, mapping = min(variants, key=lambda item: (item[0], item[1]))
    fingerprint = hashlib.sha256(representation.encode("utf-8")).hexdigest()
    inverse = {canonical: original for original, canonical in mapping.items()}
    return CanonicalTask(
        fingerprint=fingerprint,
        representation=representation,
        transform=transform,
        color_to_canonical=dict(mapping),
        canonical_to_color=inverse,
    )


def _solution_to_target(
    reference_solution: Any,
    reference_canonical: CanonicalTask,
    target_canonical: CanonicalTask,
) -> np.ndarray | None:
    transformed = d4(as_grid(reference_solution), reference_canonical.transform)
    canonical = np.empty_like(transformed, dtype=np.int8)
    for index, value in np.ndenumerate(transformed):
        color = int(value)
        if color not in reference_canonical.color_to_canonical:
            return None
        canonical[index] = reference_canonical.color_to_canonical[color]

    target_transformed = np.empty_like(canonical, dtype=np.int8)
    for index, value in np.ndenumerate(canonical):
        canonical_color = int(value)
        if canonical_color not in target_canonical.canonical_to_color:
            return None
        target_transformed[index] = target_canonical.canonical_to_color[canonical_color]
    result = d4(target_transformed, INVERSE_D4[target_canonical.transform])
    return np.asarray(result, dtype=np.int8) if valid_grid(result) else None


class RetrievalIndex:
    """Exact/canonical challenge retrieval with transformation-safe output inversion."""

    def __init__(self):
        self._records: dict[str, list[ReferenceTask]] = {}

    @classmethod
    def from_files(cls, challenge_solution_files: Iterable[tuple[str, str]]) -> "RetrievalIndex":
        index = cls()
        for challenge_path, solution_path in challenge_solution_files:
            try:
                with open(challenge_path, "r", encoding="utf-8") as f:
                    challenges = json.load(f)
                with open(solution_path, "r", encoding="utf-8") as f:
                    solutions = json.load(f)
            except FileNotFoundError:
                continue
            for key, task in challenges.items():
                if key not in solutions:
                    continue
                canonical = canonicalize_task(task)
                index._records.setdefault(canonical.fingerprint, []).append(
                    ReferenceTask(task=task, solutions=solutions[key], canonical=canonical, source_key=key)
                )
        return index

    def __len__(self) -> int:
        return sum(len(values) for values in self._records.values())

    def solve(self, task: dict[str, Any], test_idx: int) -> list[tuple[np.ndarray, str]]:
        target = canonicalize_task(task)
        references = self._records.get(target.fingerprint, [])
        outputs: dict[tuple[tuple[int, ...], ...], tuple[np.ndarray, str]] = {}
        for reference in references:
            if test_idx >= len(reference.solutions):
                continue
            solution = _solution_to_target(reference.solutions[test_idx], reference.canonical, target)
            if solution is None:
                continue
            outputs[grid_key(solution)] = (solution, reference.source_key)
        # A fingerprint collision with incompatible outputs is ambiguous. Returning
        # no answer is safer than asserting a false exact retrieval.
        if len(outputs) != 1:
            return []
        return list(outputs.values())


In [ ]:
%%writefile arc_solver.py
from __future__ import annotations

import bz2
import gc
import io
import json
import logging
import math
import os
import pickle
import sys
import time
import traceback
from collections import defaultdict
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass
from typing import Any, Union

# Unsloth must patch Transformers/PEFT before those packages are imported.
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments

import numpy as np
import torch
from datasets import Dataset
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from transformers import DataCollatorForLanguageModeling

from arc_loader import ArcDataset, QwenFormatter, stable_seed
from arc_symbolic import SymbolicSolver, as_grid, grid_key, infer_output_shapes, structural_consistency

logging.disable(logging.WARNING)
sys.setrecursionlimit(5000)


@dataclass(frozen=True)
class SolverConfig:
    # Environment overrides are deliberate: Kaggle model mounts differ between
    # interactive notebooks and competition reruns. Every resolved checkpoint is
    # still validated by ArcTokenSpec at runtime.
    model_path: str = os.getenv("ARC_MODEL_PATH", "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    competition_dir: str = os.getenv("ARC_COMPETITION_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    output_dir: str = os.getenv("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
    max_seq_length: int = int(os.getenv("ARC_MAX_SEQ_LENGTH", "8192"))
    train_augments: int = int(os.getenv("ARC_MAX_TRAIN_AUGMENTS", "15"))
    symbolic_programs: int = int(os.getenv("ARC_SYMBOLIC_PROGRAMS", "500"))
    max_puzzle_seconds: int = int(os.getenv("ARC_MAX_PUZZLE_SECONDS", "900"))
    min_puzzle_seconds: int = int(os.getenv("ARC_MIN_PUZZLE_SECONDS", "240"))
    reserve_scoring_seconds: int = int(os.getenv("ARC_RESERVE_SCORING_SECONDS", "55"))
    constrained_max_score: float = float(os.getenv("ARC_CONSTRAINED_MAX_SCORE", "2.20"))
    unconstrained_max_score: float = float(os.getenv("ARC_UNCONSTRAINED_MAX_SCORE", str(float(-np.log(0.18)))))
    constrained_branch_cap: int = int(os.getenv("ARC_CONSTRAINED_BRANCH_CAP", "4"))
    unconstrained_branch_cap: int = int(os.getenv("ARC_UNCONSTRAINED_BRANCH_CAP", "6"))
    beam_cap: int = int(os.getenv("ARC_BEAM_CAP", "14"))
    max_candidates_per_view: int = int(os.getenv("ARC_MAX_CANDIDATES_PER_VIEW", "6"))
    max_shape_hypotheses: int = int(os.getenv("ARC_MAX_SHAPE_HYPOTHESES", "3"))
    tta_color_permutations: int = int(os.getenv("ARC_TTA_COLOR_PERMUTATIONS", "2"))


CFG = SolverConfig()


class UnslothFixedTrainer(UnslothTrainer):
    """Avoid the view-tensor loss mutation issue in recent torch/Unsloth builds."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped, "_get_name") and "unsloth" in unwrapped._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


def _subsequence_positions(sequence: list[int], pattern: tuple[int, ...]) -> list[int]:
    if not pattern:
        return []
    width = len(pattern)
    return [i for i in range(len(sequence) - width + 1) if tuple(sequence[i : i + width]) == pattern]


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """Train only assistant grid spans; mask user text and padding."""

    def __init__(self, *args, assistant_marker: tuple[int, ...], eos_id: int, **kwargs):
        super().__init__(*args, **kwargs)
        self.assistant_marker = tuple(assistant_marker)
        self.eos_id = int(eos_id)

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            ids = batch["input_ids"][i].tolist()
            labels = torch.full_like(batch["input_ids"][i], -100)
            for marker_start in _subsequence_positions(ids, self.assistant_marker):
                start = marker_start + len(self.assistant_marker)
                try:
                    end = ids.index(self.eos_id, start) + 1
                except ValueError:
                    continue
                labels[start:end] = batch["input_ids"][i, start:end]
            if not torch.any(labels != -100):
                raise RuntimeError("Completion collator found no assistant answer span")
            batch["labels"][i] = labels
        return batch


def _candidate_tokens(
    logits: torch.Tensor,
    base_scores: list[float],
    allowed: list[tuple[int, ...] | list[int]],
    max_score: float,
    branch_cap: int,
) -> dict[int, list[tuple[float, int]]]:
    n = logits.size(0)
    nll = torch.tensor(base_scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    candidates: dict[int, list[tuple[float, int]]] = {}
    for i in range(n):
        row = [(float(nll[i, token]), int(token)) for token in allowed[i] if float(nll[i, token]) < max_score]
        row.sort(key=lambda item: item[0])
        candidates[i] = row[:branch_cap]
    return candidates


def turbo_dfs_constrained(
    model: Any,
    logits: torch.Tensor,
    schedule: list[tuple[int, ...]],
    step: int,
    max_score: float,
    scores: list[float],
    pos: int,
    cache: Any,
    search_deadline: float,
    pad_id: int,
    branch_cap: int,
    beam_cap: int,
) -> dict[int, list[tuple[float, list[int]]]]:
    n = logits.size(0)
    if step >= len(schedule) or time.time() >= search_deadline:
        return defaultdict(list)
    candidates = _candidate_tokens(logits, scores, [schedule[step]] * n, max_score, branch_cap)
    suffixes: dict[int, list[tuple[float, list[int]]]] = defaultdict(list)

    while time.time() < search_deadline:
        batch_tokens: list[int] = []
        batch_scores: list[float] = []
        alive = 0
        for i in range(n):
            if candidates[i]:
                score, token = candidates[i].pop(0)
                batch_tokens.append(token)
                batch_scores.append(score)
                alive += 1
            else:
                batch_tokens.append(pad_id)
                batch_scores.append(1e6)
        if alive == 0:
            break

        if step + 1 == len(schedule):
            for i, (token, score) in enumerate(zip(batch_tokens, batch_scores)):
                if score < max_score:
                    suffixes[i].append((score, [token]))
            continue

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device, dtype=torch.long),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        children = turbo_dfs_constrained(
            model=model,
            logits=outputs.logits[:, -1],
            schedule=schedule,
            step=step + 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=outputs.past_key_values,
            search_deadline=search_deadline,
            pad_id=pad_id,
            branch_cap=branch_cap,
            beam_cap=beam_cap,
        )
        for batch_id, beams in children.items():
            for score, suffix in beams:
                suffixes[batch_id].append((score, [batch_tokens[batch_id]] + suffix))
            suffixes[batch_id] = sorted(suffixes[batch_id], key=lambda item: item[0])[:beam_cap]
    return suffixes


@torch.no_grad()
def inference_constrained(
    model: Any,
    prefix_tokens: list[list[int]],
    schedule: list[tuple[int, ...]],
    max_score: float,
    search_deadline: float,
    pad_id: int,
    branch_cap: int,
    beam_cap: int,
) -> list[tuple[int, list[tuple[float, list[int]]]]]:
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs_constrained(
        model=model,
        logits=outputs.logits[:, -1],
        schedule=schedule,
        step=0,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        search_deadline=search_deadline,
        pad_id=pad_id,
        branch_cap=branch_cap,
        beam_cap=beam_cap,
    )
    return [(i, sorted(beams, key=lambda item: item[0])[:beam_cap]) for i, beams in suffixes.items()]


# Free-shape DFS still obeys the ARC rectangular-grid grammar.
# State = (completed_rows, current_row_width, fixed_width_or_zero).
def _free_allowed(state: tuple[int, int, int], token_spec: Any) -> list[int]:
    completed, current, width = state
    allowed: list[int] = []
    max_width = width if width else 30
    if current < max_width:
        allowed.extend(token_spec.digit_ids)
    if current > 0 and completed < 29 and (width == 0 or current == width):
        allowed.append(token_spec.newline_id)
    if current > 0 and (width == 0 or current == width):
        allowed.append(token_spec.eos_id)
    return allowed


def _free_next_state(state: tuple[int, int, int], token: int, token_spec: Any) -> tuple[int, int, int]:
    completed, current, width = state
    if token in token_spec.digit_ids:
        return completed, current + 1, width
    if token == token_spec.newline_id:
        return completed + 1, 0, current if width == 0 else width
    return state


def turbo_dfs_unconstrained(
    model: Any,
    logits: torch.Tensor,
    max_new_tokens: int,
    max_score: float,
    scores: list[float],
    states: list[tuple[int, int, int]],
    pos: int,
    cache: Any,
    search_deadline: float,
    token_spec: Any,
    branch_cap: int,
    beam_cap: int,
) -> dict[int, list[tuple[float, list[int]]]]:
    n = logits.size(0)
    if max_new_tokens <= 0 or time.time() >= search_deadline:
        return defaultdict(list)
    allowed = [_free_allowed(state, token_spec) for state in states]
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes: dict[int, list[tuple[float, list[int]]]] = defaultdict(list)
    candidates: dict[int, list[tuple[float, int]]] = {}
    for i in range(n):
        row: list[tuple[float, int]] = []
        for token in allowed[i]:
            score = float(nll[i, token])
            if score >= max_score:
                continue
            if token == token_spec.eos_id:
                suffixes[i].append((score, [token]))
            elif max_new_tokens > 1:
                row.append((score, token))
        row.sort(key=lambda item: item[0])
        candidates[i] = row[:branch_cap]

    while time.time() < search_deadline:
        batch_tokens: list[int] = []
        batch_scores: list[float] = []
        batch_states: list[tuple[int, int, int]] = []
        alive = 0
        for i in range(n):
            if candidates[i]:
                score, token = candidates[i].pop(0)
                batch_tokens.append(token)
                batch_scores.append(score)
                batch_states.append(_free_next_state(states[i], token, token_spec))
                alive += 1
            else:
                batch_tokens.append(token_spec.pad_id)
                batch_scores.append(1e6)
                batch_states.append(states[i])
        if alive == 0:
            break
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device, dtype=torch.long),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        children = turbo_dfs_unconstrained(
            model=model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            states=batch_states,
            pos=pos + 1,
            cache=outputs.past_key_values,
            search_deadline=search_deadline,
            token_spec=token_spec,
            branch_cap=branch_cap,
            beam_cap=beam_cap,
        )
        for batch_id, beams in children.items():
            for score, suffix in beams:
                suffixes[batch_id].append((score, [batch_tokens[batch_id]] + suffix))
            suffixes[batch_id] = sorted(suffixes[batch_id], key=lambda item: item[0])[:beam_cap]
    return suffixes


@torch.no_grad()
def inference_unconstrained(
    model: Any,
    prefix_tokens: list[list[int]],
    max_new_tokens: int,
    max_score: float,
    search_deadline: float,
    token_spec: Any,
    branch_cap: int,
    beam_cap: int,
) -> list[tuple[int, list[tuple[float, list[int]]]]]:
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs_unconstrained(
        model=model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        states=[(0, 0, 0)] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        search_deadline=search_deadline,
        token_spec=token_spec,
        branch_cap=branch_cap,
        beam_cap=beam_cap,
    )
    return [(i, sorted(beams, key=lambda item: item[0])[:beam_cap]) for i, beams in suffixes.items()]


@torch.no_grad()
def calc_scores(
    queries: list[str],
    answers: list[str],
    tokenizer: Any,
    model: Any,
    pad_id: int,
) -> tuple[list[float], list[float], list[int]]:
    query_tokens: list[list[int]] = []
    answer_tokens: list[list[int]] = []
    combined: list[list[int]] = []
    for query, answer in zip(queries, answers):
        q = list(map(int, tokenizer.encode(query)))
        a = list(map(int, tokenizer.encode(answer)))
        query_tokens.append(q)
        answer_tokens.append(a)
        combined.append(q + a)
    max_len = max(map(len, combined))
    rows: list[list[int]] = []
    masks: list[list[int]] = []
    for row in combined:
        padding = max_len - len(row)
        rows.append(row + [pad_id] * padding)
        masks.append([1] * len(row) + [0] * padding)
    input_ids = torch.tensor(rows, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(masks, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    logp = outputs.logits.float().cpu().log_softmax(-1)
    totals: list[float] = []
    normalized: list[float] = []
    lengths: list[int] = []
    for logits, q, a in zip(logp, query_tokens, answer_tokens):
        start = len(q) - 1
        answer_logits = logits[start : start + len(a)]
        score = -answer_logits[torch.arange(len(a)), a].sum().item()
        totals.append(float(score))
        normalized.append(float(score / max(1, len(a))))
        lengths.append(len(a))
    return totals, normalized, lengths


def _strict_symbolic_confidence(candidate: Any, candidates: list[Any]) -> float:
    if candidate is None or candidate.confidence < 0.995 or candidate.min_cost > 0.45:
        return 0.0
    margin = candidate.score - (candidates[1].score if len(candidates) > 1 else -99.0)
    simple_d4 = any(name.startswith("d4:") for name in candidate.programs)
    if simple_d4 and (len(candidates) == 1 or margin >= 0.65):
        return 0.999
    return 0.0


def save_symbolic_candidates(puzzle_ds: ArcDataset, puzzle_key: str, output_dir: str) -> dict[str, list[Any]]:
    solver = SymbolicSolver(max_programs=CFG.symbolic_programs, max_outputs=12)
    task = puzzle_ds.queries[puzzle_key]
    hints: dict[str, list[Any]] = {}
    for test_idx in range(len(task["test"])):
        base_key = f"{puzzle_key}_{test_idx}"
        candidates = solver.synthesize(task, test_idx)
        hints[base_key] = candidates
        records: list[dict[str, Any]] = []
        for candidate in candidates[:2]:
            confidence = _strict_symbolic_confidence(candidate, candidates)
            if confidence < 0.999:
                continue
            records.append(
                {
                    "beam_score": 0.0,
                    "beam_nll": 0.0,
                    "beam_len": int(candidate.grid.size + candidate.grid.shape[0]),
                    "score_aug": [],
                    "score_aug_norm": [],
                    "answer_len": int(candidate.grid.size + candidate.grid.shape[0]),
                    "solution": candidate.grid,
                    "source": "symbolic",
                    "symbolic_confidence": confidence,
                    "structural_score": 1.0,
                    "shape_score": 6.0,
                    "programs": candidate.programs,
                }
            )
        if records:
            with bz2.BZ2File(os.path.join(output_dir, f"{base_key}.symbolic"), "wb") as f:
                pickle.dump(records, f)
    return hints


def _shape_plan(task: dict[str, Any], test_idx: int, symbolic_candidates: list[Any]) -> tuple[list[tuple[int, int, float, str]], bool]:
    test_grid = as_grid(task["test"][test_idx]["input"])
    hypotheses = infer_output_shapes(task, test_grid, max_shapes=12)
    by_shape = {(h, w): (score, reason) for h, w, score, reason in hypotheses}
    for candidate in symbolic_candidates or []:
        shape = tuple(candidate.grid.shape)
        old = by_shape.get(shape, (-99.0, ""))
        symbolic_score = 4.5 + 0.2 * float(candidate.confidence)
        if symbolic_score > old[0]:
            by_shape[shape] = (symbolic_score, "symbolic-program")
    ordered = [(shape[0], shape[1], value[0], value[1]) for shape, value in by_shape.items()]
    ordered.sort(key=lambda item: item[2], reverse=True)
    top_score = ordered[0][2] if ordered else 0.0
    if top_score >= 5.0:
        return ordered[:1], False
    if top_score >= 3.0:
        return ordered[:2], True
    return ordered[: CFG.max_shape_hypotheses], True


def _group_eval_subkeys(eval_ds: ArcDataset) -> list[list[str]]:
    grouped: dict[tuple[int, tuple[bool, int]], list[str]] = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        base_key = subkey.split(".")[0]
        test_idx = int(base_key.rsplit("_", 1)[1])
        grouped[(test_idx, ArcDataset.geometry_group(subkey))].append(subkey)
    batches: list[list[str]] = []
    for _, subkeys in sorted(grouped.items()):
        for i in range(0, len(subkeys), 4):
            batches.append(subkeys[i : i + 4])
    return batches


def _adaptive_puzzle_budget(queue: Any, end_time: float, n_workers: int) -> float:
    seconds_left = max(1.0, end_time - time.time())
    try:
        queued = max(0, int(queue.qsize()))
    except Exception:
        queued = n_workers * 8
    # Include currently active workers in the denominator. This converges to the
    # average budget required to cover every queued puzzle before the deadline.
    fair_share = 0.92 * seconds_left * n_workers / max(n_workers, queued + n_workers)
    return float(np.clip(fair_share, CFG.min_puzzle_seconds, CFG.max_puzzle_seconds))


def worker(rank: int, queue: Any, end_time: float, n_workers: int = 4) -> None:
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() not in ("", "0", "false", "no")
    peft_params = dict(
        r=int(os.getenv("ARC60_TTT_LORA_R", "256")),
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=int(os.getenv("ARC60_TTT_LORA_ALPHA", "32")),
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )
    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        max_grad_norm=1.0,
        learning_rate=float(os.getenv("ARC60_TTT_LR", "5e-5")),
        optim=os.getenv("ARC60_TTT_OPTIMIZER", "adamw_torch"),
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG.model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=CFG.max_seq_length,
    )
    model = FastLanguageModel.get_peft_model(model, **peft_params)
    for _, parameter in model.named_parameters():
        if parameter.dtype == torch.float32:
            parameter.data = parameter.data.to(torch.bfloat16)

    default_weights = {key: value.clone().detach() for key, value in get_peft_model_state_dict(model, adapter_name="default").items()}
    formatter = QwenFormatter(tokenizer)
    token_spec = formatter.tokens
    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
        assistant_marker=token_spec.assistant_marker,
        eos_id=token_spec.eos_id,
    )
    max_new_tokens = formatter.max_new_tokens()

    filename = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    arc_test_set = ArcDataset.from_file(os.path.join(CFG.competition_dir, filename))
    os.makedirs(CFG.output_dir, exist_ok=True)

    route_path = os.getenv("ARC_SAFE_ROUTE_PATH", "/kaggle/working/arc60_dataset_safe/safe_routes.json")
    try:
        with open(route_path, "r", encoding="utf-8") as route_file:
            route_table = json.load(route_file)
        print(f"[Rank {rank}] loaded dataset-safe route table with {len(route_table)} tasks")
    except Exception as exc:
        route_table = {}
        print(f"[Rank {rank}] safe route table unavailable; using full TTT: {type(exc).__name__}")

    while True:
        if time.time() >= end_time:
            print(f"[Rank {rank}] global deadline reached")
            break
        key = queue.get()
        if key is None:
            break

        try:
            start_time = time.time()
            puzzle_budget = _adaptive_puzzle_budget(queue, end_time, n_workers)
            puzzle_deadline = min(end_time, start_time + puzzle_budget)
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass

            set_peft_model_state_dict(model, default_weights, adapter_name="default")
            puzzle_ds = arc_test_set.change_keys([key])
            symbolic_hints = save_symbolic_candidates(puzzle_ds, key, CFG.output_dir)

            route = route_table.get(key, {}) if isinstance(route_table, dict) else {}
            preferred_augments = int(route.get("train_augments", CFG.train_augments))
            preferred_augments = max(0, min(CFG.train_augments, preferred_augments))

            # The public symbolic lane solved 0/172 evaluation outputs in the
            # supplied data audit. It therefore never earns a reduced TTT budget.
            # Only exact canonical training retrieval may bypass adaptation.
            exact_retrieval = bool(route.get("exact_retrieval", False))
            allow_skip = os.getenv("ARC_ALLOW_EXACT_RETRIEVAL_SKIP", "1").strip().lower() not in ("0", "false", "no")

            # Deadline-safe cap: 15 -> ~128 views, 7 -> ~64, 3 -> ~32.
            # This is based on remaining wall time, not an unvalidated reward model.
            force_full = os.getenv("ARC_FORCE_FULL_TTT", "0").strip().lower() not in ("", "0", "false", "no")
            if force_full:
                deadline_cap = CFG.train_augments
            elif puzzle_budget < 420:
                deadline_cap = min(CFG.train_augments, 3)
            elif puzzle_budget < 690:
                deadline_cap = min(CFG.train_augments, 7)
            else:
                deadline_cap = CFG.train_augments
            task_train_augments = min(preferred_augments, deadline_cap)
            skip_ttt = bool(exact_retrieval and allow_skip) or task_train_augments == 0
            train_runtime = 0.0
            train_loss = float("nan")
            ttt_reverted = False
            if skip_ttt:
                model = FastLanguageModel.for_inference(model)
            else:
                model = FastLanguageModel.for_training(model)
                train_ds = puzzle_ds.augment_train_mixed(
                    n_total=task_train_augments,
                    seed=stable_seed(f"train:{key}:{task_train_augments}"),
                )
                train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=CFG.max_seq_length)
                train_rows = Dataset.from_list(train_ds.as_list(formatter))
                with io.StringIO() as buffer, redirect_stdout(buffer), redirect_stderr(buffer):
                    active_args = dict(train_args)
                    try:
                        trainer = UnslothFixedTrainer(
                            model=model,
                            tokenizer=tokenizer,
                            data_collator=collator,
                            train_dataset=train_rows,
                            dataset_text_field="text",
                            max_seq_length=CFG.max_seq_length,
                            args=UnslothTrainingArguments(**active_args),
                        )
                        stats = trainer.train()
                    except Exception:
                        if active_args.get("optim") == "adamw_torch":
                            raise
                        # Some Kaggle images expose bitsandbytes but not a working
                        # optimizer kernel. Reset the adapter and retry the proven
                        # torch AdamW path rather than losing the task.
                        try:
                            del trainer
                        except Exception:
                            pass
                        set_peft_model_state_dict(model, default_weights, adapter_name="default")
                        active_args["optim"] = "adamw_torch"
                        trainer = UnslothFixedTrainer(
                            model=model,
                            tokenizer=tokenizer,
                            data_collator=collator,
                            train_dataset=train_rows,
                            dataset_text_field="text",
                            max_seq_length=CFG.max_seq_length,
                            args=UnslothTrainingArguments(**active_args),
                        )
                        stats = trainer.train()
                    train_runtime = float(stats.metrics.get("train_runtime", 0.0))
                    train_loss = float(stats.metrics.get("train_loss", float("nan")))
                    model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                    del trainer
                max_ttt_loss = float(os.getenv("ARC_MAX_ACCEPTED_TTT_LOSS", "5.0"))
                if (not math.isfinite(train_loss)) or train_loss > max_ttt_loss:
                    set_peft_model_state_dict(model, default_weights, adapter_name="default")
                    ttt_reverted = True
                    print(f"[Rank {rank}] {key} reverted pathological TTT loss={train_loss}")
                model = FastLanguageModel.for_inference(model)
            gc.collect()
            torch.cuda.empty_cache()
            print(
                f"[Rank {rank}] {key} budget={puzzle_budget:.0f}s route={route.get('action', 'full_default')} "
                f"augments={task_train_augments} skip_ttt={skip_ttt} "
                f"train={train_runtime:.1f}s loss={train_loss:.6f} reverted={ttt_reverted}"
            )

            puzzle_ds_multi = puzzle_ds.split_multi_replies()
            eval_ds = puzzle_ds_multi.augment(n=CFG.tta_color_permutations, seed=stable_seed(f"eval:{key}"))
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=CFG.max_seq_length - max_new_tokens)
            original_task = puzzle_ds.queries[key]
            known_scores: dict[tuple[str, tuple[tuple[int, ...], ...]], tuple[list[float], list[float], int]] = {}

            # Split geometry groups again by actual prefix length as a defensive guard.
            batches: list[tuple[list[str], list[list[int]]]] = []
            for subkeys in _group_eval_subkeys(eval_ds):
                buckets: dict[int, list[tuple[str, list[int]]]] = defaultdict(list)
                for subkey in subkeys:
                    tokens = list(map(int, tokenizer.encode(eval_ds.get(subkey, formatter)["input"])))
                    buckets[len(tokens)].append((subkey, tokens))
                for values in buckets.values():
                    batches.append(([item[0] for item in values], [item[1] for item in values]))

            with torch.inference_mode():
                for batch_index, (subkeys, prefix_tokens) in enumerate(batches):
                    hard_decode_deadline = puzzle_deadline - CFG.reserve_scoring_seconds
                    if time.time() >= hard_decode_deadline:
                        print(f"[Rank {rank}] {key} decoding deadline")
                        break
                    batches_left = max(1, len(batches) - batch_index)
                    fair_batch_seconds = max(18.0, (hard_decode_deadline - time.time()) / batches_left)
                    batch_deadline = min(hard_decode_deadline, time.time() + fair_batch_seconds)
                    batch_score_deadline = min(
                        puzzle_deadline - 5.0,
                        batch_deadline + CFG.reserve_scoring_seconds / max(1, len(batches)),
                    )
                    base_key = subkeys[0].split(".")[0]
                    test_idx = int(base_key.rsplit("_", 1)[1])
                    shape_plan, needs_free = _shape_plan(original_task, test_idx, symbolic_hints.get(base_key, []))
                    beam_records: dict[int, list[tuple[float, list[int], tuple[int, int] | None, float, str, str]]] = defaultdict(list)

                    for shape_index, (h, w, shape_score, shape_reason) in enumerate(shape_plan):
                        remaining = batch_deadline - time.time()
                        if remaining <= 5:
                            break
                        transformed_shape = ArcDataset.transformed_shape((h, w), subkeys[0])
                        schedule = token_spec.schedule(transformed_shape)
                        calls_left = max(1, len(shape_plan) - shape_index + (1 if needs_free else 0))
                        call_deadline = min(batch_deadline, time.time() + max(8.0, remaining / calls_left))
                        results = inference_constrained(
                            model=model,
                            prefix_tokens=prefix_tokens,
                            schedule=schedule,
                            max_score=CFG.constrained_max_score,
                            search_deadline=call_deadline,
                            pad_id=token_spec.pad_id,
                            branch_cap=CFG.constrained_branch_cap,
                            beam_cap=CFG.beam_cap,
                        )
                        for subkey_id, beams in results:
                            for beam_score, out_tokens in beams:
                                beam_records[subkey_id].append(
                                    (beam_score, out_tokens, transformed_shape, shape_score, shape_reason, "shape-grammar")
                                )
                        # Confident shape is calibrated as exact on all supplied public
                        # evaluation outputs; do not waste time on redundant shapes.
                        if shape_score >= 5.0 and sum(bool(beam_records[i]) for i in range(len(subkeys))) >= max(1, len(subkeys) // 2):
                            break

                    if needs_free or not any(beam_records.values()):
                        remaining = batch_deadline - time.time()
                        if remaining > 8:
                            call_deadline = batch_deadline
                            results = inference_unconstrained(
                                model=model,
                                prefix_tokens=prefix_tokens,
                                max_new_tokens=max_new_tokens,
                                max_score=CFG.unconstrained_max_score,
                                search_deadline=call_deadline,
                                token_spec=token_spec,
                                branch_cap=CFG.unconstrained_branch_cap,
                                beam_cap=CFG.beam_cap,
                            )
                            for subkey_id, beams in results:
                                for beam_score, out_tokens in beams:
                                    beam_records[subkey_id].append((beam_score, out_tokens, None, 0.0, "free", "rect-grammar"))

                    for subkey_id, records in beam_records.items():
                        subkey = subkeys[subkey_id]
                        base_key = subkey.split(".")[0]
                        test_idx = int(base_key.rsplit("_", 1)[1])
                        test_input = as_grid(original_task["test"][test_idx]["input"])
                        by_grid: dict[tuple[tuple[int, ...], ...], tuple[Any, ...]] = {}
                        for record in sorted(records, key=lambda item: item[0]):
                            beam_score, out_tokens, expected_shape, shape_score, shape_reason, decoder_name = record
                            array = formatter.convert_tokens_to_array(out_tokens, expected_shape=expected_shape)
                            if array is None:
                                continue
                            solution = np.asarray(puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True), dtype=np.int8)
                            hkey = grid_key(solution)
                            previous = by_grid.get(hkey)
                            if previous is None or beam_score < previous[0]:
                                by_grid[hkey] = (beam_score, out_tokens, solution, shape_score, shape_reason, decoder_name)

                        decoded: list[dict[str, Any]] = []
                        for beam_score, out_tokens, solution, shape_score, shape_reason, decoder_name in sorted(by_grid.values(), key=lambda item: item[0])[: CFG.max_candidates_per_view]:
                            cache_key = (base_key, grid_key(solution))
                            if cache_key in known_scores:
                                totals, norms, answer_len = known_scores[cache_key]
                            else:
                                totals, norms, lengths = [], [], []
                                if time.time() < batch_score_deadline:
                                    score_dataset = ArcDataset(
                                        keys=[base_key],
                                        queries={base_key: puzzle_ds_multi.queries[base_key]},
                                        replies={base_key: [solution.tolist()]},
                                    )
                                    augmented = score_dataset.augment(n=1, seed=stable_seed(f"score:{base_key}:{grid_key(solution)}"))
                                    augmented = augmented.cut_to_len(
                                        formatter=formatter,
                                        name="input",
                                        max_len=CFG.max_seq_length - max_new_tokens,
                                    )
                                    samples = [score_dataset.get(base_key, formatter)] + augmented.as_list(formatter)
                                    if batch_score_deadline - time.time() < 18:
                                        samples = samples[:1]
                                    for start in range(0, len(samples), 4):
                                        if time.time() >= batch_score_deadline:
                                            break
                                        chunk = samples[start : start + 4]
                                        chunk_totals, chunk_norms, chunk_lengths = calc_scores(
                                            [sample["input"] for sample in chunk],
                                            [sample["reply"] for sample in chunk],
                                            tokenizer,
                                            model,
                                            token_spec.pad_id,
                                        )
                                        totals.extend(chunk_totals)
                                        norms.extend(chunk_norms)
                                        lengths.extend(chunk_lengths)
                                answer_len = lengths[0] if lengths else int(solution.size + solution.shape[0])
                                known_scores[cache_key] = (totals, norms, answer_len)

                            decoded.append(
                                {
                                    "beam_score": float(beam_score),
                                    "beam_nll": float(beam_score / max(1, len(out_tokens))),
                                    "beam_len": len(out_tokens),
                                    "score_aug": totals,
                                    "score_aug_norm": norms,
                                    "answer_len": answer_len,
                                    "solution": solution,
                                    "source": "llm",
                                    "decoder": decoder_name,
                                    "shape_score": float(shape_score),
                                    "shape_reason": shape_reason,
                                    "structural_score": structural_consistency(original_task, test_input, solution),
                                    "rl_route": route.get("action", "full_default"),
                                    "ttt_augments": int(task_train_augments),
                                    "ttt_skipped": bool(skip_ttt),
                                    "ttt_reverted": bool(ttt_reverted),
                                    "route_action": str(route.get("action", "full_default")),
                                    "puzzle_budget_seconds": float(puzzle_budget),
                                }
                            )
                        if decoded:
                            from arc_exec_rl.common import atomic_candidate_pickle
                            atomic_candidate_pickle(os.path.join(CFG.output_dir, subkey), decoded)

            try:
                memory = torch.cuda.max_memory_allocated() // 1024**2
            except Exception:
                memory = -1
            print(f"[Rank {rank}] finished {key} in {time.time() - start_time:.1f}s peak={memory}MB")
        except Exception as exc:
            print(f"[Rank {rank}] ERROR on {key}: {type(exc).__name__}: {exc}")
            traceback.print_exc()
            gc.collect()
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            continue


In [ ]:
%%writefile arc_symbolic.py
from __future__ import annotations

import math
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from typing import Any, Callable, Iterable

import numpy as np

Grid = np.ndarray


def as_grid(value: Any) -> Grid:
    arr = np.asarray(value, dtype=np.int8)
    if arr.ndim != 2:
        raise ValueError(f"ARC grid must be 2-D, got shape {arr.shape}")
    return arr


def grid_key(grid: Any) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(map(int, row)) for row in np.asarray(grid))


def valid_grid(grid: Any) -> bool:
    a = np.asarray(grid)
    return a.ndim == 2 and 1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30 and np.all((0 <= a) & (a <= 9))


def mode_color(grid: Grid) -> int:
    values, counts = np.unique(grid, return_counts=True)
    return int(values[int(np.argmax(counts))])


def bbox_of_mask(mask: np.ndarray) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return None
    return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1


def crop_nonbackground(grid: Grid, bg: int) -> Grid | None:
    box = bbox_of_mask(grid != bg)
    if box is None:
        return None
    y0, y1, x0, x1 = box
    return np.array(grid[y0:y1, x0:x1], copy=True)


D4_NAMES = ("id", "r1", "r2", "r3", "t", "tr1", "tr2", "tr3")


def d4(grid: Grid, name: str) -> Grid:
    if name == "id":
        return np.array(grid, copy=True)
    if name == "r1":
        return np.rot90(grid, 1).copy()
    if name == "r2":
        return np.rot90(grid, 2).copy()
    if name == "r3":
        return np.rot90(grid, 3).copy()
    if name == "t":
        return grid.T.copy()
    if name == "tr1":
        return np.rot90(grid.T, 1).copy()
    if name == "tr2":
        return np.rot90(grid.T, 2).copy()
    if name == "tr3":
        return np.rot90(grid.T, 3).copy()
    raise KeyError(name)


@dataclass(frozen=True)
class Component:
    cells: tuple[tuple[int, int], ...]
    color: int | None
    y0: int
    y1: int
    x0: int
    x1: int

    @property
    def area(self) -> int:
        return len(self.cells)

    @property
    def height(self) -> int:
        return self.y1 - self.y0

    @property
    def width(self) -> int:
        return self.x1 - self.x0


def components(grid: Grid, bg: int, diagonal: bool = False, same_color: bool = False) -> list[Component]:
    h, w = grid.shape
    seen = np.zeros((h, w), dtype=bool)
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diagonal:
        dirs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    out: list[Component] = []
    for sy in range(h):
        for sx in range(w):
            if seen[sy, sx] or int(grid[sy, sx]) == bg:
                continue
            seed_color = int(grid[sy, sx])
            q = deque([(sy, sx)])
            seen[sy, sx] = True
            cells: list[tuple[int, int]] = []
            while q:
                y, x = q.popleft()
                cells.append((y, x))
                for dy, dx in dirs:
                    yy, xx = y + dy, x + dx
                    if not (0 <= yy < h and 0 <= xx < w) or seen[yy, xx] or int(grid[yy, xx]) == bg:
                        continue
                    if same_color and int(grid[yy, xx]) != seed_color:
                        continue
                    seen[yy, xx] = True
                    q.append((yy, xx))
            ys = [p[0] for p in cells]
            xs = [p[1] for p in cells]
            colors = {int(grid[y, x]) for y, x in cells}
            out.append(
                Component(
                    cells=tuple(cells),
                    color=next(iter(colors)) if len(colors) == 1 else None,
                    y0=min(ys),
                    y1=max(ys) + 1,
                    x0=min(xs),
                    x1=max(xs) + 1,
                )
            )
    return out


def select_component(items: list[Component], selector: str) -> Component | None:
    if not items:
        return None
    if selector == "largest":
        best = max(c.area for c in items)
        winners = [c for c in items if c.area == best]
    elif selector == "smallest":
        best = min(c.area for c in items)
        winners = [c for c in items if c.area == best]
    elif selector == "tallest":
        best = max(c.height for c in items)
        winners = [c for c in items if c.height == best]
    elif selector == "widest":
        best = max(c.width for c in items)
        winners = [c for c in items if c.width == best]
    elif selector == "unique_color":
        counts = Counter(c.color for c in items if c.color is not None)
        winners = [c for c in items if c.color is not None and counts[c.color] == 1]
    else:
        raise KeyError(selector)
    if len(winners) != 1:
        return None
    return winners[0]


def component_patch(
    grid: Grid,
    bg: int,
    diagonal: bool,
    same_color: bool,
    selector: str,
    mask_only: bool,
) -> Grid | None:
    comp = select_component(components(grid, bg, diagonal, same_color), selector)
    if comp is None:
        return None
    patch = np.array(grid[comp.y0 : comp.y1, comp.x0 : comp.x1], copy=True)
    if mask_only:
        keep = {(y - comp.y0, x - comp.x0) for y, x in comp.cells}
        for y in range(patch.shape[0]):
            for x in range(patch.shape[1]):
                if (y, x) not in keep:
                    patch[y, x] = bg
    return patch


def color_role(grid: Grid, role: str) -> int | None:
    counts = Counter(map(int, grid.ravel()))
    if role == "zero":
        return 0
    if role == "mode":
        return counts.most_common(1)[0][0]
    nonzero = [(count, color) for color, count in counts.items() if color != 0]
    if not nonzero:
        return None
    if role == "rarest_nonzero":
        value = min(count for count, _ in nonzero)
        winners = [color for count, color in nonzero if count == value]
    elif role == "common_nonzero":
        value = max(count for count, _ in nonzero)
        winners = [color for count, color in nonzero if count == value]
    else:
        raise KeyError(role)
    return winners[0] if len(winners) == 1 else None


def crop_color_role(grid: Grid, role: str, keep_other: bool) -> Grid | None:
    color = color_role(grid, role)
    if color is None:
        return None
    box = bbox_of_mask(grid == color)
    if box is None:
        return None
    y0, y1, x0, x1 = box
    patch = np.array(grid[y0:y1, x0:x1], copy=True)
    if not keep_other:
        bg = mode_color(grid)
        patch[patch != color] = bg
    return patch


def compress_nonbackground(grid: Grid, bg: int) -> Grid | None:
    row_mask = np.any(grid != bg, axis=1)
    col_mask = np.any(grid != bg, axis=0)
    if not row_mask.any() or not col_mask.any():
        return None
    return np.array(grid[np.ix_(row_mask, col_mask)], copy=True)


def trim_uniform_border(grid: Grid) -> Grid:
    out = np.array(grid, copy=True)
    changed = True
    while changed and out.shape[0] > 1 and out.shape[1] > 1:
        changed = False
        if np.all(out[0] == out[0, 0]) and np.all(out[0] == out[-1, 0]) and np.all(out[0] == out[:, 0]) and np.all(out[0] == out[:, -1]):
            if np.all(out[-1] == out[0, 0]) and np.all(out[:, -1] == out[0, 0]):
                out = out[1:-1, 1:-1]
                changed = True
                continue
        if np.all(out[0] == out[0, 0]) and np.all(out[0] == mode_color(out)):
            out = out[1:]
            changed = True
        if out.shape[0] > 1 and np.all(out[-1] == out[-1, 0]) and np.all(out[-1] == mode_color(out)):
            out = out[:-1]
            changed = True
        if out.shape[1] > 1 and np.all(out[:, 0] == out[0, 0]) and np.all(out[:, 0] == mode_color(out)):
            out = out[:, 1:]
            changed = True
        if out.shape[1] > 1 and np.all(out[:, -1] == out[0, -1]) and np.all(out[:, -1] == mode_color(out)):
            out = out[:, :-1]
            changed = True
    return out


def fill_holes(grid: Grid, bg: int, fill: int | None = None) -> Grid:
    h, w = grid.shape
    outside = np.zeros((h, w), dtype=bool)
    q: deque[tuple[int, int]] = deque()
    for y in range(h):
        for x in (0, w - 1):
            if int(grid[y, x]) == bg and not outside[y, x]:
                outside[y, x] = True
                q.append((y, x))
    for x in range(w):
        for y in (0, h - 1):
            if int(grid[y, x]) == bg and not outside[y, x]:
                outside[y, x] = True
                q.append((y, x))
    while q:
        y, x = q.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and int(grid[yy, xx]) == bg and not outside[yy, xx]:
                outside[yy, xx] = True
                q.append((yy, xx))
    holes = (grid == bg) & ~outside
    out = np.array(grid, copy=True)
    if fill is None:
        neighbors: list[int] = []
        for y, x in zip(*np.where(holes)):
            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                yy, xx = y + dy, x + dx
                if 0 <= yy < h and 0 <= xx < w and int(grid[yy, xx]) != bg:
                    neighbors.append(int(grid[yy, xx]))
        fill = Counter(neighbors).most_common(1)[0][0] if neighbors else bg
    out[holes] = int(fill)
    return out


def block_reduce(grid: Grid, fy: int, fx: int, mode: str, bg: int) -> Grid | None:
    h, w = grid.shape
    if h % fy or w % fx:
        return None
    blocks = grid.reshape(h // fy, fy, w // fx, fx).transpose(0, 2, 1, 3)
    out = np.empty((h // fy, w // fx), dtype=np.int8)
    for y in range(out.shape[0]):
        for x in range(out.shape[1]):
            vals = list(map(int, blocks[y, x].ravel()))
            if mode == "tl":
                out[y, x] = vals[0]
            elif mode == "majority":
                out[y, x] = Counter(vals).most_common(1)[0][0]
            elif mode == "uniform":
                if len(set(vals)) != 1:
                    return None
                out[y, x] = vals[0]
            elif mode == "nonbg":
                present = [v for v in vals if v != bg]
                if not present:
                    out[y, x] = bg
                elif len(set(present)) == 1:
                    out[y, x] = present[0]
                else:
                    return None
            else:
                raise KeyError(mode)
    return out


def _separator_split(grid: Grid, axis: int) -> tuple[Grid, Grid] | None:
    n = grid.shape[axis]
    uniform: list[tuple[int, int]] = []
    for i in range(n):
        line = grid[i, :] if axis == 0 else grid[:, i]
        if np.all(line == line.flat[0]):
            uniform.append((i, int(line.flat[0])))
    for start, color in uniform:
        end = start + 1
        while end < n:
            line = grid[end, :] if axis == 0 else grid[:, end]
            if not (np.all(line == color)):
                break
            end += 1
        if start == n - end and start > 0:
            if axis == 0:
                return np.array(grid[:start], copy=True), np.array(grid[end:], copy=True)
            return np.array(grid[:, :start], copy=True), np.array(grid[:, end:], copy=True)
    return None


def split_two_panels(grid: Grid, mode: str) -> tuple[Grid, Grid] | None:
    h, w = grid.shape
    if mode == "h_sep":
        return _separator_split(grid, 0)
    if mode == "v_sep":
        return _separator_split(grid, 1)
    if mode == "h_half" and h % 2 == 0:
        return np.array(grid[: h // 2], copy=True), np.array(grid[h // 2 :], copy=True)
    if mode == "v_half" and w % 2 == 0:
        return np.array(grid[:, : w // 2], copy=True), np.array(grid[:, w // 2 :], copy=True)
    return None


def combine_panels(a: Grid, b: Grid, op: str, bg: int) -> Grid | None:
    if a.shape != b.shape:
        return None
    ma, mb = a != bg, b != bg
    if op == "a":
        return a.copy()
    if op == "b":
        return b.copy()
    if op == "overlay_ab":
        return np.where(ma, a, b).astype(np.int8)
    if op == "overlay_ba":
        return np.where(mb, b, a).astype(np.int8)
    if op == "and_a":
        return np.where(ma & mb, a, bg).astype(np.int8)
    if op == "and_b":
        return np.where(ma & mb, b, bg).astype(np.int8)
    if op == "xor_values":
        return np.where(ma ^ mb, np.where(ma, a, b), bg).astype(np.int8)
    if op == "or_mask":
        return (ma | mb).astype(np.int8)
    if op == "and_mask":
        return (ma & mb).astype(np.int8)
    if op == "xor_mask":
        return (ma ^ mb).astype(np.int8)
    if op == "eq_mask":
        return (a == b).astype(np.int8)
    if op == "neq_mask":
        return (a != b).astype(np.int8)
    raise KeyError(op)


def infer_color_map(srcs: list[Grid], dsts: list[Grid], allow_many_to_one: bool = True) -> dict[int, int] | None:
    mapping: dict[int, int] = {}
    reverse: dict[int, int] = {}
    for src, dst in zip(srcs, dsts):
        if src.shape != dst.shape:
            return None
        for x, y in zip(map(int, src.ravel()), map(int, dst.ravel())):
            if x in mapping and mapping[x] != y:
                return None
            mapping[x] = y
            if not allow_many_to_one:
                if y in reverse and reverse[y] != x:
                    return None
                reverse[y] = x
    return mapping


def apply_color_map(grid: Grid, mapping: dict[int, int]) -> Grid:
    out = np.array(grid, copy=True)
    for src, dst in mapping.items():
        out[grid == src] = dst
    return out


@dataclass(frozen=True)
class Program:
    name: str
    cost: float
    family: str
    fn: Callable[[Grid], Grid | None]


@dataclass
class ProgramCandidate:
    grid: Grid
    score: float
    confidence: float
    support: int
    min_cost: float
    programs: list[str]
    families: list[str]


def _safe(program_fn: Callable[[Grid], Grid | None], grid: Grid) -> Grid | None:
    try:
        out = program_fn(grid)
        if out is None:
            return None
        out = np.asarray(out, dtype=np.int8)
        return out if valid_grid(out) else None
    except Exception:
        return None


def enumerate_programs() -> list[Program]:
    programs: list[Program] = []

    def add(name: str, cost: float, family: str, fn: Callable[[Grid], Grid | None]) -> None:
        programs.append(Program(name, cost, family, fn))

    for transform in D4_NAMES:
        base_cost = 0.0 if transform == "id" else 0.35
        add(f"d4:{transform}", base_cost, "d4", lambda g, t=transform: d4(g, t))
        for bg_role in ("zero", "mode"):
            add(
                f"crop:{transform}:{bg_role}",
                base_cost + 0.75,
                "crop",
                lambda g, t=transform, r=bg_role: crop_nonbackground(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
            add(
                f"compress:{transform}:{bg_role}",
                base_cost + 1.0,
                "compress",
                lambda g, t=transform, r=bg_role: compress_nonbackground(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
            add(
                f"fillholes:{transform}:{bg_role}",
                base_cost + 1.15,
                "fill",
                lambda g, t=transform, r=bg_role: fill_holes(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
        add(f"trim:{transform}", base_cost + 0.9, "trim", lambda g, t=transform: trim_uniform_border(d4(g, t)))

        for role in ("rarest_nonzero", "common_nonzero"):
            for keep_other in (False, True):
                add(
                    f"colorcrop:{transform}:{role}:{int(keep_other)}",
                    base_cost + 1.15 + 0.15 * keep_other,
                    "colorcrop",
                    lambda g, t=transform, r=role, k=keep_other: crop_color_role(d4(g, t), r, k),
                )

        for bg_role in ("zero", "mode"):
            for diagonal in (False, True):
                for same_color in (False, True):
                    for selector in ("largest", "smallest", "tallest", "widest", "unique_color"):
                        for mask_only in (False, True):
                            add(
                                f"component:{transform}:{bg_role}:{int(diagonal)}:{int(same_color)}:{selector}:{int(mask_only)}",
                                base_cost + 1.15 + 0.15 * diagonal + 0.15 * same_color + 0.2 * mask_only,
                                "component",
                                lambda g, t=transform, br=bg_role, di=diagonal, sc=same_color, se=selector, mo=mask_only: component_patch(
                                    d4(g, t), 0 if br == "zero" else mode_color(d4(g, t)), di, sc, se, mo
                                ),
                            )

        for fy, fx in ((2, 2), (3, 3), (4, 4), (2, 1), (1, 2), (3, 1), (1, 3)):
            add(
                f"scale:{transform}:{fy}x{fx}",
                base_cost + 0.95,
                "scale",
                lambda g, t=transform, y=fy, x=fx: np.repeat(np.repeat(d4(g, t), y, axis=0), x, axis=1),
            )
        for fy, fx in ((2, 2), (3, 3), (4, 4), (2, 1), (1, 2), (3, 1), (1, 3)):
            for reduce_mode in ("uniform", "nonbg", "majority", "tl"):
                for bg_role in ("zero", "mode"):
                    add(
                        f"reduce:{transform}:{fy}x{fx}:{reduce_mode}:{bg_role}",
                        base_cost + 1.15 + (0.25 if reduce_mode in ("majority", "tl") else 0.0),
                        "reduce",
                        lambda g, t=transform, y=fy, x=fx, m=reduce_mode, br=bg_role: block_reduce(
                            d4(g, t), y, x, m, 0 if br == "zero" else mode_color(d4(g, t))
                        ),
                    )

        # Common self-compositions.
        add(f"concat_h:{transform}", base_cost + 1.2, "concat", lambda g, t=transform: np.concatenate([d4(g, t), d4(g, t)], axis=1))
        add(f"concat_v:{transform}", base_cost + 1.2, "concat", lambda g, t=transform: np.concatenate([d4(g, t), d4(g, t)], axis=0))
        add(f"mirror_h:{transform}", base_cost + 1.25, "concat", lambda g, t=transform: np.concatenate([d4(g, t), np.fliplr(d4(g, t))], axis=1))
        add(f"mirror_v:{transform}", base_cost + 1.25, "concat", lambda g, t=transform: np.concatenate([d4(g, t), np.flipud(d4(g, t))], axis=0))

    for split_mode in ("h_sep", "v_sep", "h_half", "v_half"):
        for b_transform in D4_NAMES:
            for op in ("a", "b", "overlay_ab", "overlay_ba", "and_a", "and_b", "xor_values", "or_mask", "and_mask", "xor_mask", "eq_mask", "neq_mask"):
                for bg_role in ("zero", "mode"):
                    def panel_fn(g: Grid, sm=split_mode, bt=b_transform, operation=op, br=bg_role) -> Grid | None:
                        pair = split_two_panels(g, sm)
                        if pair is None:
                            return None
                        a, b = pair
                        b = d4(b, bt)
                        bg = 0 if br == "zero" else mode_color(g)
                        return combine_panels(a, b, operation, bg)

                    add(
                        f"panel:{split_mode}:{b_transform}:{op}:{bg_role}",
                        1.25 + (0.0 if b_transform == "id" else 0.3),
                        "panel",
                        panel_fn,
                    )
    family_priority = {"d4": 0, "crop": 1, "trim": 2, "compress": 3, "colorcrop": 4, "panel": 5, "scale": 6, "reduce": 7, "component": 8, "fill": 9, "concat": 10}
    programs.sort(key=lambda p: (p.cost, family_priority.get(p.family, 99), p.name))
    return programs


_PROGRAMS: list[Program] | None = None


def get_programs() -> list[Program]:
    global _PROGRAMS
    if _PROGRAMS is None:
        _PROGRAMS = enumerate_programs()
    return _PROGRAMS


class SymbolicSolver:
    """Small exact program synthesizer used as a high-precision lane and shape hint."""

    def __init__(self, max_programs: int = 6000, max_outputs: int = 12):
        self.max_programs = max_programs
        self.max_outputs = max_outputs

    def synthesize(self, task: dict[str, Any], test_idx: int = 0) -> list[ProgramCandidate]:
        train_inputs = [as_grid(ex["input"]) for ex in task["train"]]
        train_outputs = [as_grid(ex["output"]) for ex in task["train"]]
        test_input = as_grid(task["test"][test_idx]["input"])
        grouped: dict[tuple[tuple[int, ...], ...], dict[str, Any]] = {}

        for program in get_programs()[: self.max_programs]:
            generated = [_safe(program.fn, grid) for grid in train_inputs]
            if any(value is None for value in generated):
                continue
            srcs = [value for value in generated if value is not None]
            if any(src.shape != dst.shape for src, dst in zip(srcs, train_outputs)):
                continue
            mapping: dict[int, int] | None
            if all(np.array_equal(src, dst) for src, dst in zip(srcs, train_outputs)):
                mapping = {}
            else:
                mapping = infer_color_map(srcs, train_outputs, allow_many_to_one=True)
                if mapping is None or not all(np.array_equal(apply_color_map(src, mapping), dst) for src, dst in zip(srcs, train_outputs)):
                    continue
            test_grid = _safe(program.fn, test_input)
            if test_grid is None:
                continue
            if mapping:
                test_grid = apply_color_map(test_grid, mapping)
            if not valid_grid(test_grid):
                continue
            key = grid_key(test_grid)
            rec = grouped.setdefault(
                key,
                {"grid": test_grid, "programs": [], "families": set(), "costs": []},
            )
            suffix = "" if not mapping else ":map=" + ",".join(f"{a}>{b}" for a, b in sorted(mapping.items()))
            rec["programs"].append(program.name + suffix)
            rec["families"].add(program.family)
            rec["costs"].append(program.cost + (0.35 if mapping else 0.0))

        candidates: list[ProgramCandidate] = []
        for rec in grouped.values():
            support = len(rec["programs"])
            family_count = len(rec["families"])
            min_cost = float(min(rec["costs"]))
            score = 4.0 - min_cost + 0.5 * math.log1p(support) + 0.35 * math.log1p(family_count)
            simple = min_cost <= 0.45 and any(name.startswith("d4:") for name in rec["programs"])
            consensus = family_count >= 2 and support >= 3 and min_cost <= 1.6
            confidence = 0.995 if simple else 0.975 if consensus else min(0.94, 0.60 + 0.06 * support + 0.05 * family_count - 0.05 * min_cost)
            candidates.append(
                ProgramCandidate(
                    grid=np.asarray(rec["grid"], dtype=np.int8),
                    score=float(score),
                    confidence=float(confidence),
                    support=support,
                    min_cost=min_cost,
                    programs=sorted(rec["programs"], key=len)[:16],
                    families=sorted(rec["families"]),
                )
            )
        candidates.sort(key=lambda c: (c.score, c.confidence, -c.min_cost), reverse=True)
        return candidates[: self.max_outputs]


# ------------------------- Output-shape inference -------------------------

def _shape_features(grid: Grid) -> dict[str, int]:
    h, w = grid.shape
    out: dict[str, int] = {
        "h": h,
        "w": w,
        "area": h * w,
        "ncolors": len(np.unique(grid)),
        "ncolors_nonzero": len([v for v in np.unique(grid) if int(v) != 0]),
    }
    for bg_name, bg in (("zero", 0), ("mode", mode_color(grid))):
        mask = grid != bg
        box = bbox_of_mask(mask)
        out[f"rows:{bg_name}"] = int(np.any(mask, axis=1).sum())
        out[f"cols:{bg_name}"] = int(np.any(mask, axis=0).sum())
        if box is not None:
            out[f"bbox_h:{bg_name}"] = box[1] - box[0]
            out[f"bbox_w:{bg_name}"] = box[3] - box[2]
        for diagonal in (False, True):
            for same_color in (False, True):
                comps = components(grid, bg, diagonal, same_color)
                tag = f"{bg_name}:{int(diagonal)}:{int(same_color)}"
                out[f"ncomp:{tag}"] = len(comps)
                if comps:
                    out[f"max_area:{tag}"] = max(c.area for c in comps)
                    out[f"min_area:{tag}"] = min(c.area for c in comps)
                    out[f"max_h:{tag}"] = max(c.height for c in comps)
                    out[f"max_w:{tag}"] = max(c.width for c in comps)
                    out[f"min_h:{tag}"] = min(c.height for c in comps)
                    out[f"min_w:{tag}"] = min(c.width for c in comps)
                    for selector in ("largest", "smallest", "tallest", "widest", "unique_color"):
                        comp = select_component(comps, selector)
                        if comp is not None:
                            out[f"comp_h:{tag}:{selector}"] = comp.height
                            out[f"comp_w:{tag}:{selector}"] = comp.width
                            out[f"comp_area:{tag}:{selector}"] = comp.area
    for color in range(10):
        mask = grid == color
        out[f"count:c{color}"] = int(mask.sum())
        out[f"rows:c{color}"] = int(np.any(mask, axis=1).sum())
        out[f"cols:c{color}"] = int(np.any(mask, axis=0).sum())
        box = bbox_of_mask(mask)
        if box is not None:
            out[f"bbox_h:c{color}"] = box[1] - box[0]
            out[f"bbox_w:c{color}"] = box[3] - box[2]
    for mode in ("h_sep", "v_sep", "h_half", "v_half"):
        pair = split_two_panels(grid, mode)
        if pair is not None:
            out[f"panel_h:{mode}"] = pair[0].shape[0]
            out[f"panel_w:{mode}"] = pair[0].shape[1]
    return {name: int(value) for name, value in out.items() if 0 <= int(value) <= 900}


def _expanded_dim_features(grid: Grid) -> dict[str, tuple[int, float]]:
    base = _shape_features(grid)
    out: dict[str, tuple[int, float]] = {}
    for name, value in base.items():
        if name in ("h", "w"):
            base_cost = 0.15
        elif name.startswith(("bbox_", "rows:", "cols:", "panel_")):
            base_cost = 0.45
        else:
            base_cost = 0.75
        out[name] = (value, base_cost)
        for delta in (-3, -2, -1, 1, 2, 3):
            candidate = value + delta
            if 1 <= candidate <= 30:
                out[f"{name}{delta:+d}"] = (candidate, base_cost + 0.55 + 0.08 * abs(delta))
        for factor in (2, 3, 4):
            candidate = value * factor
            if 1 <= candidate <= 30:
                out[f"{name}*{factor}"] = (candidate, base_cost + 0.5 + 0.12 * factor)
            if value % factor == 0 and 1 <= value // factor <= 30:
                out[f"{name}/{factor}"] = (value // factor, base_cost + 0.65 + 0.1 * factor)
    for constant in range(1, 31):
        out[f"const:{constant}"] = (constant, 1.25)
    return out


def infer_output_shapes(task: dict[str, Any], test_grid: Grid, max_shapes: int = 12) -> list[tuple[int, int, float, str]]:
    """Rank output dimensions fitted exactly on demonstrations.

    High confidence is deliberately withheld when a rule must extrapolate from a
    dimension/feature that never varied in demonstrations. This prevents a
    constant-shape shortcut from blocking the unconstrained decoder.
    """
    train_inputs = [as_grid(ex["input"]) for ex in task["train"]]
    train_outputs = [as_grid(ex["output"]) for ex in task["train"]]
    pairs = [(x.shape, y.shape) for x, y in zip(train_inputs, train_outputs)]
    th, tw = test_grid.shape
    scored: dict[tuple[int, int], tuple[float, str]] = {}

    def add(shape: tuple[int, int], score: float, reason: str) -> None:
        h, w = map(int, shape)
        if 1 <= h <= 30 and 1 <= w <= 30:
            old = scored.get((h, w))
            if old is None or score > old[0]:
                scored[(h, w)] = (float(score), reason)

    in_heights = {shape[0] for shape, _ in pairs}
    in_widths = {shape[1] for shape, _ in pairs}
    direct_safe = (th in in_heights or len(in_heights) >= 2) and (tw in in_widths or len(in_widths) >= 2)

    if len({out_shape for _, out_shape in pairs}) == 1:
        add(pairs[0][1], 5.0 if direct_safe else 2.65, "constant" if direct_safe else "constant-ambiguous")
    if all(inp == out for inp, out in pairs):
        add((th, tw), 6.0, "same")
    if all((inp[1], inp[0]) == out for inp, out in pairs):
        add((tw, th), 5.8, "transpose")

    deltas = {(out[0] - inp[0], out[1] - inp[1]) for inp, out in pairs}
    if len(deltas) == 1:
        dy, dx = next(iter(deltas))
        add((th + dy, tw + dx), 3.4 if direct_safe else 2.60, "delta" if direct_safe else "delta-ambiguous")

    ratios = [(out[0] / inp[0], out[1] / inp[1]) for inp, out in pairs]
    if len(set(ratios)) == 1:
        ry, rx = ratios[0]
        hh, ww = round(th * ry), round(tw * rx)
        if abs(hh - th * ry) < 1e-9 and abs(ww - tw * rx) < 1e-9:
            add((hh, ww), 3.8 if direct_safe else 2.75, "ratio" if direct_safe else "ratio-ambiguous")

    train_features = [_expanded_dim_features(grid) for grid in train_inputs]
    test_features = _expanded_dim_features(test_grid)
    common = set(test_features)
    for features in train_features:
        common &= set(features)
    h_rules: list[tuple[float, str, int]] = []
    w_rules: list[tuple[float, str, int]] = []
    for name in common:
        values = [features[name][0] for features in train_features]
        base_cost = max(features[name][1] for features in train_features)
        test_value = test_features[name][0]
        # A feature that was constant in all examples cannot safely be treated as
        # causal when it changes at test time; keep it as a low-confidence hint.
        extrapolation_penalty = 0.85 if len(set(values)) == 1 and test_value != values[0] else 0.0
        cost = base_cost + extrapolation_penalty
        if all(value == output.shape[0] for value, output in zip(values, train_outputs)):
            h_rules.append((cost, name, test_value))
        if all(value == output.shape[1] for value, output in zip(values, train_outputs)):
            w_rules.append((cost, name, test_value))
    h_rules.sort()
    w_rules.sort()
    for h_cost, h_name, h_value in h_rules[:24]:
        for w_cost, w_name, w_value in w_rules[:24]:
            add((h_value, w_value), 4.7 - h_cost - w_cost, f"features:{h_name}|{w_name}")

    add((th, tw), 0.9, "fallback-same")
    add((tw, th), 0.7, "fallback-transpose")
    for output in train_outputs:
        add(output.shape, 0.55, "observed-output-shape")

    ordered = sorted(scored.items(), key=lambda item: item[1][0], reverse=True)
    return [(h, w, score, reason) for (h, w), (score, reason) in ordered[:max_shapes]]


# ------------------------- Candidate verification -------------------------

def _shape_relation(inp: Grid, out: Grid) -> str:
    if out.shape == inp.shape:
        return "same"
    if out.shape == inp.T.shape:
        return "transpose"
    if out.shape[0] <= inp.shape[0] and out.shape[1] <= inp.shape[1]:
        return "crop"
    if out.shape[0] >= inp.shape[0] and out.shape[1] >= inp.shape[1]:
        return "expand"
    return "mixed"


def _relation_vector(inp: Grid, out: Grid) -> tuple[str, np.ndarray]:
    ivals = set(map(int, np.unique(inp)))
    ovals = set(map(int, np.unique(out)))
    vec = np.asarray(
        [
            out.shape[0] / inp.shape[0],
            out.shape[1] / inp.shape[1],
            len(ovals) / 10.0,
            (len(ovals - ivals) - len(ivals - ovals)) / 10.0,
            float(np.mean(out != mode_color(out))),
            float(np.mean(out == np.fliplr(out))),
            float(np.mean(out == np.flipud(out))),
        ],
        dtype=float,
    )
    return _shape_relation(inp, out), vec


def structural_consistency(task: dict[str, Any], test_input: Grid, candidate: Grid) -> float:
    candidate = as_grid(candidate)
    train_relations = [_relation_vector(as_grid(ex["input"]), as_grid(ex["output"])) for ex in task["train"]]
    test_relation, test_vec = _relation_vector(test_input, candidate)
    categories = [category for category, _ in train_relations]
    category_score = categories.count(test_relation) / max(1, len(categories))
    vectors = np.stack([vector for _, vector in train_relations])
    center = np.median(vectors, axis=0)
    scale = np.maximum(np.median(np.abs(vectors - center), axis=0), 0.08)
    distance = float(np.mean(np.minimum(4.0, np.abs(test_vec - center) / scale)))
    return float(np.clip(0.55 * category_score + 0.45 * math.exp(-distance), 0.0, 1.0))


def fallback_grids(task: dict[str, Any], test_idx: int, limit: int = 2) -> list[Grid]:
    """Cheap valid attempts used only when no neural result is available."""
    test = as_grid(task["test"][test_idx]["input"])
    candidates: list[Grid] = []
    symbolic = SymbolicSolver(max_outputs=4).synthesize(task, test_idx)
    candidates.extend(c.grid for c in symbolic)
    candidates.append(test.copy())
    candidates.append(test.T.copy())
    for bg in (0, mode_color(test)):
        cropped = crop_nonbackground(test, bg)
        if cropped is not None:
            candidates.append(cropped)
    unique: list[Grid] = []
    seen: set[tuple[tuple[int, ...], ...]] = set()
    for grid in candidates:
        if valid_grid(grid) and grid_key(grid) not in seen:
            seen.add(grid_key(grid))
            unique.append(np.asarray(grid, dtype=np.int8))
        if len(unique) >= limit:
            break
    return unique


In [ ]:
%%writefile starter.py
from __future__ import annotations

import argparse
import bz2
import json
import os
import pickle
import shutil
import tempfile
import time
from typing import Any

import numpy as np
import torch
import torch.multiprocessing as mp


COMPETITION_DIR = os.getenv("ARC_COMPETITION_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
OUTPUT_DIR = os.getenv("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
DEFAULT_DEBUG_KEYS = ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]


def _strict_symbolic(candidate: Any, candidates: list[Any]) -> float:
    if candidate.confidence < 0.995 or candidate.min_cost > 0.45:
        return 0.0
    margin = candidate.score - (candidates[1].score if len(candidates) > 1 else -99.0)
    if any(name.startswith("d4:") for name in candidate.programs) and (len(candidates) == 1 or margin >= 0.65):
        return 0.999
    return 0.0


def _record(grid: np.ndarray, source: str, priority: float, confidence: float = 0.0, programs: list[str] | None = None) -> dict[str, Any]:
    return {
        "beam_score": 99.0,
        "beam_nll": 9.9,
        "beam_len": int(grid.size + grid.shape[0]),
        "score_aug": [],
        "score_aug_norm": [],
        "answer_len": int(grid.size + grid.shape[0]),
        "solution": np.asarray(grid, dtype=np.int8),
        "source": source,
        "fallback_priority": float(priority),
        "symbolic_confidence": float(confidence),
        "structural_score": 0.0,
        "shape_score": 0.0,
        "programs": programs or [],
    }


def precompute_fallbacks(data: dict[str, Any], keys: list[str], output_dir: str, retrieval_index: Any | None = None) -> set[str]:
    """Write valid pass@2 records before GPU work so timeouts never leave blanks."""
    from arc_symbolic import (
        SymbolicSolver,
        as_grid,
        crop_nonbackground,
        grid_key,
        infer_output_shapes,
        mode_color,
        valid_grid,
    )

    solver = SymbolicSolver(max_programs=500, max_outputs=8)
    start = time.time()
    written = 0
    fully_retrieved: set[str] = set()
    for key in keys:
        task = data[key]
        key_retrieved = True
        for test_idx, test in enumerate(task["test"]):
            retrieved = retrieval_index.solve(task, test_idx) if retrieval_index is not None else []
            key_retrieved = key_retrieved and bool(retrieved)
            candidates = [] if retrieved else solver.synthesize(task, test_idx)
            test_grid = as_grid(test["input"])
            records: list[dict[str, Any]] = []
            seen: set[tuple[tuple[int, ...], ...]] = set()

            for retrieved_grid, source_key in retrieved:
                h = grid_key(retrieved_grid)
                if h not in seen:
                    seen.add(h)
                    records.append(
                        _record(
                            retrieved_grid,
                            source="retrieval",
                            priority=1000.0,
                            confidence=1.0,
                            programs=[f"canonical-retrieval:{source_key}"],
                        )
                    )

            for rank, candidate in enumerate(candidates[:4]):
                h = grid_key(candidate.grid)
                if h in seen:
                    continue
                seen.add(h)
                confidence = _strict_symbolic(candidate, candidates)
                source = "symbolic" if confidence >= 0.999 else "symbolic_low"
                records.append(
                    _record(
                        candidate.grid,
                        source=source,
                        priority=80.0 - rank + candidate.score,
                        confidence=confidence,
                        programs=candidate.programs,
                    )
                )

            generic: list[tuple[np.ndarray, float]] = [(test_grid.copy(), 20.0), (test_grid.T.copy(), 18.0)]
            for bg, priority in ((0, 16.0), (mode_color(test_grid), 15.0)):
                cropped = crop_nonbackground(test_grid, bg)
                if cropped is not None:
                    generic.append((cropped, priority))

            # A monochrome shape prior covers the rare all-background/all-color task.
            shapes = infer_output_shapes(task, test_grid, max_shapes=2)
            if shapes:
                h, w = int(shapes[0][0]), int(shapes[0][1])
                train_modes = [mode_color(as_grid(example["output"])) for example in task["train"]]
                fill = train_modes[0] if len(set(train_modes)) == 1 else 0
                generic.append((np.full((h, w), fill, dtype=np.int8), 5.0))

            for grid, priority in generic:
                if not valid_grid(grid):
                    continue
                h = grid_key(grid)
                if h in seen:
                    continue
                seen.add(h)
                records.append(_record(grid, source="fallback", priority=priority))
                if len(records) >= 6:
                    break

            # ARC requires two attempts; ensure at least two distinct valid grids.
            if len(records) < 2:
                for value in range(10):
                    candidate = np.full(test_grid.shape, value, dtype=np.int8)
                    h = grid_key(candidate)
                    if h not in seen:
                        seen.add(h)
                        records.append(_record(candidate, source="fallback", priority=-float(value)))
                    if len(records) >= 2:
                        break

            from arc_exec_rl.common import atomic_candidate_pickle
            atomic_candidate_pickle(os.path.join(output_dir, f"{key}_{test_idx}.fallback"), records)
            written += 1
        if key_retrieved:
            fully_retrieved.add(key)
    print(
        f"*** Precomputed fallbacks for {written} outputs in {time.time() - start:.1f}s; "
        f"fully retrieved puzzles={len(fully_retrieved)}"
    )
    return fully_retrieved


def local_worker(rank: int, queue: Any, end_time: float, nprocs: int, marker_dir: str) -> None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    torch.set_default_device("cpu")
    torch.set_num_threads(max(1, int(os.getenv("OMP_NUM_THREADS", "12"))))

    if rank > 0:
        ready = os.path.join(marker_dir, f"worker_{rank - 1}.ready")
        failed = os.path.join(marker_dir, f"worker_{rank - 1}.failed")
        deadline = time.time() + 900
        while not os.path.exists(ready):
            if os.path.exists(failed):
                raise RuntimeError(f"Previous worker failed during Unsloth import: {failed}")
            if time.time() > deadline:
                raise TimeoutError(f"Timed out waiting for {ready}")
            time.sleep(2)

    try:
        # Sequential import prevents concurrent global patching inside Unsloth.
        from arc_solver import worker
    except Exception:
        open(os.path.join(marker_dir, f"worker_{rank}.failed"), "w").close()
        raise

    open(os.path.join(marker_dir, f"worker_{rank}.ready"), "w").close()
    print(f"[Rank {rank}] start on visible GPU 0")
    worker(rank, queue, end_time, n_workers=nprocs)
    print(f"[Rank {rank}] done")


def _complexity(task: dict[str, Any]) -> int:
    total = 0
    for split in ("train", "test"):
        for example in task[split]:
            total += int(np.asarray(example["input"]).size)
            if "output" in example:
                total += int(np.asarray(example["output"]).size)
    return total * max(1, len(task["test"]))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--nprocs", type=int, default=4)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() not in ("", "0", "false", "no")
    filename = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    test_path = os.path.join(COMPETITION_DIR, filename)
    with open(test_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    requested_raw = os.getenv("ARC_DEV_KEYS", "").strip() or os.getenv("ARC_DEBUG_KEYS", "").strip()
    if requested_raw:
        requested = [item.strip() for item in requested_raw.split(",") if item.strip()]
        keys = [key for key in requested if key in data]
    else:
        # A serious interactive run evaluates all 120 public tasks; a competition
        # rerun evaluates every hidden task.  No silent four-task debug subset.
        keys = sorted(data, key=lambda key: (_complexity(data[key]), key), reverse=True)
    if not keys:
        raise RuntimeError("No ARC tasks selected; inspect ARC_DEV_KEYS/competition input")

    # Explicit deadlines must NEVER be re-armed after they expire.
    end_time = args.end_time if args.end_time > 0 else time.time() + 11.5 * 3600
    if end_time <= time.time():
        print("*** Explicit deadline already expired; GPU work not started")
        return

    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    from arc_retrieval import RetrievalIndex

    retrieval_index = RetrievalIndex.from_files(
        [
            (
                os.path.join(COMPETITION_DIR, "arc-agi_training_challenges.json"),
                os.path.join(COMPETITION_DIR, "arc-agi_training_solutions.json"),
            )
        ]
    )
    print(f"*** Retrieval index contains {len(retrieval_index)} reference puzzles")
    fully_retrieved = precompute_fallbacks(data, keys, OUTPUT_DIR, retrieval_index=retrieval_index)
    gpu_keys = [key for key in keys if key not in fully_retrieved]
    if not gpu_keys:
        print("*** Every requested puzzle was solved by exact canonical retrieval; GPU stage skipped")
        return

    nprocs = max(1, min(int(args.nprocs), 4, len(gpu_keys)))
    manager = mp.Manager()
    queue = manager.Queue()
    for key in gpu_keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    marker_dir = tempfile.mkdtemp(prefix="arc_unsloth_", dir="/kaggle")
    print(f"*** Solving {len(gpu_keys)} unresolved puzzles with {nprocs} workers; deadline={end_time:.0f}")
    mp.spawn(local_worker, args=(queue, end_time, nprocs, marker_dir), nprocs=nprocs, join=True)


if __name__ == "__main__":
    main()


## 1. Data contract and complete fallback before model loading

In [ ]:
from grid_rl.common import atomic_json,initial_submission,public_task,truthy,json_hash,validate_submission,file_hash
from grid_rl.inputs import resolve_data,resolve_model,scan,install_offline,dependency_report,model_contract
from grid_rl.modeling import check_export,source_fingerprint
from grid_rl.evidence import decoder_identity
import subprocess
subprocess.run([sys.executable,"-m","compileall","-q","grid_rl"],check=True)
subprocess.run([sys.executable,"-m","py_compile","arc_loader.py","arc_solver.py","starter.py"],check=True)
resolved_data,data_audit=resolve_data(INPUT_ROOT,SAFE_DIR/"normalized_data",COMPETITION_DIR_OVERRIDE)
competition_dir=Path(resolved_data)
atomic_json(SAFE_DIR/"dataset_audit.json",data_audit)
os.environ["ARC_COMPETITION_DIR"]=str(competition_dir)
rerun=truthy(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
challenge_path=competition_dir/("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")
all_tasks={k:public_task(v) for k,v in json.loads(challenge_path.read_text()).items()}
tasks=all_tasks
if ARC_DEV_KEYS:
    if rerun:raise ValueError("Development keys are forbidden in a hidden rerun")
    keys={k.strip() for k in ARC_DEV_KEYS.split(",") if k.strip()}
    if not keys or not keys<=set(tasks):raise ValueError("Invalid development keys")
    tasks={k:v for k,v in tasks.items() if k in keys}
submission_path=Path("/kaggle/working/submission.json")
atomic_json(submission_path,initial_submission(tasks))
print("FALLBACK BANKED:",len(tasks),"tasks",sum(len(t["test"]) for t in tasks.values()),"outputs")
print("PLACEHOLDER WARNING:",data_audit.get("test_is_public_training_placeholder"))
# This evidence compares configuration, not host path or chosen checkpoint.
DECODER_CONFIG={k:v for k,v in globals().copy().items() if k in (
    "RUN_PROFILE","ARC_DEV_KEYS","NPROCS","TTT_OPTIMIZER","TTT_LEARNING_RATE","TTT_LORA_R","TTT_LORA_ALPHA",
    "MAX_TRAIN_AUGMENTS","FORCE_FULL_TTT","MAX_ACCEPTED_TTT_LOSS","ALLOW_EXACT_RETRIEVAL_SKIP","MAX_SEQ_LENGTH",
    "SYMBOLIC_PROGRAMS","ALLOW_SYMBOLIC_ATTEMPT2","TTA_COLOR_PERMUTATIONS","MAX_SHAPE_HYPOTHESES",
    "CONSTRAINED_BRANCH_CAP","UNCONSTRAINED_BRANCH_CAP","BEAM_CAP","MAX_CANDIDATES_PER_VIEW",
    "MAX_PUZZLE_SECONDS","MIN_PUZZLE_SECONDS","RESERVE_SCORING_SECONDS","TOTAL_NOTEBOOK_HOURS","FINAL_RESERVE_MINUTES")}
DECODER_HASH=decoder_identity(str(ROOT));DECODER_CONFIG_HASH=json_hash(DECODER_CONFIG)
atomic_json(SAFE_DIR/"decoder_config.json",DECODER_CONFIG)
TRAINED_MANIFEST=None;BASE_INFO=None
if MODEL_CHOICE in ("trained_grid_rl","trained_sft_control"):
    if not TRAINED_MODEL_DIR:
        candidates=scan(INPUT_ROOT,{"grid_rl_manifest.json"})
        if len(candidates)!=1:
            raise ValueError("Requested trained weights are absent or ambiguous. Attach one completed export or set TRAINED_MODEL_DIR. Baseline is NOT selected automatically.")
        TRAINED_MODEL_DIR=str(candidates[0].parent)
    TRAINED_MANIFEST=check_export(TRAINED_MODEL_DIR,require_rl=(MODEL_CHOICE=="trained_grid_rl"),require_reload=True)
    if MODEL_CHOICE=="trained_sft_control":
        if TRAINED_MANIFEST["training"].get("sft_control_steps",0)<1 or TRAINED_MANIFEST["training"].get("mode")!="sft_control":
            raise ValueError("This is not the requested supervised-control export")
        if rerun:raise ValueError("Supervised control is an interactive experiment, not a hidden-RL submission")
    model_path=str(Path(TRAINED_MODEL_DIR).resolve())
    if rerun:
        if not PROMOTION_FILE:raise ValueError("Trained hidden rerun requires paired_promotion.json; run public/held-out comparison first")
        gate=json.loads(Path(PROMOTION_FILE).read_text())
        if not (gate.get("passed") is True and gate.get("weights_id")==TRAINED_MANIFEST["weights_id"] and
                gate.get("decoder_hash")==DECODER_HASH and gate.get("decoder_config_hash")==DECODER_CONFIG_HASH):
            raise ValueError("Missing, failed, or mismatched score gate; no override to pretend success")
else:
    model_path,BASE_CONTRACT=resolve_model(INPUT_ROOT,"grid",BASELINE_MODEL_PATH,prefer="sft139")
    if (Path(model_path)/"grid_rl_manifest.json").exists():raise ValueError("Baseline selection points at an RL export; select untouched sft139 explicitly")
    BASE_INFO=source_fingerprint(model_path)
os.environ["ARC_MODEL_PATH"]=model_path
atomic_json(SAFE_DIR/"chosen_model_header.json",model_contract(model_path,"grid"))
if INSTALL_OFFLINE_DEPS:install_offline(WHEELHOUSE)
atomic_json(SAFE_DIR/"dependencies.json",dependency_report())
from arc_dataset_guard import build_safe_routes
route_report=build_safe_routes(SAFE_DIR,RUN_PROFILE)
GLOBAL_END_TIME=NOTEBOOK_START_TIME+TOTAL_NOTEBOOK_HOURS*3600-FINAL_RESERVE_MINUTES*60
PLAN={"model_choice":MODEL_CHOICE,"actual_model_path":model_path,
      "trained_weights_id":TRAINED_MANIFEST["weights_id"] if TRAINED_MANIFEST else None,
      "rl_optimizer_steps":TRAINED_MANIFEST["training"]["rl_optimizer_steps"] if TRAINED_MANIFEST else 0,
      "method_used":{"trained_grid_rl":"RL-merged grid checkpoint + inherited TTT",
                     "trained_sft_control":"supervised-control grid checkpoint + inherited TTT",
                     "baseline":"explicit untouched baseline"}[MODEL_CHOICE],
      "neural_deadline":GLOBAL_END_TIME,"fallback_only_is_success":False,"score_claim":None}
atomic_json(SAFE_DIR/"execution_plan.json",PLAN)
print(json.dumps(PLAN,indent=2))
if time.time()>=GLOBAL_END_TIME:raise TimeoutError("Deadline already expired; no new budget granted")


## 2. Four independent workers; absolute deadline

In [ ]:
import subprocess,signal

def run_logged(command, log_file, timeout_seconds, check=True):
    """Kill the whole child group on expiry; never reset an expired deadline."""
    log_file=Path(log_file);log_file.parent.mkdir(parents=True,exist_ok=True)
    if timeout_seconds<=0:raise TimeoutError("No time remains before launching child")
    start=time.monotonic();offset=0
    with log_file.open("w") as stream:
        proc=subprocess.Popen(command,cwd=ROOT,stdout=stream,stderr=subprocess.STDOUT,
                              env=dict(os.environ,PYTHONUNBUFFERED="1",OMP_NUM_THREADS="2"),start_new_session=True)
        try:
            while proc.poll() is None:
                if time.monotonic()-start>=timeout_seconds:
                    os.killpg(proc.pid,signal.SIGTERM)
                    try:proc.wait(timeout=30)
                    except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()
                    break
                time.sleep(2)
                with log_file.open() as reader:
                    reader.seek(offset);part=reader.read();offset=reader.tell()
                if part:print(part,end="",flush=True)
        except BaseException:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
                try:proc.wait(timeout=30)
                except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()
            raise
    with log_file.open() as reader:reader.seek(offset);print(reader.read(),end="",flush=True)
    result={"returncode":proc.returncode,"elapsed_seconds":time.monotonic()-start,"log":str(log_file)}
    print(result)
    if check and proc.returncode!=0:raise RuntimeError("Child failed; preserved log: "+str(log_file))
    return result


In [ ]:
RESULT=run_logged([sys.executable,"-u","starter.py","--end-time",str(GLOBAL_END_TIME),"--nprocs",str(NPROCS)],
                  SAFE_DIR/"inference.log",GLOBAL_END_TIME-time.time(),check=False)
atomic_json(SAFE_DIR/"worker_process.json",RESULT)


## 3. Recover retained candidates and preserve valid baseline selection

In [ ]:
import numpy as np
from arc_decoder import ArcDecoder
from arc_loader import ArcDataset
from grid_rl.common import validate_grid
data=ArcDataset.from_file(str(challenge_path))
decoder=ArcDecoder(data.split_multi_replies(),n_guesses=2)
decoder.load_decoded_results(str(OUTPUT_DIR))
selection=decoder.run_selection_algo()
anchor=json.loads(submission_path.read_text())
anchor_bank={}; primary_ready=0
for key,task in tasks.items():
    for i in range(len(task["test"])):
        bk=f"{key}_{i}"; chosen=selection.get(bk,[])
        if chosen:
            a=np.asarray(chosen[0]).tolist();validate_grid(a);anchor[key][i]["attempt_1"]=a
            b=np.asarray(chosen[1]).tolist() if len(chosen)>1 else a
            validate_grid(b);anchor[key][i]["attempt_2"]=b
        values=decoder.decoded_results.get(bk,{})
        primary_ready+=int(any(v.get("source")=="llm" for v in values.values()))
        grids=[]
        for v in values.values():
            try:
                g=np.asarray(v["solution"]).tolist();validate_grid(g)
                if g not in grids: grids.append(g)
            except (KeyError,ValueError,TypeError):pass
        anchor_bank[f"{key}:{i}"]=grids
validate_submission(tasks,anchor)
atomic_json(submission_path,anchor)
anchor_path=SAFE_DIR/"anchor_submission.json";atomic_json(anchor_path,anchor)
atomic_json(SAFE_DIR/"anchor_candidate_bank.json",anchor_bank)
coverage=primary_ready/max(1,sum(len(t["test"]) for t in tasks.values()))
print("ANCHOR SUBMISSION VALID; neural coverage:",coverage)
# A deficient primary pass is an error signal, not a reason to claim solver success.


## 4. Score only after prediction is frozen

In [ ]:
from grid_rl.common import score_submission
final=json.loads(submission_path.read_text());validate_submission(tasks,final)
counts=sum(len(t["test"]) for t in tasks.values())
evidence={**PLAN,"tasks":len(tasks),"pairs":counts,"full_dataset":not bool(ARC_DEV_KEYS),
    "challenge_hash":json_hash(tasks),"solutions_hash":None,"decoder_hash":DECODER_HASH,
    "decoder_config_hash":DECODER_CONFIG_HASH,"submission_hash":json_hash(final),
    "worker_returncode":RESULT["returncode"],"neural_coverage":coverage,
    "base_weights_id":BASE_INFO["weights_id"] if BASE_INFO else None,
    "training_base_weights_id":TRAINED_MANIFEST["training"]["base_weights_id"] if TRAINED_MANIFEST else None,
    "evaluation_scope":"hidden_rerun_no_answers" if rerun else "public_evaluation_diagnostic",
    "hidden_score_guarantee":False,"wall_seconds":time.time()-NOTEBOOK_START_TIME}
if not rerun:
    all_solutions=json.loads((competition_dir/"arc-agi_evaluation_solutions.json").read_text())
    solutions={k:all_solutions[k] for k in tasks}
    evidence["solutions_hash"]=json_hash(solutions)
    score=score_submission(tasks,solutions,final)
    oracle=sum(any(g==y for g in anchor_bank.get(f"{k}:{i}",[])) for k in tasks for i,y in enumerate(solutions[k]))
    evidence["score"]=score;evidence["candidate_union_oracle_exact"]=oracle
    needed=math.ceil(.37*counts);evidence["target_37_public_pair_micro_met"]=score["pass2_exact"]>=needed
    print("PUBLIC EXACT PASS@2:",score["pass2_exact"],"/",counts)
    print("37% PAIR-MICRO TARGET:","MET" if score["pass2_exact"]>=needed else "NOT MET","— requires",needed)
    print("CANDIDATE UNION ORACLE:",oracle,"/",counts)
    print("PUBLIC SCORE DOES NOT GUARANTEE HIDDEN SCORE")
    atomic_json(SAFE_DIR/"evaluation_solutions_for_scoring_only.json",solutions)
else:print("HIDDEN SCORE UNKNOWN: no solutions were read")
atomic_json(SAFE_DIR/"challenge_inputs_only.json",tasks)
atomic_json(SAFE_DIR/"final_submission.json",final)
atomic_json(SAFE_DIR/"run_evidence.json",evidence)
print("FINAL SUBMISSION VALID:",submission_path)
print("MODEL USED:",MODEL_CHOICE,"; recorded RL updates:",PLAN["rl_optimizer_steps"])
if coverage<.95 or RESULT["returncode"]!=0:print("WARNING: output validity is not a successful neural run; inspect run_evidence.json")
if SAVE_DIAGNOSTICS_ZIP:
    import zipfile
    archive=Path("/kaggle/working/arc_gridrl_diagnostics.zip")
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for p in SAFE_DIR.iterdir():
            if p.is_file() and p.suffix in (".json",".jsonl",".log") and p.stat().st_size<100_000_000:z.write(p,p.name)
    print("DIAGNOSTICS:",archive)
